# Module 3 — MCP Architecture & Cross-LLM Evaluation
## Version A — DeepSeek V4 Flash 0731

**Purpose:** Remote query-time retrieval + MCP + DeepSeek V4 Flash 0731 evaluation.

**Repository note**
- This notebook preserves the executed experiment outputs for auditability and portfolio review.
- API credentials are loaded from environment variables or Google Colab secrets; no literal API keys are stored in this notebook.
- Large corpora and persistent BM25/DuckDB artifacts are intentionally excluded from Git and are accessed remotely or through the documented Kaggle artifact.
- Frozen benchmark, Top-K, MCP search limits and system-prompt controls are preserved from the executed experiment.

> Reproducibility dependencies and required secrets are documented in `README.md` and `requirements-module3.txt`.


```text
SECTION 0 — Runtime Environment
Cell 0 — Install Runtime Libraries
Cell 1 — Imports + Global Version A Configuration

SECTION 1 — Remote Corpus Access
Cell 2 — Configure Remote GSMA Telecom Corpora
Cell 3 — Validate Remote TCC + 3GPP Access

SECTION 2 — Version A Retrieval Engine
Cell 4  — Query Processing + Dynamic Source / Shard Selection
Cell 5  — Remote TCC + 3GPP Retrieval Engines
Cell 5A — Optimized 3GPP Evidence Selection
Cell 6A — TCC Retrieval Validation
Cell 6B — Dedicated 3GPP Retrieval Validation
Cell 6C — Dynamic Source Selection Validation
Cell 6D — 3GPP Evidence Quality + Latency Validation

SECTION 3 — MCP Knowledge Service
Cell 7 — Unified MCP Telecom Knowledge Search Tool
Cell 8 — Three-Route MCP Tool Validation

SECTION 4 — DeepSeek V4 + MCP Orchestration
Cell 9  — DeepSeek V4 Setup + Frozen Shared System Prompt
Cell 10 — End-to-End MCP Orchestration
Cell 11 — End-to-End Pilot

SECTION 5 — Version A Evaluation
Cell 12 — Module 3 Evaluation Benchmark v2


============================================================
VERSION A — FROZEN RUNTIME ARCHITECTURE
============================================================

Version A retrieves knowledge directly from remote source documents at
query time and does not construct or restore a persistent local index.

Knowledge sources
-----------------
GSMA Telco Common Corpus (TCC)
→ remote Parquet via DuckDB HTTPFS

GSMA 3GPP
→ remote marked/**/raw.md specifications via direct HTTP access


Deterministic route semantics
-----------------------------
3GPP-native query → dedicated 3GPP
TCC/non-3GPP query → selected TCC collections
Mixed-domain query → Hybrid 3GPP + TCC

Hybrid excludes TCC 3GPP-TSG contribution material because dedicated
3GPP specifications provide the normative standards path.


End-to-end path
---------------
User Question
→ DeepSeek V4 Flash 0731
→ ONE MCP tool: search_telecom_knowledge(query, top_k=5)
→ deterministic 3GPP / TCC / Hybrid router
→ direct remote retrieval
→ bounded evidence
→ DeepSeek V4 Flash 0731
→ final answer

The LLM never selects a source family, collection, shard or retrieval
profile. Those choices remain internal to the knowledge service.


Frozen controls
---------------
Provider: OpenRouter
Maximum MCP searches: 3
Top-K: 5
Maximum evidence excerpt: 2,500 characters/source
Maximum output tokens: 1,800
Target answer: approximately 300–500 words
Benchmark: 5 × 3GPP | 2 × TCC | 1 × Hybrid
Benchmark SHA-256: d40c0090c371f0a99ea6057bbb3fa8024e6b7d4174e037e7a6cf9fef9b9667f5


A/B experimental control
------------------------
Within each paired model comparison, Version A and Version B hold the
model, frozen system prompt, route semantics, source domains, benchmark,
Top-K, evidence limits, search budget and tracing controls constant.

Primary architectural variable:

Version A → direct remote, non-persistent retrieval
Version B → persistent DuckDB BM25/FTS retrieval

Do not tune routing against observed Benchmark v2 failures. Final
correctness, completeness, relevance and grounding are evaluated
separately from runtime execution metrics.
```


# **SECTION 0 — Runtime Environment**

## **Cell 0 — Install Runtime Libraries**

In [ ]:
# ============================================================
# CELL 0 — INSTALL VERSION A RUNTIME LIBRARIES
# ============================================================


# ============================================================
# CORE RUNTIME DEPENDENCIES
# ============================================================

!pip install -q \
    duckdb \
    pandas \
    fastmcp \
    openai \
    huggingface_hub


# ============================================================
# VERIFY RUNTIME INSTALLATION
# ============================================================

from importlib.metadata import (
    version,
    PackageNotFoundError
)


RUNTIME_PACKAGES = [
    "duckdb",
    "pandas",
    "fastmcp",
    "openai",
    "huggingface_hub"
]


print("=" * 80)
print("VERSION A — DEEPSEEK V4 RUNTIME DEPENDENCY CHECK")
print("=" * 80)


for package in RUNTIME_PACKAGES:

    try:

        package_version = version(
            package
        )

        print(
            f"{package:<20} : "
            f"{package_version}"
        )

    except PackageNotFoundError:

        raise RuntimeError(
            f"Required Version A runtime package "
            f"'{package}' is not installed."
        )


print("=" * 80)

print(
    "✓ Version A runtime dependencies installed."
)

print(
    "✓ OpenAI-compatible client available for OpenRouter."
)

print(
    "✓ Runtime ready for remote corpus configuration."
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 858.1/858.1 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.2/237.2 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.9/357.9 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.0/170.0 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.0/273.0 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.4/196.4 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 20.5 MB/s eta 0:00:00
VERSION A — DEEPSEEK V4 RUNTIME DEPENDENCY CHECK
duckdb               : 1.3.2
pandas               : 2.2.3
fastmcp    

**Observation — Runtime Environment**

The Version A DeepSeek V4 notebook installs only the libraries required for remote telecom corpus access, DuckDB retrieval, MCP orchestration and OpenRouter-based LLM integration.

The OpenAI-compatible client replaces the Anthropic SDK at the model-provider layer, while the underlying Version A retrieval and MCP architecture remains unchanged.

***Key Decision:*** Isolate the DeepSeek V4 experiment to the LLM integration and orchestration layers so that retrieval behaviour remains directly comparable with the Claude Sonnet 5 and Claude Haiku 4.5 experiments.


## **Cell 1 — Imports + Global Version A Configuration**

In [ ]:
# ============================================================
# CELL 1 — IMPORTS + GLOBAL VERSION A CONFIGURATION
# ============================================================

import os
import re
import json
import time
import math
import asyncio
import hashlib

import duckdb
import pandas as pd

import urllib.request

from huggingface_hub import HfApi
from fastmcp import FastMCP, Client


# ============================================================
# VERSION A ARCHITECTURE
# ============================================================

ARCHITECTURE_VERSION = (
    "Version A"
)

RETRIEVAL_ARCHITECTURE = (
    "Remote raw-corpus retrieval"
)


# ============================================================
# REMOTE TELECOM CORPORA
# ============================================================

REMOTE_CORPORA = {

    "tcc": {
        "repo_id":
            "GSMA/Telco-Common-Corpus",

        "repo_type":
            "dataset",

        "source_family":
            "TCC",

        "file_pattern":
            "data/*.parquet",

        "format":
            "parquet"
    },

    "3gpp": {
        "repo_id":
            "GSMA/3GPP",

        "repo_type":
            "dataset",

        "source_family":
            "3GPP",

        "file_pattern":
            "marked/**/raw.md",

        "format":
            "markdown"
    }
}


# ============================================================
# FROZEN VERSION A RETRIEVAL CONFIGURATION
# ============================================================

# Version A adaptively searches only a subset of
# remote Parquet shards for each query.
MAX_REMOTE_SHARDS = 5


# Maximum number of selected remote shards searched
# concurrently.
PARALLEL_SHARD_CONCURRENCY = 5


# Final globally ranked evidence returned by retrieval.
TOP_K_RESULTS = 5


# ============================================================
# VERSION A RETRIEVAL SCORING
# ============================================================

# Manual lexical / phrase / proximity ranking.
#
# These labels identify the frozen Version A retrieval
# approach. The detailed scoring logic is reconstructed
# in the retrieval-engine cells.
RETRIEVAL_SCORING = (
    "lexical + phrase + proximity"
)


# ============================================================
# MCP / EVALUATION LIMITS
# ============================================================

# Standardized with Version B for controlled A/B evaluation.
MAX_MCP_SEARCHES = 3

MAX_RETRIEVED_SOURCES = 5

MCP_EXCERPT_CHARS = 2500


# ============================================================
# DUCKDB RUNTIME CONFIGURATION
# ============================================================

DUCKDB_THREADS = (
    os.cpu_count()
    or 4
)


# Version A performs remote scans rather than maintaining
# a persistent local database.
DUCKDB_DATABASE = (
    ":memory:"
)


# ============================================================
# STRICT FROZEN-CONFIG VALIDATION
# ============================================================

if (
    MAX_REMOTE_SHARDS
    != 5
):

    raise RuntimeError(
        "Version A requires "
        "MAX_REMOTE_SHARDS = 5."
    )


if (
    PARALLEL_SHARD_CONCURRENCY
    != 5
):

    raise RuntimeError(
        "Version A requires "
        "PARALLEL_SHARD_CONCURRENCY = 5."
    )


if (
    TOP_K_RESULTS
    != 5
):

    raise RuntimeError(
        "Version A requires "
        "TOP_K_RESULTS = 5."
    )


if (
    MAX_MCP_SEARCHES
    != 3
):

    raise RuntimeError(
        "Version A evaluation requires "
        "MAX_MCP_SEARCHES = 3."
    )


if (
    MAX_RETRIEVED_SOURCES
    != 5
):

    raise RuntimeError(
        "Version A evaluation requires "
        "MAX_RETRIEVED_SOURCES = 5."
    )


# ============================================================
# CONFIGURATION SUMMARY
# ============================================================

print("=" * 80)

print(
    "VERSION A — TELECOM AI MCP RUNTIME CONFIGURATION"
)

print("=" * 80)


print(
    f"Architecture          : "
    f"{RETRIEVAL_ARCHITECTURE}"
)


print(
    "\nRemote Knowledge Sources:"
)


for source_name, source_config in (
    REMOTE_CORPORA.items()
):

    print(
        f"  {source_name.upper():<6} : "
        f"{source_config['repo_id']}"
    )

    print(
        f"           Pattern : "
        f"{source_config['file_pattern']}"
    )

    print(
        f"           Format  : "
        f"{source_config['format']}"
    )


print(
    f"\nDuckDB Database       : "
    f"{DUCKDB_DATABASE}"
)

print(
    f"DuckDB Threads        : "
    f"{DUCKDB_THREADS}"
)

print(
    f"Retrieval Scoring     : "
    f"{RETRIEVAL_SCORING}"
)

print(
    f"Max Remote Shards     : "
    f"{MAX_REMOTE_SHARDS}"
)

print(
    f"Parallel Shard Calls  : "
    f"{PARALLEL_SHARD_CONCURRENCY}"
)

print(
    f"Top-K Results         : "
    f"{TOP_K_RESULTS}"
)

print(
    f"MCP Excerpt           : "
    f"{MCP_EXCERPT_CHARS:,} chars/source"
)

print(
    f"Max MCP Searches      : "
    f"{MAX_MCP_SEARCHES}"
)

print("=" * 80)


print(
    "✓ Version A runtime configuration loaded."
)

print(
    "✓ TCC remote source configured."
)

print(
    "✓ Dedicated 3GPP remote source configured."
)

print(
    "✓ No persistent local knowledge base configured."
)

print(
    "✓ Ready for Cell 2 — Configure Remote GSMA Telecom Corpora."
)

VERSION A — TELECOM AI MCP RUNTIME CONFIGURATION
Architecture          : Remote raw-corpus retrieval

Remote Knowledge Sources:
  TCC    : GSMA/Telco-Common-Corpus
           Pattern : data/*.parquet
           Format  : parquet
  3GPP   : GSMA/3GPP
           Pattern : marked/**/raw.md
           Format  : markdown

DuckDB Database       : :memory:
DuckDB Threads        : 8
Retrieval Scoring     : lexical + phrase + proximity
Max Remote Shards     : 5
Parallel Shard Calls  : 5
Top-K Results         : 5
MCP Excerpt           : 2,500 chars/source
Max MCP Searches      : 3
✓ Version A runtime configuration loaded.
✓ TCC remote source configured.
✓ Dedicated 3GPP remote source configured.
✓ No persistent local knowledge base configured.
✓ Ready for Cell 2 — Configure Remote GSMA Telecom Corpora.


**Observation — Runtime Configuration**

Version A is configured for direct query-time access to the remote GSMA Telco Common Corpus using adaptive remote shard selection and lexical, phrase and proximity ranking.

***Key Decision:*** Preserve the original Version A remote-retrieval architecture while standardizing MCP and evaluation limits with Version B for a controlled A/B comparison.

# **SECTION 1 — Remote Corpus Access**

## **Cell 2 — Configure Remote GSMA Telecom Corpora**

In [ ]:
# ============================================================
# CELL 2 — CONFIGURE REMOTE GSMA TELECOM CORPORA
# ============================================================


# ============================================================
# HUGGING FACE API
# ============================================================

hf_api = HfApi()


# ============================================================
# REMOTE FILE REGISTRY
# ============================================================

REMOTE_CORPUS_FILES = {

    "tcc":
        [],

    "3gpp":
        []
}


REMOTE_CORPUS_URLS = {

    "tcc":
        [],

    "3gpp":
        []
}


# ============================================================
# HUGGING FACE URL HELPER
# ============================================================

def build_hf_resolve_url(
    repo_id,
    file_path
):
    """
    Build a direct Hugging Face resolve URL
    without downloading the source file.
    """

    return (
        "https://huggingface.co/datasets/"
        f"{repo_id}/resolve/main/"
        f"{file_path}"
    )


# ============================================================
# FILE-SCOPE HELPERS
# ============================================================

def is_tcc_runtime_file(
    file_path
):
    """
    Version A TCC scope:

        data/*.parquet

    Only Parquet files directly under data/
    are included.
    """

    file_path = str(
        file_path
    )


    return (

        file_path.startswith(
            "data/"
        )

        and

        file_path.endswith(
            ".parquet"
        )

        and

        file_path.count("/")
        == 1
    )


def is_3gpp_runtime_file(
    file_path
):
    """
    Version A dedicated 3GPP scope:

        marked/**/raw.md

    This matches the source scope used in
    the Version B Knowledge Base build.
    """

    file_path = str(
        file_path
    )


    return (

        file_path.startswith(
            "marked/"
        )

        and

        file_path.endswith(
            "/raw.md"
        )
    )


# ============================================================
# DISCOVER TCC REMOTE FILES
# ============================================================

print("=" * 90)

print(
    "VERSION A — REMOTE GSMA TELECOM CORPUS CONFIGURATION"
)

print("=" * 90)


print(
    "\nDiscovering GSMA Telco Common Corpus..."
)


tcc_repo_files = (
    hf_api.list_repo_files(
        repo_id=
            REMOTE_CORPORA[
                "tcc"
            ][
                "repo_id"
            ],

        repo_type=
            REMOTE_CORPORA[
                "tcc"
            ][
                "repo_type"
            ]
    )
)


tcc_files = sorted([

    file_path

    for file_path
    in tcc_repo_files

    if is_tcc_runtime_file(
        file_path
    )
])


if not tcc_files:

    raise RuntimeError(
        "No TCC runtime Parquet files "
        "were discovered."
    )


tcc_urls = [

    build_hf_resolve_url(

        REMOTE_CORPORA[
            "tcc"
        ][
            "repo_id"
        ],

        file_path
    )

    for file_path
    in tcc_files
]


REMOTE_CORPUS_FILES[
    "tcc"
] = tcc_files


REMOTE_CORPUS_URLS[
    "tcc"
] = tcc_urls


# ============================================================
# DISCOVER DEDICATED 3GPP REMOTE FILES
# ============================================================

print(
    "Discovering dedicated GSMA 3GPP specifications..."
)


gpp_repo_files = (
    hf_api.list_repo_files(
        repo_id=
            REMOTE_CORPORA[
                "3gpp"
            ][
                "repo_id"
            ],

        repo_type=
            REMOTE_CORPORA[
                "3gpp"
            ][
                "repo_type"
            ]
    )
)


gpp_files = sorted([

    file_path

    for file_path
    in gpp_repo_files

    if is_3gpp_runtime_file(
        file_path
    )
])


if not gpp_files:

    raise RuntimeError(
        "No dedicated 3GPP raw.md files "
        "were discovered."
    )


gpp_urls = [

    build_hf_resolve_url(

        REMOTE_CORPORA[
            "3gpp"
        ][
            "repo_id"
        ],

        file_path
    )

    for file_path
    in gpp_files
]


REMOTE_CORPUS_FILES[
    "3gpp"
] = gpp_files


REMOTE_CORPUS_URLS[
    "3gpp"
] = gpp_urls


# ============================================================
# GLOBAL REMOTE CORPUS REGISTRY
# ============================================================

REMOTE_SOURCE_REGISTRY = {

    "tcc": {

        "repo_id":
            REMOTE_CORPORA[
                "tcc"
            ][
                "repo_id"
            ],

        "source_family":
            "TCC",

        "format":
            "parquet",

        "file_count":
            len(
                tcc_files
            ),

        "files":
            tcc_files,

        "urls":
            tcc_urls
    },


    "3gpp": {

        "repo_id":
            REMOTE_CORPORA[
                "3gpp"
            ][
                "repo_id"
            ],

        "source_family":
            "3GPP",

        "format":
            "markdown",

        "file_count":
            len(
                gpp_files
            ),

        "files":
            gpp_files,

        "urls":
            gpp_urls
    }
}


# ============================================================
# VALIDATION
# ============================================================

if (
    len(
        REMOTE_SOURCE_REGISTRY
    )
    != 2
):

    raise RuntimeError(
        "Version A requires both "
        "TCC and dedicated 3GPP sources."
    )


if (
    REMOTE_SOURCE_REGISTRY[
        "tcc"
    ][
        "file_count"
    ]
    <= 0
):

    raise RuntimeError(
        "TCC source discovery failed."
    )


if (
    REMOTE_SOURCE_REGISTRY[
        "3gpp"
    ][
        "file_count"
    ]
    <= 0
):

    raise RuntimeError(
        "3GPP source discovery failed."
    )


# ============================================================
# CORPUS SUMMARY
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "VERSION A — REMOTE SOURCE REGISTRY"
)

print("=" * 90)


for (
    source_name,
    source_info
) in REMOTE_SOURCE_REGISTRY.items():

    print(
        f"{source_name.upper():<8} | "
        f"{source_info['repo_id']:<30} | "
        f"{source_info['format']:<8} | "
        f"{source_info['file_count']:,} files"
    )


print("=" * 90)


# ============================================================
# SAMPLE FILES
# ============================================================

print(
    "\nTCC Remote File Examples:"
)


for file_path in (
    tcc_files[:3]
):

    print(
        f"  - {file_path}"
    )


print(
    "\n3GPP Remote File Examples:"
)


for file_path in (
    gpp_files[:3]
):

    print(
        f"  - {file_path}"
    )


# ============================================================
# FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 90
)


print(
    "✓ TCC remote corpus discovered."
)

print(
    "✓ Dedicated 3GPP specification corpus discovered."
)

print(
    "✓ No source files downloaded."
)

print(
    "✓ No normalization or indexing performed."
)

print(
    "✓ Version A remote source scope aligned with Version B."
)

print(
    "✓ Ready for Cell 3 — "
    "Validate Remote Corpus Access."
)

print("=" * 90)

VERSION A — REMOTE GSMA TELECOM CORPUS CONFIGURATION

Discovering GSMA Telco Common Corpus...
Discovering dedicated GSMA 3GPP specifications...

VERSION A — REMOTE SOURCE REGISTRY
TCC      | GSMA/Telco-Common-Corpus       | parquet  | 100 files
3GPP     | GSMA/3GPP                      | markdown | 15,052 files

TCC Remote File Examples:
  - data/tcc_001.parquet
  - data/tcc_002.parquet
  - data/tcc_003.parquet

3GPP Remote File Examples:
  - marked/Rel-10/21_series/21101/raw.md
  - marked/Rel-10/21_series/21111/raw.md
  - marked/Rel-10/21_series/21201/raw.md

✓ TCC remote corpus discovered.
✓ Dedicated 3GPP specification corpus discovered.
✓ No source files downloaded.
✓ No normalization or indexing performed.
✓ Version A remote source scope aligned with Version B.
✓ Ready for Cell 3 — Validate Remote Corpus Access.


**Observation — Remote Corpus Configuration**

Version A successfully discovered **100 remote TCC Parquet files** and **15,052 dedicated 3GPP specification documents** without downloading or indexing the source corpora.

***Key Decision:*** Keep both telecom knowledge sources remote and perform selective query-time access so that Version A remains a non-persistent retrieval architecture while matching Version B's source scope.

## **Cell 3 — Validate Remote TCC + 3GPP Access.**

In [ ]:
# ============================================================
# CELL 3 — VALIDATE REMOTE TCC + 3GPP ACCESS
# ============================================================

# ============================================================
# VALIDATION CONFIGURATION
# ============================================================

TCC_VALIDATION_URL = (
    REMOTE_SOURCE_REGISTRY[
        "tcc"
    ][
        "urls"
    ][0]
)

GPP_VALIDATION_URL = (
    REMOTE_SOURCE_REGISTRY[
        "3gpp"
    ][
        "urls"
    ][0]
)

GPP_VALIDATION_BYTES = 8192


print("=" * 90)

print(
    "VERSION A — VALIDATE REMOTE TCC + 3GPP ACCESS"
)

print("=" * 90)


# ============================================================
# 1. VALIDATE DUCKDB HTTPFS
# ============================================================

print(
    "\n[1/2] Validating TCC remote Parquet access "
    "through DuckDB HTTPFS..."
)


tcc_conn = duckdb.connect(
    DUCKDB_DATABASE
)


try:

    # --------------------------------------------------------
    # DUCKDB RUNTIME
    # --------------------------------------------------------

    tcc_conn.execute(
        f"SET threads = {int(DUCKDB_THREADS)}"
    )

    tcc_conn.execute(
        "INSTALL httpfs"
    )

    tcc_conn.execute(
        "LOAD httpfs"
    )


    # --------------------------------------------------------
    # READ REMOTE PARQUET SCHEMA
    # --------------------------------------------------------

    tcc_schema = (
        tcc_conn.execute(
            """
            DESCRIBE
            SELECT *
            FROM read_parquet(?)
            """,
            [
                TCC_VALIDATION_URL
            ]
        )
        .df()
    )


    if tcc_schema.empty:

        raise RuntimeError(
            "Remote TCC Parquet schema "
            "could not be read."
        )


    tcc_columns = (
        tcc_schema[
            "column_name"
        ]
        .astype(str)
        .tolist()
    )


    if (
        "text"
        not in tcc_columns
    ):

        raise RuntimeError(
            "Expected 'text' column was not found "
            "in the remote TCC Parquet schema."
        )


    # --------------------------------------------------------
    # READ A SMALL REMOTE SAMPLE
    #
    # LIMIT 3 ensures this is an access validation,
    # not a corpus retrieval operation.
    # --------------------------------------------------------

    tcc_sample = (
        tcc_conn.execute(
            """
            SELECT
                collection,
                identifier,
                title,
                LEFT(
                    CAST(text AS VARCHAR),
                    500
                ) AS text_preview

            FROM read_parquet(?)

            WHERE text IS NOT NULL

            LIMIT 3
            """,
            [
                TCC_VALIDATION_URL
            ]
        )
        .df()
    )


    if tcc_sample.empty:

        raise RuntimeError(
            "Remote TCC Parquet file was reachable "
            "but returned no validation rows."
        )


finally:

    tcc_conn.close()


print(
    "TCC Repository        : "
    f"{REMOTE_CORPORA['tcc']['repo_id']}"
)

print(
    "Validation File       : "
    f"{REMOTE_SOURCE_REGISTRY['tcc']['files'][0]}"
)

print(
    "Access Method         : "
    "DuckDB HTTPFS"
)

print(
    f"Columns Detected      : "
    f"{len(tcc_columns)}"
)

print(
    f"Sample Rows           : "
    f"{len(tcc_sample)}"
)

print(
    "TCC Remote Access     : ✓ PASS"
)


print(
    "\nTCC Schema:"
)

display(
    tcc_schema
)


print(
    "\nTCC Validation Sample:"
)

display(
    tcc_sample
)


# ============================================================
# 2. VALIDATE DEDICATED 3GPP REMOTE MARKDOWN ACCESS
# ============================================================

print(
    "\n" + "-" * 90
)

print(
    "[2/2] Validating dedicated 3GPP "
    "remote specification access..."
)


# Request only the beginning of one document where
# the remote server supports HTTP byte-range requests.
gpp_request = urllib.request.Request(

    GPP_VALIDATION_URL,

    headers={
        "Range":
            f"bytes=0-{GPP_VALIDATION_BYTES - 1}",

        "User-Agent":
            "Telecom-AI-MCP-Version-A"
    }
)


try:

    with urllib.request.urlopen(
        gpp_request,
        timeout=30
    ) as response:

        gpp_raw = (
            response.read(
                GPP_VALIDATION_BYTES
            )
        )

        gpp_http_status = (
            response.status
        )

        gpp_content_type = (
            response.headers.get(
                "Content-Type",
                ""
            )
        )


except Exception as exc:

    raise RuntimeError(
        "Unable to access the dedicated "
        "3GPP remote raw.md document."
    ) from exc


gpp_text = (
    gpp_raw
    .decode(
        "utf-8",
        errors="replace"
    )
    .strip()
)


if not gpp_text:

    raise RuntimeError(
        "Dedicated 3GPP remote document "
        "was reachable but returned no text."
    )


print(
    "3GPP Repository       : "
    f"{REMOTE_CORPORA['3gpp']['repo_id']}"
)

print(
    "Validation File       : "
    f"{REMOTE_SOURCE_REGISTRY['3gpp']['files'][0]}"
)

print(
    "Access Method         : "
    "Direct remote raw.md"
)

print(
    f"HTTP Status           : "
    f"{gpp_http_status}"
)

print(
    f"Content Type          : "
    f"{gpp_content_type}"
)

print(
    f"Validation Bytes      : "
    f"{len(gpp_raw):,}"
)

print(
    "3GPP Remote Access    : ✓ PASS"
)


print(
    "\n3GPP Text Preview:"
)

print(
    gpp_text[:1500]
)


# ============================================================
# STRICT FINAL VALIDATION
# ============================================================

tcc_access_valid = (
    not tcc_sample.empty
    and
    "text" in tcc_columns
)

gpp_access_valid = (
    len(gpp_text) > 0
)


if not (
    tcc_access_valid
    and
    gpp_access_valid
):

    raise RuntimeError(
        "Version A remote corpus "
        "access validation failed."
    )


# ============================================================
# FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "✓ VERSION A REMOTE CORPUS ACCESS VALIDATION PASSED"
)

print(
    "✓ TCC remote Parquet access confirmed through DuckDB HTTPFS."
)

print(
    "✓ Dedicated 3GPP raw.md access confirmed."
)

print(
    "✓ Only one validation file per source was accessed."
)

print(
    "✓ No full-corpus scan was performed."
)

print(
    "✓ No files were persisted locally."
)

print(
    "✓ Ready for Cell 4 — "
    "Query Processing + Adaptive Source Selection."
)

print("=" * 90)

VERSION A — VALIDATE REMOTE TCC + 3GPP ACCESS

[1/2] Validating TCC remote Parquet access through DuckDB HTTPFS...
TCC Repository        : GSMA/Telco-Common-Corpus
Validation File       : data/tcc_001.parquet
Access Method         : DuckDB HTTPFS
Columns Detected      : 13
Sample Rows           : 3
TCC Remote Access     : ✓ PASS

TCC Schema:


,column_name,column_type,null,key,default,extra
0,identifier,VARCHAR,YES,None,None,None
1,collection,VARCHAR,YES,None,None,None
2,open_type,VARCHAR,YES,None,None,None
3,curator,VARCHAR,YES,None,None,None
4,license,VARCHAR,YES,None,None,None
5,date,VARCHAR,YES,None,None,None
6,title,VARCHAR,YES,None,None,None
7,creator,VARCHAR,YES,None,None,None
8,language,VARCHAR,YES,None,None,None
9,language_type,VARCHAR,YES,None,None,None



TCC Validation Sample:


,collection,identifier,title,text_preview
0,IETF-RFCs,RFC8202,June 2017,### 9.2 Informative References\n\n[Err4519] R...
1,3GPP-TSG,C1-072238,C1-072238,"**Source:** Huawei\n\n**Title:** Draft CR, MRF..."
2,USPTO,US-12347254-B2,Using combination of GPS and BLE beaconing to ...,# Using combination of GPS and BLE beaconing t...



------------------------------------------------------------------------------------------
[2/2] Validating dedicated 3GPP remote specification access...
3GPP Repository       : GSMA/3GPP
Validation File       : marked/Rel-10/21_series/21101/raw.md
Access Method         : Direct remote raw.md
HTTP Status           : 206
Content Type          : text/plain; charset=utf-8
Validation Bytes      : 8,192
3GPP Remote Access    : ✓ PASS

3GPP Text Preview:
# **3rd Generation Partnership Project; Technical Specification Group Services and System Aspects; Technical Specifications and Technical Reports for a UTRAN-based 3GPP system (Release 10)**

![3GPP logo](935eed7aa61f7777f62cfc032e11bee9_img.jpg)

The 3GPP logo is displayed within a rectangular border. It features the letters '3GPP' in a stylized, bold font. The '3' is on the left, followed by 'G', 'P', and 'P'. Below the 'G' is a small red signal icon consisting of three curved lines. A small 'TM' trademark symbol is located in the top rig

**Observation — Remote Corpus Access**

Version A successfully validated direct runtime access to both telecom knowledge sources using **DuckDB HTTPFS for TCC Parquet data** and HTTP byte-range access for dedicated **3GPP specification documents**.

***Key Decision:*** Preserve selective remote access to both corpora without downloading, indexing or scanning the complete source collections.

# **SECTION 2 — Version A Retrieval Engine**

## **Cell 4 — Query Processing + Dynamic Source / Shard Selection**

In [ ]:
# ============================================================
# CELL 4 — QUERY PROCESSING + DYNAMIC SOURCE / SHARD SELECTION
# ============================================================

from collections import defaultdict


# ============================================================
# QUERY PROCESSING
# ============================================================

LOW_INFORMATION_TERMS = {
    "5g",
    "nr",
    "network",
    "procedure",
    "function",
    "role",
    "system",
    "handling",
    "management"
}


QUERY_STOPWORDS = {
    "a",
    "an",
    "the",
    "and",
    "or",
    "of",
    "to",
    "for",
    "in",
    "on",
    "with",
    "by",
    "from",
    "as",
    "at",
    "what",
    "which",
    "how",
    "why",
    "when",
    "where",
    "is",
    "are",
    "was",
    "were",
    "be",
    "been",
    "being",
    "do",
    "does",
    "did",
    "explain",
    "describe",
    "identify",
    "including",
    "according",
    "primary"
}


def normalize_query(query):
    """
    Normalize query text for consistent matching.
    """

    if not query or not query.strip():

        raise ValueError(
            "Query must not be empty."
        )

    return re.sub(
        r"\s+",
        " ",
        query.strip().lower()
    )


def tokenize_query(
    query,
    remove_stopwords=True
):
    """
    Extract telecom-aware query terms.

    Preserves identifiers such as:
    5G, N4, S-NSSAI, NG-RAN, 23.501, GTP-U.
    """

    tokens = re.findall(
        r"[a-z0-9]+(?:[.-][a-z0-9]+)*",
        normalize_query(query)
    )

    if not remove_stopwords:
        return tokens

    return [
        token
        for token in tokens
        if token not in QUERY_STOPWORDS
    ]


def build_query_phrases(query):
    """
    Create bigrams and trigrams from searchable terms.
    """

    terms = tokenize_query(
        query,
        remove_stopwords=True
    )

    bigrams = [
        " ".join(
            terms[i:i + 2]
        )
        for i in range(
            len(terms) - 1
        )
    ]

    trigrams = [
        " ".join(
            terms[i:i + 3]
        )
        for i in range(
            len(terms) - 2
        )
    ]

    return (
        bigrams,
        trigrams
    )


def build_proximity_pairs(query):
    """
    Create adjacent searchable-term pairs
    for proximity scoring.
    """

    terms = tokenize_query(
        query,
        remove_stopwords=True
    )

    return [
        (
            terms[i],
            terms[i + 1]
        )
        for i in range(
            len(terms) - 1
        )
        if terms[i] != terms[i + 1]
    ]


def term_weight(term):
    """
    Lightweight Version A lexical weighting.
    """

    if term in LOW_INFORMATION_TERMS:
        return 1.0

    if len(term) <= 2:
        return 0.5

    return 2.0


def escape_sql_literal(value):
    """
    Escape SQL string literals.
    """

    return str(
        value
    ).replace(
        "'",
        "''"
    )


def build_query_features(query):
    """
    Build reusable Version A query features.
    """

    normalized = normalize_query(
        query
    )

    terms = tokenize_query(
        query,
        remove_stopwords=True
    )

    (
        bigrams,
        trigrams
    ) = build_query_phrases(
        query
    )

    proximity_pairs = build_proximity_pairs(
        query
    )

    return {
        "normalized_query":
            normalized,

        "terms":
            terms,

        "bigrams":
            bigrams,

        "trigrams":
            trigrams,

        "proximity_pairs":
            proximity_pairs
    }


# ============================================================
# BOUNDARY-AWARE SIGNAL MATCHING
# ============================================================

def contains_signal(
    normalized_query,
    signal
):
    """
    Match complete telecom terms or phrases rather
    than arbitrary substrings.
    """

    pattern = (
        r"(?<![a-z0-9])"
        +
        re.escape(
            signal.lower()
        )
        +
        r"(?![a-z0-9])"
    )

    return (
        re.search(
            pattern,
            normalized_query
        )
        is not None
    )


# ============================================================
# TCC REMOTE SHARD REGISTRY
# ============================================================

tcc_file_paths = (
    REMOTE_SOURCE_REGISTRY[
        "tcc"
    ][
        "files"
    ]
)

tcc_urls = (
    REMOTE_SOURCE_REGISTRY[
        "tcc"
    ][
        "urls"
    ]
)


if len(tcc_file_paths) != len(tcc_urls):

    raise RuntimeError(
        "TCC file-path and URL registries "
        "are inconsistent."
    )


TCC_REMOTE_SHARDS = [

    {
        "shard_idx":
            shard_idx,

        "file_path":
            file_path,

        "url":
            url
    }

    for shard_idx, (
        file_path,
        url
    ) in enumerate(
        zip(
            tcc_file_paths,
            tcc_urls
        )
    )
]


if not TCC_REMOTE_SHARDS:

    raise RuntimeError(
        "No TCC remote shards are available."
    )


# ============================================================
# ADAPTIVE TCC SHARD MEMORY
# ============================================================

SHARD_TERM_STATS = defaultdict(
    lambda:
        defaultdict(float)
)


# ============================================================
# TCC SPREAD SELECTION
# ============================================================

def select_spread_shards(
    parquet_files,
    max_shards=MAX_REMOTE_SHARDS
):
    """
    Deterministic fallback when no adaptive
    shard knowledge exists.
    """

    total_files = len(
        parquet_files
    )

    if total_files <= max_shards:

        return list(
            parquet_files
        )

    if max_shards <= 1:

        return [
            parquet_files[0]
        ]

    selected = []

    selected_indices = set()

    step = (
        (total_files - 1)
        /
        (max_shards - 1)
    )

    for i in range(
        max_shards
    ):

        idx = round(
            i * step
        )

        if idx not in selected_indices:

            selected.append(
                parquet_files[
                    idx
                ]
            )

            selected_indices.add(
                idx
            )

    if len(selected) < max_shards:

        for idx, item in enumerate(
            parquet_files
        ):

            if idx not in selected_indices:

                selected.append(
                    item
                )

                selected_indices.add(
                    idx
                )

            if len(selected) >= max_shards:
                break

    return selected[
        :max_shards
    ]


# ============================================================
# ADAPTIVE TCC SHARD SCORING
# ============================================================

def get_query_shard_scores(
    query,
    parquet_files
):
    """
    Score TCC shards using lightweight
    query-term search history.
    """

    terms = tokenize_query(
        query
    )

    shard_scores = []

    for shard in parquet_files:

        shard_idx = shard[
            "shard_idx"
        ]

        score = 0.0

        for term in terms:

            score += (
                SHARD_TERM_STATS[
                    term
                ].get(
                    shard_idx,
                    0.0
                )
            )

        shard_scores.append({

            "shard_idx":
                shard_idx,

            "score":
                score,

            "parquet_file":
                shard
        })

    return shard_scores


def select_adaptive_shards(
    query,
    parquet_files,
    max_shards=MAX_REMOTE_SHARDS
):
    """
    Fresh runtime:
        SPREAD

    After successful retrieval history:
        ADAPTIVE
    """

    if not parquet_files:

        return {
            "mode":
                "EMPTY",

            "shards":
                []
        }

    max_shards = min(
        max_shards,
        len(parquet_files)
    )

    scores = get_query_shard_scores(
        query,
        parquet_files
    )

    useful = [
        item
        for item in scores
        if item[
            "score"
        ] > 0
    ]

    if not useful:

        return {
            "mode":
                "SPREAD",

            "shards":
                select_spread_shards(
                    parquet_files,
                    max_shards=
                        max_shards
                )
        }

    ranked = sorted(
        scores,
        key=lambda item:
            item[
                "score"
            ],
        reverse=True
    )

    selected = [
        item[
            "parquet_file"
        ]
        for item in ranked[
            :max_shards
        ]
    ]

    return {
        "mode":
            "ADAPTIVE",

        "shards":
            selected
    }


def update_shard_term_stats(
    query,
    shard_idx,
    relevance_score
):
    """
    Update lightweight query-term → TCC shard
    statistics.

    No document content is persisted.
    """

    if relevance_score <= 0:
        return

    for term in tokenize_query(
        query
    ):

        SHARD_TERM_STATS[
            term
        ][
            shard_idx
        ] += float(
            relevance_score
        )


# ============================================================
# DYNAMIC SOURCE SIGNALS
# ============================================================

GPP_SOURCE_SIGNALS = {

    "3gpp",
    "3gpp ts",
    "3gpp tr",

    "5g standalone",
    "5g sa",
    "5gs",
    "5g core",
    "ng-ran",

    "amf",
    "smf",
    "upf",
    "ausf",
    "udm",
    "nssf",
    "pcf",

    "pdu session",
    "registration",
    "mobility management",

    "s-nssai",
    "network slicing",
    "network slice",

    "5qi",
    "qos flow",

    "ngap",
    "xnap",
    "xn interface",

    "rrc",
    "radio link failure",
    "rlf",

    "n1 interface",
    "n2 interface",
    "n3 interface",
    "n4 interface",

    "pfcp"
}


TCC_IETF_SIGNALS = {
    "ietf",
    "rfc",
    "quic",
    "http",
    "http3",
    "http/3",
    "tls",
    "tcp",
    "udp",
    "dns"
}


TCC_RESEARCH_SIGNALS = {
    "research",
    "paper",
    "study",
    "ieee",
    "openalex"
}


TCC_PATENT_SIGNALS = {
    "patent",
    "invention",
    "uspto",
    "epo"
}


TCC_KNOWLEDGE_SIGNALS = {
    "wikipedia",
    "wikidata",
    "general telecom",
    "general telecommunications"
}


# ============================================================
# 3GPP DOMAIN → SPECIFICATION INFERENCE
# ============================================================

GPP_SPEC_RULES = [

    {
        "name":
            "5GS architecture",

        "signals": {
            "amf",
            "smf",
            "upf",
            "nssf",
            "s-nssai",
            "network slice",
            "network slicing",
            "5qi",
            "qos flow",
            "pdu session",
            "5g core",
            "5g standalone",
            "5g sa"
        },

        "specs": [
            "23.501"
        ]
    },

    {
        "name":
            "5GS procedures",

        "signals": {
            "registration",
            "mobility management",
            "pdu session",
            "handover",
            "inter-gnb handover",
            "service request",
            "session release",
            "pdu session release"
        },

        "specs": [
            "23.502"
        ]
    },

    {
        "name":
            "5GS policy and QoS",

        "signals": {
            "policy control",
            "pcf",
            "qos policy",
            "5qi"
        },

        "specs": [
            "23.503"
        ]
    },

    {
        "name":
            "5G security",

        "signals": {
            "authentication",
            "ausf",
            "udm",
            "security",
            "5g aka",
            "aka"
        },

        "specs": [
            "33.501"
        ]
    },

    {
        "name":
            "5GS NAS",

        "signals": {
            "nas",
            "5gmm",
            "5gsm"
        },

        "specs": [
            "24.501"
        ]
    },

    {
        "name":
            "PFCP and N4",

        "signals": {
            "pfcp",
            "n4",
            "n4 interface"
        },

        "specs": [
            "29.244"
        ]
    },

    {
        "name":
            "NR RRC",

        "signals": {
            "rrc",
            "radio link failure",
            "rlf",
            "rrc re-establishment"
        },

        "specs": [
            "38.331"
        ]
    },

    {
        "name":
            "NR architecture",

        "signals": {
            "nr architecture",
            "gnb",
            "ng-ran",
            "handover",
            "inter-gnb handover"
        },

        "specs": [
            "38.300"
        ]
    },

    {
        "name":
            "NGAP",

        "signals": {
            "ngap",
            "n2",
            "n2 interface"
        },

        "specs": [
            "38.413"
        ]
    },

    {
        "name":
            "XnAP",

        "signals": {
            "xnap",
            "xn",
            "xn interface",
            "inter-gnb handover"
        },

        "specs": [
            "38.423"
        ]
    }
]


MAX_GPP_SPEC_CANDIDATES = 3


# ============================================================
# EXPLICIT 3GPP SPEC EXTRACTION
# ============================================================

def extract_explicit_3gpp_specs(
    query
):
    """
    Examples:
        3GPP TS 23.501
        TS 38.331
        29.244
    """

    normalized = normalize_query(
        query
    )

    matches = re.findall(
        r"\b"
        r"(?:3gpp\s*)?"
        r"(?:ts\s*|tr\s*)?"
        r"(\d{2}\.\d{3})"
        r"\b",
        normalized
    )

    return list(
        dict.fromkeys(
            matches
        )
    )


# ============================================================
# 3GPP SPEC INFERENCE
# ============================================================

def infer_3gpp_spec_candidates(
    query,
    max_specs=MAX_GPP_SPEC_CANDIDATES
):
    """
    Infer likely 3GPP specifications from
    telecom concepts.

    Users do not need to know specification numbers.
    """

    normalized = normalize_query(
        query
    )

    candidates = []

    reasons = []

    # Explicit references have highest priority.
    for spec in extract_explicit_3gpp_specs(
        query
    ):

        if spec not in candidates:

            candidates.append(
                spec
            )

            reasons.append(
                f"explicit:{spec}"
            )

    # Semantic telecom-domain rules.
    for rule in GPP_SPEC_RULES:

        matched_signals = [
            signal
            for signal in rule[
                "signals"
            ]
            if contains_signal(
                normalized,
                signal
            )
        ]

        if not matched_signals:
            continue

        for spec in rule[
            "specs"
        ]:

            if spec not in candidates:

                candidates.append(
                    spec
                )

        reasons.append(
            (
                f"{rule['name']}: "
                +
                ", ".join(
                    matched_signals
                )
            )
        )

        if len(candidates) >= max_specs:
            break

    return {
        "specs":
            candidates[
                :max_specs
            ],

        "reasons":
            reasons
    }


# ============================================================
# TCC COLLECTION ROUTING
# ============================================================

def select_tcc_collections(
    query
):
    """
    Dynamically identify TCC collections relevant
    to the query.

    This prevents all TCC collections from being
    searched unnecessarily.
    """

    normalized = normalize_query(
        query
    )

    collections = []

    reasons = []

    # --------------------------------------------------------
    # IETF / INTERNET PROTOCOLS
    # --------------------------------------------------------

    ietf_hits = [
        signal
        for signal in TCC_IETF_SIGNALS
        if contains_signal(
            normalized,
            signal
        )
    ]

    if ietf_hits:

        collections.extend([
            "IETF-RFCs",
            "IETF-Drafts"
        ])

        reasons.append(
            "IETF: "
            +
            ", ".join(
                ietf_hits
            )
        )

    # --------------------------------------------------------
    # RESEARCH
    # --------------------------------------------------------

    research_hits = [
        signal
        for signal in TCC_RESEARCH_SIGNALS
        if contains_signal(
            normalized,
            signal
        )
    ]

    if research_hits:

        collections.extend([
            "IEEE-Access",
            "OpenAlex"
        ])

        reasons.append(
            "Research: "
            +
            ", ".join(
                research_hits
            )
        )

    # --------------------------------------------------------
    # PATENTS
    # --------------------------------------------------------

    patent_hits = [
        signal
        for signal in TCC_PATENT_SIGNALS
        if contains_signal(
            normalized,
            signal
        )
    ]

    if patent_hits:

        collections.extend([
            "USPTO",
            "EPO"
        ])

        reasons.append(
            "Patents: "
            +
            ", ".join(
                patent_hits
            )
        )

    # --------------------------------------------------------
    # GENERAL TELECOM KNOWLEDGE
    # --------------------------------------------------------

    knowledge_hits = [
        signal
        for signal in TCC_KNOWLEDGE_SIGNALS
        if contains_signal(
            normalized,
            signal
        )
    ]

    if knowledge_hits:

        collections.extend([
            "Wikipedia-Telecom",
            "Wikidata-Telecom"
        ])

        reasons.append(
            "General knowledge: "
            +
            ", ".join(
                knowledge_hits
            )
        )

    # --------------------------------------------------------
    # TCC 3GPP CONTRIBUTIONS
    # --------------------------------------------------------
    #
    # These remain available when TCC itself is used for
    # a 3GPP-oriented search.
    #
    # HYBRID routing removes this collection later because
    # dedicated 3GPP already supplies standards evidence.
    # --------------------------------------------------------

    gpp_hits = [
        signal
        for signal in GPP_SOURCE_SIGNALS
        if contains_signal(
            normalized,
            signal
        )
    ]

    if gpp_hits:

        collections.append(
            "3GPP-TSG"
        )

        reasons.append(
            "TCC 3GPP contribution material"
        )

    # --------------------------------------------------------
    # GENERIC FALLBACK
    # --------------------------------------------------------

    if not collections:

        collections = [
            "Wikipedia-Telecom"
        ]

        reasons.append(
            "generic TCC fallback"
        )

    collections = list(
        dict.fromkeys(
            collections
        )
    )

    return {
        "collections":
            collections,

        "reasons":
            reasons
    }


# ============================================================
# DYNAMIC SOURCE ROUTER
# ============================================================

def route_telecom_query(
    query
):
    """
    Dynamically route to:

        3gpp
        tcc
        hybrid

    In HYBRID mode:

        - dedicated 3GPP provides standards evidence;
        - TCC provides complementary non-3GPP evidence.

    TCC 3GPP-TSG is therefore excluded from HYBRID
    retrieval to avoid redundant standards-domain scans.
    """

    normalized = normalize_query(
        query
    )

    # --------------------------------------------------------
    # DETECT 3GPP SIGNALS
    # --------------------------------------------------------

    gpp_hits = [
        signal
        for signal in GPP_SOURCE_SIGNALS
        if contains_signal(
            normalized,
            signal
        )
    ]

    # --------------------------------------------------------
    # DETECT NON-3GPP TCC SIGNALS
    # --------------------------------------------------------

    tcc_signals = (
        TCC_IETF_SIGNALS
        |
        TCC_RESEARCH_SIGNALS
        |
        TCC_PATENT_SIGNALS
        |
        TCC_KNOWLEDGE_SIGNALS
    )

    tcc_hits = [
        signal
        for signal in tcc_signals
        if contains_signal(
            normalized,
            signal
        )
    ]

    # --------------------------------------------------------
    # EXPLICIT SPECIFICATIONS
    # --------------------------------------------------------

    explicit_specs = (
        extract_explicit_3gpp_specs(
            query
        )
    )

    # --------------------------------------------------------
    # ROUTING DECISION
    # --------------------------------------------------------

    if explicit_specs:

        route = "3gpp"

    elif gpp_hits and tcc_hits:

        route = "hybrid"

    elif gpp_hits:

        route = "3gpp"

    elif tcc_hits:

        route = "tcc"

    else:

        route = "tcc"

    # --------------------------------------------------------
    # SOURCE-SPECIFIC SELECTION
    # --------------------------------------------------------

    gpp_selection = (
        infer_3gpp_spec_candidates(
            query
        )
    )

    tcc_selection = (
        select_tcc_collections(
            query
        )
    )

    # --------------------------------------------------------
    # HYBRID SOURCE DE-DUPLICATION
    # --------------------------------------------------------
    #
    # Dedicated 3GPP already provides authoritative
    # standards evidence.
    #
    # Therefore remove TCC 3GPP-TSG from the TCC side
    # of hybrid retrieval.
    # --------------------------------------------------------

    if route == "hybrid":

        tcc_selection[
            "collections"
        ] = [
            collection
            for collection
            in tcc_selection[
                "collections"
            ]
            if collection
            !=
            "3GPP-TSG"
        ]

        tcc_selection[
            "reasons"
        ] = [
            reason
            for reason
            in tcc_selection[
                "reasons"
            ]
            if reason
            !=
            "TCC 3GPP contribution material"
        ]

        # Defensive fallback only.
        #
        # A true hybrid route should already contain
        # a non-3GPP TCC collection because tcc_hits
        # must be present.

        if not tcc_selection[
            "collections"
        ]:

            tcc_selection[
                "collections"
            ] = [
                "Wikipedia-Telecom"
            ]

            tcc_selection[
                "reasons"
            ].append(
                "hybrid TCC fallback"
            )

    # --------------------------------------------------------
    # RETURN ROUTING PLAN
    # --------------------------------------------------------

    return {
        "route":
            route,

        "gpp_signal_hits":
            gpp_hits,

        "tcc_signal_hits":
            tcc_hits,

        "gpp_specs":
            gpp_selection[
                "specs"
            ],

        "gpp_reasons":
            gpp_selection[
                "reasons"
            ],

        "tcc_collections":
            tcc_selection[
                "collections"
            ],

        "tcc_reasons":
            tcc_selection[
                "reasons"
            ]
    }


# ============================================================
# STATIC ROUTER VALIDATION
# ============================================================

ROUTER_TEST_QUERIES = [

    (
        "Explain AMF registration and mobility "
        "management in a 5G Standalone network"
    ),

    (
        "Explain QUIC transport and relevant "
        "IETF RFC mechanisms"
    ),

    (
        "Explain PFCP session establishment "
        "over the N4 interface"
    ),

    (
        "Explain AMF traffic transported "
        "using QUIC"
    )
]


print("=" * 90)

print(
    "VERSION A — DYNAMIC SOURCE ROUTING VALIDATION"
)

print("=" * 90)


for test_query in ROUTER_TEST_QUERIES:

    routing = route_telecom_query(
        test_query
    )

    print(
        f"\nQuery : {test_query}"
    )

    print(
        f"Route : {routing['route'].upper()}"
    )

    print(
        f"3GPP  : {routing['gpp_specs']}"
    )

    print(
        f"TCC   : {routing['tcc_collections']}"
    )


print(
    "\n" + "=" * 90
)

print(
    "✓ Query processing loaded."
)

print(
    "✓ Telecom-aware tokenization loaded."
)

print(
    "✓ Adaptive TCC shard selection loaded."
)

print(
    "✓ Dynamic source routing loaded."
)

print(
    "✓ 3GPP specification inference loaded."
)

print(
    "✓ TCC collection routing loaded."
)

print(
    "✓ Hybrid TCC/3GPP source duplication removed."
)

print(
    "✓ No remote content was accessed."
)

print(
    "✓ Ready for Cell 5 — "
    "Remote TCC + 3GPP Retrieval Engines."
)

print("=" * 90)

VERSION A — DYNAMIC SOURCE ROUTING VALIDATION

Query : Explain AMF registration and mobility management in a 5G Standalone network
Route : 3GPP
3GPP  : ['23.501', '23.502']
TCC   : ['3GPP-TSG']

Query : Explain QUIC transport and relevant IETF RFC mechanisms
Route : TCC
3GPP  : []
TCC   : ['IETF-RFCs', 'IETF-Drafts']

Query : Explain PFCP session establishment over the N4 interface
Route : 3GPP
3GPP  : ['29.244']
TCC   : ['3GPP-TSG']

Query : Explain AMF traffic transported using QUIC
Route : HYBRID
3GPP  : ['23.501']
TCC   : ['IETF-RFCs', 'IETF-Drafts']

✓ Query processing loaded.
✓ Telecom-aware tokenization loaded.
✓ Adaptive TCC shard selection loaded.
✓ Dynamic source routing loaded.
✓ 3GPP specification inference loaded.
✓ TCC collection routing loaded.
✓ Hybrid TCC/3GPP source duplication removed.
✓ No remote content was accessed.
✓ Ready for Cell 5 — Remote TCC + 3GPP Retrieval Engines.


**Observation — Dynamic Source Selection**

- Version A separates **source selection from retrieval execution** using lightweight query-time routing. Telecom-standard queries can route directly to dedicated 3GPP specifications, non-3GPP queries can route to relevant TCC collections, and mixed-domain queries can use a **hybrid 3GPP + TCC path**.

***Key Decision:*** Use dynamic source and collection selection to avoid unnecessary remote corpus scans while retaining Version A's non-persistent architecture. Users are not required to know 3GPP specification numbers; likely specifications and relevant TCC collections are inferred from the technical concepts in the query before retrieval begins.


## **Cell 5 — Remote TCC + 3GPP Retrieval Engines**

In [ ]:
# ============================================================
# CELL 5 — REMOTE TCC + 3GPP RETRIEVAL ENGINES
# ============================================================

# ============================================================
# VERSION A RETRIEVAL PARAMETERS
# ============================================================

PER_SHARD_LIMIT = 10

TCC_SEARCH_TIMEOUT_SECONDS = 120

EARLY_TEXT_CHARS = 3000

PROXIMITY_WINDOW = 200


GPP_FETCH_TIMEOUT_SECONDS = 60

GPP_WINDOW_CHARS = 5000

GPP_WINDOW_OVERLAP = 1000

GPP_WINDOWS_PER_SPEC = 2


# ============================================================
# DUCKDB HTTPFS
# ============================================================

def load_httpfs(con):
    """
    Load HTTPFS into one DuckDB connection.
    """

    try:

        con.execute(
            "LOAD httpfs"
        )

    except Exception:

        con.execute(
            "INSTALL httpfs"
        )

        con.execute(
            "LOAD httpfs"
        )


# Bootstrap extension availability once.
_bootstrap_conn = duckdb.connect(
    database=":memory:"
)

load_httpfs(
    _bootstrap_conn
)

_bootstrap_conn.close()


# ============================================================
# TCC COLLECTION FILTER SQL
# ============================================================

def build_collection_filter_sql(
    collections
):
    """
    Build a safe SQL IN-list for selected
    TCC collections.
    """

    if not collections:

        raise ValueError(
            "At least one TCC collection "
            "must be selected."
        )

    values = [
        (
            "'"
            +
            escape_sql_literal(
                collection
            )
            +
            "'"
        )
        for collection in collections
    ]

    return (
        "("
        +
        ", ".join(
            values
        )
        +
        ")"
    )


# ============================================================
# TCC SEARCH SQL
# ============================================================

def build_tcc_search_sql(
    query,
    parquet_url,
    collections,
    limit=PER_SHARD_LIMIT
):
    """
    Build one remote TCC shard query.

    Important:
    collection filtering occurs before expensive
    lexical / phrase / proximity ranking.
    """

    features = build_query_features(
        query
    )

    normalized_query = (
        escape_sql_literal(
            features[
                "normalized_query"
            ]
        )
    )

    terms = features[
        "terms"
    ]

    bigrams = features[
        "bigrams"
    ]

    trigrams = features[
        "trigrams"
    ]

    proximity_pairs = features[
        "proximity_pairs"
    ]

    if not terms:

        raise ValueError(
            "Query contains no searchable terms."
        )

    collection_sql = (
        build_collection_filter_sql(
            collections
        )
    )

    score_parts = []

    matched_term_parts = []

    matched_phrase_parts = []

    proximity_parts = []


    # --------------------------------------------------------
    # EXACT QUERY
    # --------------------------------------------------------

    if normalized_query:

        score_parts.append(
            f"""
            CASE
                WHEN title_l
                     LIKE '%{normalized_query}%'
                    THEN 40

                WHEN early_text_l
                     LIKE '%{normalized_query}%'
                    THEN 30

                WHEN text_l
                     LIKE '%{normalized_query}%'
                    THEN 20

                ELSE 0
            END
            """
        )


    # --------------------------------------------------------
    # INDIVIDUAL TERMS
    # --------------------------------------------------------

    for term in terms:

        safe_term = (
            escape_sql_literal(
                term
            )
        )

        weight = (
            term_weight(
                term
            )
        )

        score_parts.append(
            f"""
            CASE
                WHEN title_l
                     LIKE '%{safe_term}%'
                    THEN {8 * weight}

                WHEN early_text_l
                     LIKE '%{safe_term}%'
                    THEN {5 * weight}

                WHEN text_l
                     LIKE '%{safe_term}%'
                    THEN {2 * weight}

                ELSE 0
            END
            """
        )

        matched_term_parts.append(
            f"""
            CASE
                WHEN title_l
                     LIKE '%{safe_term}%'

                  OR text_l
                     LIKE '%{safe_term}%'

                THEN 1
                ELSE 0
            END
            """
        )


    # --------------------------------------------------------
    # BIGRAMS
    # --------------------------------------------------------

    for phrase in bigrams:

        safe_phrase = (
            escape_sql_literal(
                phrase
            )
        )

        score_parts.append(
            f"""
            CASE
                WHEN title_l
                     LIKE '%{safe_phrase}%'
                    THEN 14

                WHEN early_text_l
                     LIKE '%{safe_phrase}%'
                    THEN 10

                WHEN text_l
                     LIKE '%{safe_phrase}%'
                    THEN 6

                ELSE 0
            END
            """
        )

        matched_phrase_parts.append(
            f"""
            CASE
                WHEN title_l
                     LIKE '%{safe_phrase}%'

                  OR text_l
                     LIKE '%{safe_phrase}%'

                THEN 1
                ELSE 0
            END
            """
        )


    # --------------------------------------------------------
    # TRIGRAMS
    # --------------------------------------------------------

    for phrase in trigrams:

        safe_phrase = (
            escape_sql_literal(
                phrase
            )
        )

        score_parts.append(
            f"""
            CASE
                WHEN title_l
                     LIKE '%{safe_phrase}%'
                    THEN 20

                WHEN early_text_l
                     LIKE '%{safe_phrase}%'
                    THEN 14

                WHEN text_l
                     LIKE '%{safe_phrase}%'
                    THEN 8

                ELSE 0
            END
            """
        )

        matched_phrase_parts.append(
            f"""
            CASE
                WHEN title_l
                     LIKE '%{safe_phrase}%'

                  OR text_l
                     LIKE '%{safe_phrase}%'

                THEN 1
                ELSE 0
            END
            """
        )


    # --------------------------------------------------------
    # PROXIMITY
    # --------------------------------------------------------

    for (
        term_a,
        term_b
    ) in proximity_pairs:

        safe_a = escape_sql_literal(
            re.escape(
                term_a
            )
        )

        safe_b = escape_sql_literal(
            re.escape(
                term_b
            )
        )

        proximity_parts.append(
            f"""
            CASE
                WHEN regexp_matches(
                    text_l,
                    '{safe_a}.{{0,{PROXIMITY_WINDOW}}}{safe_b}'
                )

                OR regexp_matches(
                    text_l,
                    '{safe_b}.{{0,{PROXIMITY_WINDOW}}}{safe_a}'
                )

                THEN 10
                ELSE 0
            END
            """
        )


    # --------------------------------------------------------
    # COMBINE EXPRESSIONS
    # --------------------------------------------------------

    score_expr = (
        " + ".join(
            score_parts
        )
        if score_parts
        else "0"
    )

    matched_terms_expr = (
        " + ".join(
            matched_term_parts
        )
        if matched_term_parts
        else "0"
    )

    matched_phrases_expr = (
        " + ".join(
            matched_phrase_parts
        )
        if matched_phrase_parts
        else "0"
    )

    proximity_expr = (
        " + ".join(
            proximity_parts
        )
        if proximity_parts
        else "0"
    )

    safe_url = escape_sql_literal(
        parquet_url
    )


    # --------------------------------------------------------
    # FINAL SQL
    # --------------------------------------------------------

    sql = f"""
        WITH scoped AS (

            SELECT
                identifier,
                collection,
                date,
                title,
                creator,
                text,

                lower(
                    coalesce(
                        title,
                        ''
                    )
                ) AS title_l,

                lower(
                    coalesce(
                        text,
                        ''
                    )
                ) AS text_l,

                lower(
                    substr(
                        coalesce(
                            text,
                            ''
                        ),
                        1,
                        {EARLY_TEXT_CHARS}
                    )
                ) AS early_text_l

            FROM read_parquet(
                '{safe_url}'
            )

            WHERE collection IN
                {collection_sql}
        )

        SELECT
            identifier,
            collection,
            date,
            title,
            creator,
            text,

            'TCC'
                AS source_family,

            ({matched_terms_expr})
                AS matched_terms,

            ({matched_phrases_expr})
                AS matched_phrases,

            ({proximity_expr})
                AS proximity_score,

            (
                ({score_expr})
                +
                ({proximity_expr})
            )
                AS relevance_score

        FROM scoped

        WHERE
            (
                title_l
                    LIKE '%{normalized_query}%'

                OR

                text_l
                    LIKE '%{normalized_query}%'

                OR

                ({matched_terms_expr}) > 0
            )

        ORDER BY
            relevance_score DESC,
            date DESC NULLS LAST

        LIMIT {int(limit)}
    """

    return sql


# ============================================================
# SEARCH ONE TCC SHARD
# ============================================================

def search_single_tcc_shard(
    query,
    shard,
    collections,
    limit=PER_SHARD_LIMIT
):
    """
    Search one remote TCC Parquet shard.
    """

    start_time = (
        time.perf_counter()
    )

    con = duckdb.connect(
        database=":memory:"
    )

    try:

        load_httpfs(
            con
        )

        sql = build_tcc_search_sql(
            query=
                query,

            parquet_url=
                shard[
                    "url"
                ],

            collections=
                collections,

            limit=
                limit
        )

        result_df = (
            con.execute(
                sql
            )
            .fetchdf()
        )

        elapsed = (
            time.perf_counter()
            -
            start_time
        )

        records = (
            result_df.to_dict(
                orient="records"
            )
        )

        for item in records:

            item[
                "shard_idx"
            ] = shard[
                "shard_idx"
            ]

            item[
                "source_path"
            ] = shard[
                "file_path"
            ]

            item[
                "source_shard"
            ] = shard[
                "url"
            ]

        return {
            "shard_idx":
                shard[
                    "shard_idx"
                ],

            "file_path":
                shard[
                    "file_path"
                ],

            "elapsed_s":
                elapsed,

            "results":
                records,

            "error":
                None
        }

    except Exception as exc:

        return {
            "shard_idx":
                shard[
                    "shard_idx"
                ],

            "file_path":
                shard[
                    "file_path"
                ],

            "elapsed_s":
                (
                    time.perf_counter()
                    -
                    start_time
                ),

            "results":
                [],

            "error":
                (
                    f"{type(exc).__name__}: "
                    f"{exc}"
                )
        }

    finally:

        con.close()


# ============================================================
# TCC SHARD TIMEOUT
# ============================================================

async def search_tcc_shard_with_timeout(
    query,
    shard,
    collections,
    limit=PER_SHARD_LIMIT
):

    try:

        return await asyncio.wait_for(

            asyncio.to_thread(
                search_single_tcc_shard,
                query,
                shard,
                collections,
                limit
            ),

            timeout=
                TCC_SEARCH_TIMEOUT_SECONDS
        )

    except asyncio.TimeoutError:

        return {
            "shard_idx":
                shard[
                    "shard_idx"
                ],

            "file_path":
                shard[
                    "file_path"
                ],

            "elapsed_s":
                float(
                    TCC_SEARCH_TIMEOUT_SECONDS
                ),

            "results":
                [],

            "error":
                (
                    "TimeoutError: TCC shard search "
                    f"exceeded "
                    f"{TCC_SEARCH_TIMEOUT_SECONDS}s"
                )
        }


# ============================================================
# PARALLEL TCC SEARCH
# ============================================================

async def search_selected_tcc_shards(
    query,
    selected_shards,
    collections,
    limit=PER_SHARD_LIMIT
):
    """
    Search selected TCC shards concurrently.
    """

    semaphore = asyncio.Semaphore(
        PARALLEL_SHARD_CONCURRENCY
    )

    async def run_one(
        shard
    ):

        async with semaphore:

            return await (
                search_tcc_shard_with_timeout(
                    query=
                        query,

                    shard=
                        shard,

                    collections=
                        collections,

                    limit=
                        limit
                )
            )

    tasks = [
        run_one(
            shard
        )
        for shard in selected_shards
    ]

    return await asyncio.gather(
        *tasks
    )


# ============================================================
# MERGE TCC RESULTS
# ============================================================

def merge_tcc_results(
    shard_results
):
    """
    Merge and globally rank remote TCC results.
    """

    combined = []

    for shard_result in shard_results:

        for item in shard_result[
            "results"
        ]:

            combined.append(
                dict(
                    item
                )
            )

    combined.sort(
        key=lambda item:
            float(
                item.get(
                    "relevance_score",
                    0
                )
                or 0
            ),
        reverse=True
    )

    unique = []

    seen = set()

    for item in combined:

        key = (
            str(
                item.get(
                    "collection",
                    ""
                )
            ),

            str(
                item.get(
                    "identifier",
                    ""
                )
            ),

            str(
                item.get(
                    "title",
                    ""
                )
            )
        )

        if key in seen:
            continue

        seen.add(
            key
        )

        unique.append(
            item
        )

    return unique


# ============================================================
# LEARN FROM TCC RESULTS
# ============================================================

def learn_from_tcc_results(
    query,
    ranked_results
):
    """
    Update adaptive TCC shard statistics.
    """

    for item in ranked_results:

        shard_idx = item.get(
            "shard_idx"
        )

        relevance_score = float(
            item.get(
                "relevance_score",
                0
            )
            or 0
        )

        if (
            shard_idx is None
            or relevance_score <= 0
        ):
            continue

        update_shard_term_stats(
            query=
                query,

            shard_idx=
                shard_idx,

            relevance_score=
                relevance_score
        )


# ============================================================
# INDEPENDENT TCC RETRIEVAL ENGINE
# ============================================================

async def retrieve_tcc_remote(
    query,
    collections=None,
    top_k=TOP_K_RESULTS,
    max_shards=MAX_REMOTE_SHARDS
):
    """
    Retrieve evidence from the remote
    GSMA Telco Common Corpus only.
    """

    start_time = (
        time.perf_counter()
    )

    if collections is None:

        selection = (
            select_tcc_collections(
                query
            )
        )

        collections = selection[
            "collections"
        ]

    else:

        collections = list(
            dict.fromkeys(
                collections
            )
        )

    shard_selection = (
        select_adaptive_shards(
            query=
                query,

            parquet_files=
                TCC_REMOTE_SHARDS,

            max_shards=
                max_shards
        )
    )

    selected_shards = (
        shard_selection[
            "shards"
        ]
    )

    shard_results = (
        await search_selected_tcc_shards(
            query=
                query,

            selected_shards=
                selected_shards,

            collections=
                collections,

            limit=
                PER_SHARD_LIMIT
        )
    )

    ranked_results = (
        merge_tcc_results(
            shard_results
        )
    )

    final_results = (
        ranked_results[
            :top_k
        ]
    )

    learn_from_tcc_results(
        query=
            query,

        ranked_results=
            final_results
    )

    errors = [
        item
        for item in shard_results
        if item.get(
            "error"
        )
        is not None
    ]

    return {
        "engine":
            "tcc_remote",

        "query":
            query,

        "collections":
            collections,

        "shard_selection":
            shard_selection[
                "mode"
            ],

        "selected_shards":
            selected_shards,

        "shard_results":
            shard_results,

        "shard_errors":
            len(
                errors
            ),

        "results":
            final_results,

        "result_count":
            len(
                final_results
            ),

        "retrieval_time_s":
            (
                time.perf_counter()
                -
                start_time
            )
    }


# ============================================================
# BUILD DEDICATED 3GPP SPECIFICATION REGISTRY
# ============================================================

def format_3gpp_spec_number(
    identifier
):
    """
    23501 → 23.501
    38423 → 38.423
    """

    value = str(
        identifier
    )

    match = re.match(
        r"^(\d{5})(-\d+)?$",
        value
    )

    if not match:
        return None

    base = match.group(1)

    suffix = (
        match.group(2)
        or ""
    )

    return (
        f"{base[:2]}."
        f"{base[2:]}"
        f"{suffix}"
    )


def parse_3gpp_path(
    file_path,
    url
):
    """
    Parse release and specification number
    from repository path metadata.
    """

    release_match = re.search(
        r"/Rel-(\d+)/",
        f"/{file_path}"
    )

    spec_match = re.search(
        r"/(\d{5}(?:-\d+)?)/raw\.md$",
        f"/{file_path}"
    )

    if not spec_match:
        return None

    spec_number = (
        format_3gpp_spec_number(
            spec_match.group(1)
        )
    )

    if not spec_number:
        return None

    release = (
        int(
            release_match.group(1)
        )
        if release_match
        else None
    )

    return {
        "spec_number":
            spec_number,

        "release":
            release,

        "file_path":
            file_path,

        "url":
            url
    }


GPP_SPEC_PATH_INDEX = defaultdict(
    list
)


for (
    file_path,
    url
) in zip(

    REMOTE_SOURCE_REGISTRY[
        "3gpp"
    ][
        "files"
    ],

    REMOTE_SOURCE_REGISTRY[
        "3gpp"
    ][
        "urls"
    ]
):

    metadata = parse_3gpp_path(
        file_path,
        url
    )

    if metadata:

        GPP_SPEC_PATH_INDEX[
            metadata[
                "spec_number"
            ]
        ].append(
            metadata
        )


# Newest available release first.
for spec_number in GPP_SPEC_PATH_INDEX:

    GPP_SPEC_PATH_INDEX[
        spec_number
    ].sort(
        key=lambda item:
            (
                item[
                    "release"
                ]
                if item[
                    "release"
                ]
                is not None
                else -1
            ),
        reverse=True
    )


# ============================================================
# SELECT LATEST AVAILABLE SPEC
# ============================================================

def get_latest_3gpp_spec(
    spec_number
):
    """
    Return latest available remote version of
    one 3GPP specification.
    """

    available = (
        GPP_SPEC_PATH_INDEX.get(
            spec_number,
            []
        )
    )

    if not available:
        return None

    return available[
        0
    ]


# ============================================================
# FETCH ONE DEDICATED 3GPP SPEC
# ============================================================

def fetch_3gpp_document(
    candidate
):
    """
    Fetch one selected raw.md specification.

    Content remains in memory only.
    """

    start_time = (
        time.perf_counter()
    )

    request = urllib.request.Request(
        candidate[
            "url"
        ],
        headers={
            "User-Agent":
                "Telecom-AI-MCP-Version-A"
        }
    )

    try:

        with urllib.request.urlopen(
            request,
            timeout=
                GPP_FETCH_TIMEOUT_SECONDS
        ) as response:

            raw = response.read()

            status = getattr(
                response,
                "status",
                None
            )

        text = raw.decode(
            "utf-8",
            errors="replace"
        )

        return {
            **candidate,

            "http_status":
                status,

            "bytes":
                len(
                    raw
                ),

            "text":
                text,

            "elapsed_s":
                (
                    time.perf_counter()
                    -
                    start_time
                ),

            "error":
                None
        }

    except Exception as exc:

        return {
            **candidate,

            "http_status":
                None,

            "bytes":
                0,

            "text":
                "",

            "elapsed_s":
                (
                    time.perf_counter()
                    -
                    start_time
                ),

            "error":
                (
                    f"{type(exc).__name__}: "
                    f"{exc}"
                )
        }


# ============================================================
# 3GPP TITLE EXTRACTION
# ============================================================

def extract_3gpp_title(
    text,
    spec_number
):
    """
    Extract first useful Markdown heading.
    """

    for line in str(
        text
    ).splitlines()[
        :120
    ]:

        clean = re.sub(
            r"^#+\s*",
            "",
            line
        ).strip()

        clean = re.sub(
            r"[*_~]+",
            "",
            clean
        ).strip()

        if (
            len(clean) >= 20
            and
            not clean.startswith(
                "!["
            )
            and
            "3gpp logo"
            not in clean.lower()
        ):

            return clean[
                :500
            ]

    return (
        f"3GPP Specification "
        f"{spec_number}"
    )


# ============================================================
# PYTHON LEXICAL / PHRASE / PROXIMITY SCORING
# ============================================================

def score_text_block(
    query,
    title,
    text
):
    """
    Apply the same Version A ranking principles
    to an in-memory 3GPP text window.
    """

    features = build_query_features(
        query
    )

    normalized_query = features[
        "normalized_query"
    ]

    terms = features[
        "terms"
    ]

    bigrams = features[
        "bigrams"
    ]

    trigrams = features[
        "trigrams"
    ]

    proximity_pairs = features[
        "proximity_pairs"
    ]

    title_l = str(
        title
        or ""
    ).lower()

    text_l = str(
        text
        or ""
    ).lower()

    early_text_l = text_l[
        :EARLY_TEXT_CHARS
    ]

    score = 0.0

    matched_terms = 0

    matched_phrases = 0

    proximity_score = 0.0


    # Exact query
    if normalized_query:

        if normalized_query in title_l:

            score += 40

        elif normalized_query in early_text_l:

            score += 30

        elif normalized_query in text_l:

            score += 20


    # Terms
    for term in terms:

        weight = term_weight(
            term
        )

        if (
            term in title_l
            or
            term in text_l
        ):

            matched_terms += 1

        if term in title_l:

            score += (
                8
                * weight
            )

        elif term in early_text_l:

            score += (
                5
                * weight
            )

        elif term in text_l:

            score += (
                2
                * weight
            )


    # Bigrams
    for phrase in bigrams:

        if (
            phrase in title_l
            or
            phrase in text_l
        ):

            matched_phrases += 1

        if phrase in title_l:

            score += 14

        elif phrase in early_text_l:

            score += 10

        elif phrase in text_l:

            score += 6


    # Trigrams
    for phrase in trigrams:

        if (
            phrase in title_l
            or
            phrase in text_l
        ):

            matched_phrases += 1

        if phrase in title_l:

            score += 20

        elif phrase in early_text_l:

            score += 14

        elif phrase in text_l:

            score += 8


    # Proximity
    for (
        term_a,
        term_b
    ) in proximity_pairs:

        pattern_ab = re.compile(
            re.escape(
                term_a
            )
            +
            rf".{{0,{PROXIMITY_WINDOW}}}"
            +
            re.escape(
                term_b
            ),
            flags=
                re.DOTALL
        )

        pattern_ba = re.compile(
            re.escape(
                term_b
            )
            +
            rf".{{0,{PROXIMITY_WINDOW}}}"
            +
            re.escape(
                term_a
            ),
            flags=
                re.DOTALL
        )

        if (
            pattern_ab.search(
                text_l
            )
            or
            pattern_ba.search(
                text_l
            )
        ):

            proximity_score += 10

    score += proximity_score

    return {
        "matched_terms":
            matched_terms,

        "matched_phrases":
            matched_phrases,

        "proximity_score":
            proximity_score,

        "relevance_score":
            score
    }


# ============================================================
# SPLIT + RANK ONE 3GPP DOCUMENT
# ============================================================

def rank_3gpp_document(
    query,
    document,
    windows_per_spec=GPP_WINDOWS_PER_SPEC
):
    """
    Split a selected specification into temporary
    overlapping windows and rank them.

    No window index is persisted.
    """

    text = document.get(
        "text",
        ""
    )

    if not text:
        return []

    spec_number = document[
        "spec_number"
    ]

    title = extract_3gpp_title(
        text,
        spec_number
    )

    step = max(
        1,
        (
            GPP_WINDOW_CHARS
            -
            GPP_WINDOW_OVERLAP
        )
    )

    ranked = []

    for start in range(
        0,
        len(text),
        step
    ):

        window = text[
            start:
            start
            +
            GPP_WINDOW_CHARS
        ]

        if not window.strip():
            continue

        scoring = score_text_block(
            query=
                query,

            title=
                title,

            text=
                window
        )

        if scoring[
            "relevance_score"
        ] <= 0:

            continue

        ranked.append({

            "identifier":
                spec_number,

            "collection":
                "3GPP-Specifications",

            "date":
                None,

            "title":
                title,

            "creator":
                "3GPP",

            "text":
                window,

            "source_family":
                "3GPP",

            "source_path":
                document[
                    "file_path"
                ],

            "source_shard":
                document[
                    "url"
                ],

            "release":
                document[
                    "release"
                ],

            "window_start":
                start,

            **scoring
        })

    ranked.sort(
        key=lambda item:
            float(
                item[
                    "relevance_score"
                ]
            ),
        reverse=True
    )

    return ranked[
        :windows_per_spec
    ]


# ============================================================
# INDEPENDENT 3GPP RETRIEVAL ENGINE
# ============================================================

async def retrieve_3gpp_remote(
    query,
    specs=None,
    top_k=TOP_K_RESULTS
):
    """
    Retrieve evidence from dedicated
    GSMA/3GPP raw.md specifications only.
    """

    start_time = (
        time.perf_counter()
    )

    if specs is None:

        inference = (
            infer_3gpp_spec_candidates(
                query
            )
        )

        specs = inference[
            "specs"
        ]

    specs = list(
        dict.fromkeys(
            specs
        )
    )

    selected_specs = []

    missing_specs = []

    for spec in specs:

        candidate = (
            get_latest_3gpp_spec(
                spec
            )
        )

        if candidate is None:

            missing_specs.append(
                spec
            )

            continue

        selected_specs.append(
            candidate
        )

    fetched_documents = []

    if selected_specs:

        fetched_documents = (
            await asyncio.gather(
                *[
                    asyncio.to_thread(
                        fetch_3gpp_document,
                        candidate
                    )
                    for candidate
                    in selected_specs
                ]
            )
        )

    document_errors = [
        item
        for item in fetched_documents
        if item.get(
            "error"
        )
        is not None
    ]

    ranked_results = []

    for document in fetched_documents:

        if document.get(
            "error"
        ) is not None:

            continue

        ranked_results.extend(
            rank_3gpp_document(
                query=
                    query,

                document=
                    document
            )
        )

    ranked_results.sort(
        key=lambda item:
            float(
                item.get(
                    "relevance_score",
                    0
                )
                or 0
            ),
        reverse=True
    )

    final_results = (
        ranked_results[
            :top_k
        ]
    )

    return {
        "engine":
            "3gpp_remote",

        "query":
            query,

        "requested_specs":
            specs,

        "selected_specs":
            selected_specs,

        "missing_specs":
            missing_specs,

        "fetched_documents":
            fetched_documents,

        "document_errors":
            len(
                document_errors
            ),

        "results":
            final_results,

        "result_count":
            len(
                final_results
            ),

        "retrieval_time_s":
            (
                time.perf_counter()
                -
                start_time
            )
    }


# ============================================================
# CROSS-SOURCE MERGE
# ============================================================

def merge_cross_source_results(
    result_groups,
    top_k=TOP_K_RESULTS
):
    """
    Merge evidence produced by TCC and 3GPP
    using the common Version A relevance score.
    """

    combined = []

    for group in result_groups:

        combined.extend(
            group
        )

    combined.sort(
        key=lambda item:
            float(
                item.get(
                    "relevance_score",
                    0
                )
                or 0
            ),
        reverse=True
    )

    unique = []

    seen = set()

    for item in combined:

        key = (
            str(
                item.get(
                    "source_family",
                    ""
                )
            ),

            str(
                item.get(
                    "source_path",
                    ""
                )
            ),

            str(
                item.get(
                    "identifier",
                    ""
                )
            ),

            int(
                item.get(
                    "window_start",
                    0
                )
                or 0
            )
        )

        if key in seen:
            continue

        seen.add(
            key
        )

        unique.append(
            item
        )

        if len(unique) >= top_k:
            break

    return unique


# ============================================================
# DYNAMIC VERSION A RETRIEVER
# ============================================================

async def retrieve_version_a(
    query,
    top_k=TOP_K_RESULTS
):
    """
    Unified Version A retrieval capability.

    Internally routes to:
        3GPP
        TCC
        HYBRID

    This function will later sit behind ONE MCP tool.
    """

    overall_start = (
        time.perf_counter()
    )

    routing = route_telecom_query(
        query
    )

    route = routing[
        "route"
    ]


    # ========================================================
    # 3GPP ROUTE
    # ========================================================

    if route == "3gpp":

        gpp_result = (
            await retrieve_3gpp_remote(
                query=
                    query,

                specs=
                    routing[
                        "gpp_specs"
                    ],

                top_k=
                    top_k
            )
        )

        final_results = gpp_result[
            "results"
        ]

        return {
            "query":
                query,

            "route":
                route,

            "routing":
                routing,

            "sources_searched":
                [
                    "3GPP"
                ],

            "tcc":
                None,

            "3gpp":
                gpp_result,

            "results":
                final_results,

            "result_count":
                len(
                    final_results
                ),

            "retrieval_time_s":
                (
                    time.perf_counter()
                    -
                    overall_start
                )
        }


    # ========================================================
    # TCC ROUTE
    # ========================================================

    if route == "tcc":

        tcc_result = (
            await retrieve_tcc_remote(
                query=
                    query,

                collections=
                    routing[
                        "tcc_collections"
                    ],

                top_k=
                    top_k,

                max_shards=
                    MAX_REMOTE_SHARDS
            )
        )

        final_results = tcc_result[
            "results"
        ]

        return {
            "query":
                query,

            "route":
                route,

            "routing":
                routing,

            "sources_searched":
                [
                    "TCC"
                ],

            "tcc":
                tcc_result,

            "3gpp":
                None,

            "results":
                final_results,

            "result_count":
                len(
                    final_results
                ),

            "retrieval_time_s":
                (
                    time.perf_counter()
                    -
                    overall_start
                )
        }


    # ========================================================
    # HYBRID ROUTE
    # ========================================================

    if route == "hybrid":

        (
            gpp_result,
            tcc_result
        ) = await asyncio.gather(

            retrieve_3gpp_remote(
                query=
                    query,

                specs=
                    routing[
                        "gpp_specs"
                    ],

                top_k=
                    top_k
            ),

            retrieve_tcc_remote(
                query=
                    query,

                collections=
                    routing[
                        "tcc_collections"
                    ],

                top_k=
                    top_k,

                max_shards=
                    MAX_REMOTE_SHARDS
            )
        )

        final_results = (
            merge_cross_source_results(

                result_groups=[
                    gpp_result[
                        "results"
                    ],
                    tcc_result[
                        "results"
                    ]
                ],

                top_k=
                    top_k
            )
        )

        return {
            "query":
                query,

            "route":
                route,

            "routing":
                routing,

            "sources_searched":
                [
                    "3GPP",
                    "TCC"
                ],

            "tcc":
                tcc_result,

            "3gpp":
                gpp_result,

            "results":
                final_results,

            "result_count":
                len(
                    final_results
                ),

            "retrieval_time_s":
                (
                    time.perf_counter()
                    -
                    overall_start
                )
        }


    raise RuntimeError(
        f"Unsupported Version A route: {route}"
    )


# ============================================================
# STATIC VALIDATION
# ============================================================

_test_sql = build_tcc_search_sql(
    query=(
        "Explain QUIC connection migration"
    ),

    parquet_url=
        TCC_REMOTE_SHARDS[
            0
        ][
            "url"
        ],

    collections=[
        "IETF-RFCs"
    ],

    limit=3
)


if not _test_sql.strip():

    raise RuntimeError(
        "TCC SQL generation failed."
    )


if not GPP_SPEC_PATH_INDEX:

    raise RuntimeError(
        "3GPP specification registry is empty."
    )


print("=" * 90)

print(
    "VERSION A — REMOTE RETRIEVAL ENGINES"
)

print("=" * 90)


print(
    f"TCC Remote Shards       : "
    f"{len(TCC_REMOTE_SHARDS)}"
)

print(
    f"TCC Max Shards/Search   : "
    f"{MAX_REMOTE_SHARDS}"
)

print(
    f"TCC Parallel Workers    : "
    f"{PARALLEL_SHARD_CONCURRENCY}"
)

print(
    f"TCC Per-Shard Limit     : "
    f"{PER_SHARD_LIMIT}"
)

print(
    f"TCC Timeout             : "
    f"{TCC_SEARCH_TIMEOUT_SECONDS}s"
)


print(
    "\nDedicated 3GPP"
)

print(
    f"Remote raw.md Files     : "
    f"{len(REMOTE_SOURCE_REGISTRY['3gpp']['files']):,}"
)

print(
    f"Unique Specifications   : "
    f"{len(GPP_SPEC_PATH_INDEX):,}"
)

print(
    f"Max Inferred Specs      : "
    f"{MAX_GPP_SPEC_CANDIDATES}"
)

print(
    f"Windows / Spec          : "
    f"{GPP_WINDOWS_PER_SPEC}"
)


print(
    "\nUnified Retrieval"
)

print(
    "Routes                  : "
    "3GPP / TCC / HYBRID"
)

print(
    "MCP Design              : "
    "ONE future search tool"
)


print(
    "\n" + "=" * 90
)

print(
    "✓ Independent TCC retrieval engine loaded."
)

print(
    "✓ Independent 3GPP retrieval engine loaded."
)

print(
    "✓ Dynamic Version A retriever loaded."
)

print(
    "✓ TCC collection filtering occurs before "
    "expensive lexical scoring."
)

print(
    "✓ No BM25 or persistent search index used."
)

print(
    "✓ No corpus retrieval query executed in Cell 5."
)

print(
    "✓ Ready for Cell 5A — TCC Retrieval Validation."
)

print("=" * 90)

VERSION A — REMOTE RETRIEVAL ENGINES
TCC Remote Shards       : 100
TCC Max Shards/Search   : 5
TCC Parallel Workers    : 5
TCC Per-Shard Limit     : 10
TCC Timeout             : 120s

Dedicated 3GPP
Remote raw.md Files     : 15,052
Unique Specifications   : 2,864
Max Inferred Specs      : 3
Windows / Spec          : 2

Unified Retrieval
Routes                  : 3GPP / TCC / HYBRID
MCP Design              : ONE future search tool

✓ Independent TCC retrieval engine loaded.
✓ Independent 3GPP retrieval engine loaded.
✓ Dynamic Version A retriever loaded.
✓ TCC collection filtering occurs before expensive lexical scoring.
✓ No BM25 or persistent search index used.
✓ No corpus retrieval query executed in Cell 5.
✓ Ready for Cell 5A — TCC Retrieval Validation.


### **Cell 5A — Optimized 3GPP Evidence Selection**

In [ ]:
# ============================================================
# CELL 5A — OPTIMIZED 3GPP EVIDENCE SELECTION
# ============================================================

# Purpose:
#
# Improve dedicated 3GPP retrieval quality while preserving
# Version A's low-latency direct-remote architecture.
#
# Architecture remains:
#
#     Query
#       ↓
# 3GPP Spec Inference
#       ↓
# Concurrent Remote Spec Fetch
#       ↓
# Cheap Section Shortlist
#       ↓
# Full Scoring on Shortlist Only
#       ↓
# Hit-Centred MCP-Sized Evidence
#       ↓
# Diversified Results
#
# NO:
#     - embeddings
#     - vector database
#     - persistent index
#     - additional remote request
#     - change to Cell 5
#
# This cell overrides rank_3gpp_document() only.


# ============================================================
# CONFIGURATION
# ============================================================

GPP_EVIDENCE_CHARS = min(
    2400,
    MCP_EXCERPT_CHARS
)

# Expensive scoring is restricted to this many sections
# per specification.
GPP_SECTION_SHORTLIST_PER_SPEC = 8

# Context retained before strongest local hit.
GPP_HIT_CONTEXT_BEFORE = 600

# Local evidence window used to choose the best hit.
GPP_LOCAL_WINDOW_RADIUS = 850

# Maximum candidate hit positions evaluated per section.
GPP_MAX_HIT_CANDIDATES = 24

# Require meaningful section content.
GPP_MIN_SECTION_CHARS = 80


# ============================================================
# QUERY CONCEPT GROUPS
# ============================================================

# Lightweight deterministic telecom associations.
#
# These are used only for:
#
#     - cheap section shortlisting
#     - evidence diversification
#     - hit positioning
#
# They do NOT replace the user's query.

GPP_CONCEPT_RULES = [

    {
        "name":
            "5qi_qos",

        "triggers":
            [
                "5qi",
                "qos",
                "qos flow"
            ],

        "signals":
            [
                "5qi",
                "qos flow",
                "qfi",
                "qos characteristics",
                "resource type",
                "priority level",
                "packet delay budget",
                "packet error rate",
                "averaging window",
                "maximum data burst volume"
            ]
    },

    {
        "name":
            "smf",

        "triggers":
            [
                "smf",
                "session management function"
            ],

        "signals":
            [
                "smf",
                "session management function",
                "pdu session",
                "qos flow binding",
                "pcc rule",
                "qos profile",
                "n4"
            ]
    },

    {
        "name":
            "upf",

        "triggers":
            [
                "upf",
                "user plane function"
            ],

        "signals":
            [
                "upf",
                "user plane function",
                "qer",
                "pdr",
                "far",
                "urr",
                "qos enforcement",
                "n4",
                "pfcp"
            ]
    },

    {
        "name":
            "amf",

        "triggers":
            [
                "amf",
                "access and mobility management function"
            ],

        "signals":
            [
                "amf",
                "registration management",
                "mobility management",
                "connection management",
                "ue context",
                "nas",
                "5gmm"
            ]
    },

    {
        "name":
            "authentication",

        "triggers":
            [
                "authentication",
                "ausf",
                "udm",
                "5g-aka"
            ],

        "signals":
            [
                "authentication",
                "ausf",
                "udm",
                "amf",
                "authentication vector",
                "5g-aka",
                "eap-aka"
            ]
    },

    {
        "name":
            "n4_pfcp",

        "triggers":
            [
                "n4",
                "pfcp"
            ],

        "signals":
            [
                "n4",
                "pfcp",
                "smf",
                "upf",
                "pdr",
                "far",
                "qer",
                "urr"
            ]
    },

    {
        "name":
            "network_slicing",

        "triggers":
            [
                "s-nssai",
                "network slice",
                "network slicing"
            ],

        "signals":
            [
                "s-nssai",
                "allowed nssai",
                "configured nssai",
                "nssf",
                "network slice",
                "network slicing"
            ]
    },

    {
        "name":
            "handover",

        "triggers":
            [
                "handover",
                "inter-gnb",
                "xnap"
            ],

        "signals":
            [
                "handover",
                "source gnb",
                "target gnb",
                "xnap",
                "xn",
                "path switch"
            ]
    },

    {
        "name":
            "radio_link_failure",

        "triggers":
            [
                "radio link failure",
                "rlf",
                "rrc re-establishment"
            ],

        "signals":
            [
                "radio link failure",
                "rlf",
                "rrc re-establishment",
                "radio link monitoring",
                "out-of-sync",
                "t310"
            ]
    }
]


# ============================================================
# QUERY PLAN
# ============================================================

def build_3gpp_query_plan(
    query
):
    """
    Build lightweight query signals for:
        - shortlist ranking
        - diversification
        - hit centering
    """

    normalized = normalize_query(
        query
    )


    features = build_query_features(
        query
    )


    primary_terms = list(
        dict.fromkeys(
            tokenize_query(
                query,
                remove_stopwords=True
            )
        )
    )


    active_concepts = {}


    for rule in GPP_CONCEPT_RULES:

        active = any(

            trigger
            in normalized

            for trigger
            in rule[
                "triggers"
            ]
        )


        if active:

            active_concepts[
                rule[
                    "name"
                ]
            ] = list(
                rule[
                    "signals"
                ]
            )


    # --------------------------------------------------------
    # WEIGHTED HIT SIGNALS
    # --------------------------------------------------------

    weighted_signals = []


    for phrase in features.get(
        "trigrams",
        []
    ):

        weighted_signals.append(
            (
                phrase,
                14.0
            )
        )


    for phrase in features.get(
        "bigrams",
        []
    ):

        weighted_signals.append(
            (
                phrase,
                10.0
            )
        )


    for term in primary_terms:

        weighted_signals.append(
            (
                term,
                4.0
                *
                term_weight(
                    term
                )
            )
        )


    # Concept-expansion signals deliberately receive
    # lower weight than direct query terms.
    for signals in active_concepts.values():

        for signal in signals:

            weighted_signals.append(
                (
                    signal,
                    2.0
                    if " " not in signal
                    else 3.0
                )
            )


    # Deduplicate while retaining highest signal weight.
    signal_weight_map = {}


    for signal, weight in weighted_signals:

        signal = signal.lower().strip()

        if not signal:

            continue


        signal_weight_map[
            signal
        ] = max(
            weight,
            signal_weight_map.get(
                signal,
                0.0
            )
        )


    weighted_signals = sorted(

        signal_weight_map.items(),

        key=lambda item:
            (
                item[
                    1
                ],
                len(
                    item[
                        0
                    ]
                )
            ),

        reverse=True
    )


    return {

        "normalized_query":
            normalized,

        "primary_terms":
            primary_terms,

        "bigrams":
            features.get(
                "bigrams",
                []
            ),

        "trigrams":
            features.get(
                "trigrams",
                []
            ),

        "active_concepts":
            active_concepts,

        "weighted_signals":
            weighted_signals
    }


# ============================================================
# STRUCTURAL HEADING FILTER
# ============================================================

def is_structural_3gpp_heading(
    heading
):
    """
    Exclude obvious non-technical document-management sections.
    """

    normalized = re.sub(

        r"[^a-z0-9]+",

        " ",

        str(
            heading
            or ""
        ).lower()

    ).strip()


    if not normalized:

        return True


    structural_exact = {

        "contents",
        "table of contents",
        "foreword",
        "references",
        "bibliography",
        "document history",
        "change history",
        "history"
    }


    if normalized in structural_exact:

        return True


    # Handles headings such as:
    #
    #     --- Contents
    #     Annex A Contents

    if (
        "contents"
        in normalized
        and
        len(
            normalized
        )
        <= 40
    ):

        return True


    return False


# ============================================================
# FAST MARKDOWN SECTION EXTRACTION
# ============================================================

def extract_3gpp_sections_fast(
    text
):
    """
    Parse Markdown sections in a single linear pass.

    Unlike the previous Cell 5A, no expensive ranking work
    occurs during section extraction.
    """

    if not text:

        return []


    heading_pattern = re.compile(
        r"^\s*(#{1,6})\s+(.+?)\s*$"
    )


    headings = []

    cursor = 0


    for line in text.splitlines(
        keepends=True
    ):

        match = heading_pattern.match(
            line.strip()
        )


        if match:

            headings.append({

                "heading":
                    match.group(
                        2
                    ).strip(),

                "level":
                    len(
                        match.group(
                            1
                        )
                    ),

                "start":
                    cursor,

                "body_start":
                    (
                        cursor
                        +
                        len(
                            line
                        )
                    )
            })


        cursor += len(
            line
        )


    # --------------------------------------------------------
    # FALLBACK
    # --------------------------------------------------------

    if not headings:

        return [

            {
                "heading":
                    "Document",

                "level":
                    1,

                "start":
                    0,

                "end":
                    len(
                        text
                    ),

                "body":
                    text
            }
        ]


    sections = []


    for index, item in enumerate(
        headings
    ):

        end = (

            headings[
                index + 1
            ][
                "start"
            ]

            if (
                index + 1
                <
                len(
                    headings
                )
            )

            else len(
                text
            )
        )


        body = text[
            item[
                "body_start"
            ]
            :
            end
        ].strip()


        if (
            len(
                body
            )
            <
            GPP_MIN_SECTION_CHARS
        ):

            continue


        if is_structural_3gpp_heading(
            item[
                "heading"
            ]
        ):

            continue


        sections.append({

            "heading":
                item[
                    "heading"
                ],

            "level":
                item[
                    "level"
                ],

            "start":
                item[
                    "start"
                ],

            "end":
                end,

            "body":
                body
        })


    return sections


# ============================================================
# CHEAP SECTION SCORE
# ============================================================

def cheap_3gpp_section_score(
    section,
    query_plan
):
    """
    Very cheap first-stage section ranking.

    IMPORTANT:
    This deliberately does NOT call score_text_block().

    It uses basic string membership/counts only.
    """

    heading = section[
        "heading"
    ].lower()


    body = section[
        "body"
    ].lower()


    score = 0.0


    matched_primary = set()

    matched_concepts = set()


    # --------------------------------------------------------
    # PRIMARY TERMS
    # --------------------------------------------------------

    for term in query_plan[
        "primary_terms"
    ]:

        term_lower = term.lower()


        if term_lower in heading:

            score += (
                8.0
                *
                term_weight(
                    term
                )
            )

            matched_primary.add(
                term
            )


        if term_lower in body:

            # Cap frequency contribution.
            count = min(
                body.count(
                    term_lower
                ),
                3
            )


            score += (
                count
                *
                2.0
                *
                term_weight(
                    term
                )
            )


            matched_primary.add(
                term
            )


    # --------------------------------------------------------
    # PRIMARY PHRASES
    # --------------------------------------------------------

    for phrase in query_plan[
        "trigrams"
    ]:

        phrase_lower = phrase.lower()


        if phrase_lower in heading:

            score += 14.0


        if phrase_lower in body:

            score += 9.0


    for phrase in query_plan[
        "bigrams"
    ]:

        phrase_lower = phrase.lower()


        if phrase_lower in heading:

            score += 10.0


        if phrase_lower in body:

            score += 6.0


    # --------------------------------------------------------
    # CONCEPT GROUPS
    # --------------------------------------------------------

    for (
        concept_name,
        signals
    ) in query_plan[
        "active_concepts"
    ].items():

        concept_matched = False


        for signal in signals:

            signal_lower = signal.lower()


            if signal_lower in heading:

                score += 8.0

                concept_matched = True


            elif signal_lower in body:

                score += (
                    3.0
                    if " " in signal_lower
                    else 2.0
                )

                concept_matched = True


        if concept_matched:

            matched_concepts.add(
                concept_name
            )


    # Strong reward for sections addressing several
    # requested aspects simultaneously.
    score += (
        10.0
        *
        len(
            matched_concepts
        )
    )


    primary_coverage = (

        len(
            matched_primary
        )
        /
        max(
            1,
            len(
                query_plan[
                    "primary_terms"
                ]
            )
        )
    )


    score += (
        12.0
        *
        primary_coverage
    )


    return {

        "cheap_score":
            score,

        "matched_primary":
            matched_primary,

        "matched_concepts":
            matched_concepts,

        "primary_coverage":
            primary_coverage
    }


# ============================================================
# DIVERSIFIED CHEAP SHORTLIST
# ============================================================

def shortlist_3gpp_sections(
    sections,
    query_plan,
    limit=
        GPP_SECTION_SHORTLIST_PER_SPEC
):
    """
    Select a small set of promising sections before running
    the expensive Version A lexical/proximity scorer.

    Concept diversity is considered during shortlisting.
    """

    candidates = []


    for section in sections:

        metrics = cheap_3gpp_section_score(
            section,
            query_plan
        )


        if (
            metrics[
                "cheap_score"
            ]
            <= 0
        ):

            continue


        candidate = dict(
            section
        )


        candidate[
            "_cheap_score"
        ] = metrics[
            "cheap_score"
        ]


        candidate[
            "_matched_primary"
        ] = metrics[
            "matched_primary"
        ]


        candidate[
            "_matched_concepts"
        ] = metrics[
            "matched_concepts"
        ]


        candidate[
            "_primary_coverage"
        ] = metrics[
            "primary_coverage"
        ]


        candidates.append(
            candidate
        )


    if not candidates:

        return []


    candidates.sort(

        key=lambda item:
            item[
                "_cheap_score"
            ],

        reverse=True
    )


    selected = []

    selected_keys = set()


    # --------------------------------------------------------
    # FIRST: ensure active concepts have representation
    # --------------------------------------------------------

    for concept_name in query_plan[
        "active_concepts"
    ]:

        concept_candidates = [

            candidate

            for candidate
            in candidates

            if (
                concept_name
                in candidate[
                    "_matched_concepts"
                ]
            )
            and
            candidate[
                "start"
            ]
            not in selected_keys
        ]


        if concept_candidates:

            best = concept_candidates[
                0
            ]


            selected.append(
                best
            )


            selected_keys.add(
                best[
                    "start"
                ]
            )


        if (
            len(
                selected
            )
            >=
            limit
        ):

            break


    # --------------------------------------------------------
    # THEN: fill remaining shortlist by cheap relevance
    # --------------------------------------------------------

    for candidate in candidates:

        if (
            len(
                selected
            )
            >=
            limit
        ):

            break


        if (
            candidate[
                "start"
            ]
            in selected_keys
        ):

            continue


        selected.append(
            candidate
        )


        selected_keys.add(
            candidate[
                "start"
            ]
        )


    return selected[
        :limit
    ]


# ============================================================
# FAST HIT-CENTRE SELECTION
# ============================================================

def find_best_3gpp_evidence_position(
    body,
    query_plan
):
    """
    Select the best evidence position using a bounded number
    of local candidate windows.

    This replaces the previous O(n²) hit-density calculation.
    """

    if not body:

        return 0


    lowered = body.lower()


    candidate_positions = []


    # --------------------------------------------------------
    # FIND BOUNDED CANDIDATE HITS
    # --------------------------------------------------------

    for (
        signal,
        weight
    ) in query_plan[
        "weighted_signals"
    ]:

        search_start = 0

        occurrences = 0


        while occurrences < 2:

            position = lowered.find(
                signal,
                search_start
            )


            if position < 0:

                break


            candidate_positions.append(
                (
                    position,
                    weight
                )
            )


            occurrences += 1


            search_start = (
                position
                +
                max(
                    1,
                    len(
                        signal
                    )
                )
            )


            if (
                len(
                    candidate_positions
                )
                >=
                GPP_MAX_HIT_CANDIDATES
            ):

                break


        if (
            len(
                candidate_positions
            )
            >=
            GPP_MAX_HIT_CANDIDATES
        ):

            break


    if not candidate_positions:

        return 0


    best_position = (
        candidate_positions[
            0
        ][
            0
        ]
    )


    best_score = -1.0


    # --------------------------------------------------------
    # SCORE SMALL LOCAL WINDOWS ONLY
    # --------------------------------------------------------

    for (
        position,
        direct_weight
    ) in candidate_positions:

        start = max(
            0,
            position
            -
            GPP_LOCAL_WINDOW_RADIUS
        )


        end = min(
            len(
                lowered
            ),
            position
            +
            GPP_LOCAL_WINDOW_RADIUS
        )


        local_text = lowered[
            start:end
        ]


        local_score = (
            direct_weight
        )


        for (
            signal,
            weight
        ) in query_plan[
            "weighted_signals"
        ]:

            if signal in local_text:

                local_score += weight


        if local_score > best_score:

            best_score = local_score

            best_position = position


    return int(
        best_position
    )


# ============================================================
# HIT-CENTRED MCP-SIZED EVIDENCE
# ============================================================

def build_3gpp_evidence_excerpt(
    heading,
    body,
    hit_position
):
    """
    Build evidence already sized for the MCP evidence payload.
    """

    heading = (
        heading
        or
        "3GPP Section"
    ).strip()


    prefix = (
        f"Section: {heading}\n\n"
    )


    available_chars = max(
        400,
        GPP_EVIDENCE_CHARS
        -
        len(
            prefix
        )
    )


    # If the whole section fits, preserve it.
    if (
        len(
            body
        )
        <=
        available_chars
    ):

        return (
            prefix
            +
            body.strip()
        )[:GPP_EVIDENCE_CHARS]


    hit_position = max(
        0,
        min(
            int(
                hit_position
            ),
            len(
                body
            )
        )
    )


    start = max(
        0,
        hit_position
        -
        GPP_HIT_CONTEXT_BEFORE
    )


    end = min(
        len(
            body
        ),
        start
        +
        available_chars
    )


    # Shift backwards when approaching section end.
    if (
        end
        -
        start
        <
        available_chars
    ):

        start = max(
            0,
            end
            -
            available_chars
        )


    excerpt = body[
        start:end
    ].strip()


    if start > 0:

        excerpt = (
            "... "
            +
            excerpt
        )


    if end < len(
        body
    ):

        excerpt += " ..."


    return (
        prefix
        +
        excerpt
    )[:GPP_EVIDENCE_CHARS]


# ============================================================
# FULL SCORING — SHORTLIST ONLY
# ============================================================

def fully_score_3gpp_section(
    query,
    section,
    query_plan
):
    """
    Apply the existing Version A quality scorer only after
    cheap shortlisting.
    """

    base_score = score_text_block(

        query=
            query,

        title=
            section[
                "heading"
            ],

        text=
            section[
                "body"
            ]
    )


    matched_concepts = set(
        section.get(
            "_matched_concepts",
            set()
        )
    )


    primary_coverage = float(
        section.get(
            "_primary_coverage",
            0.0
        )
    )


    # Preserve existing lexical score while rewarding:
    #
    #     - concept breadth
    #     - primary-query coverage
    #
    # This is a local re-rank only.

    relevance_score = (

        float(
            base_score.get(
                "relevance_score",
                0
            )
            or 0
        )

        +
        (
            8.0
            *
            len(
                matched_concepts
            )
        )

        +
        (
            15.0
            *
            primary_coverage
        )
    )


    return {

        "matched_terms":
            int(
                base_score.get(
                    "matched_terms",
                    0
                )
                or 0
            ),

        "matched_phrases":
            int(
                base_score.get(
                    "matched_phrases",
                    0
                )
                or 0
            ),

        "proximity_score":
            float(
                base_score.get(
                    "proximity_score",
                    0
                )
                or 0
            ),

        "relevance_score":
            relevance_score,

        "primary_coverage":
            primary_coverage,

        "matched_concepts":
            matched_concepts
    }


# ============================================================
# DIVERSIFIED FINAL SELECTION
# ============================================================

def select_diverse_3gpp_results(
    candidates,
    limit
):
    """
    Select high-quality sections while rewarding evidence
    that adds a previously uncovered requested concept.
    """

    if not candidates:

        return []


    remaining = sorted(

        candidates,

        key=lambda item:
            item[
                "relevance_score"
            ],

        reverse=True
    )


    selected = []

    covered_concepts = set()

    used_section_keys = set()


    while (
        remaining
        and
        len(
            selected
        )
        <
        limit
    ):

        best_item = None

        best_adjusted_score = None


        for item in remaining:

            section_key = item[
                "_section_key"
            ]


            if section_key in used_section_keys:

                continue


            concepts = set(
                item.get(
                    "_matched_concepts",
                    set()
                )
            )


            new_concepts = (
                concepts
                -
                covered_concepts
            )


            adjusted_score = (

                item[
                    "relevance_score"
                ]

                +

                (
                    12.0
                    *
                    len(
                        new_concepts
                    )
                )
            )


            if (
                best_adjusted_score is None
                or
                adjusted_score
                >
                best_adjusted_score
            ):

                best_item = item

                best_adjusted_score = (
                    adjusted_score
                )


        if best_item is None:

            break


        selected.append(
            best_item
        )


        used_section_keys.add(
            best_item[
                "_section_key"
            ]
        )


        covered_concepts.update(
            best_item.get(
                "_matched_concepts",
                set()
            )
        )


        remaining.remove(
            best_item
        )


    return selected


# ============================================================
# REPLACEMENT DOCUMENT RANKER
# ============================================================

def rank_3gpp_document(
    query,
    document,
    windows_per_spec=
        GPP_WINDOWS_PER_SPEC
):
    """
    Optimized section-aware 3GPP document ranker.

    Critical optimization:

        Full lexical / phrase / proximity scoring
        executes only on the cheap shortlist.

    Return structure remains compatible with the existing
    retrieve_3gpp_remote() function in Cell 5.
    """

    if not isinstance(
        document,
        dict
    ):

        return []


    if document.get(
        "error"
    ):

        return []


    text = str(
        document.get(
            "text",
            ""
        )
        or ""
    )


    if not text.strip():

        return []


    # --------------------------------------------------------
    # METADATA
    # --------------------------------------------------------

    spec_number = str(
        document.get(
            "spec_number",
            ""
        )
        or ""
    )


    release = str(
        document.get(
            "release",
            ""
        )
        or ""
    )


    source_path = str(

        document.get(
            "file_path",
            document.get(
                "source_path",
                ""
            )
        )

        or ""
    )


    source_url = str(
        document.get(
            "url",
            ""
        )
        or ""
    )


    # Existing Cell 5 helper requires:
    #
    #     extract_3gpp_title(text, spec_number)

    document_title = (

        extract_3gpp_title(
            text,
            spec_number
        )

        or

        (
            f"3GPP TS/TR "
            f"{spec_number}"
        )
    )


    # --------------------------------------------------------
    # QUERY PLAN
    # --------------------------------------------------------

    query_plan = (
        build_3gpp_query_plan(
            query
        )
    )


    # --------------------------------------------------------
    # FAST SECTION EXTRACTION
    # --------------------------------------------------------

    sections = (
        extract_3gpp_sections_fast(
            text
        )
    )


    # --------------------------------------------------------
    # CHEAP SHORTLIST
    # --------------------------------------------------------

    shortlisted_sections = (
        shortlist_3gpp_sections(

            sections=
                sections,

            query_plan=
                query_plan,

            limit=
                GPP_SECTION_SHORTLIST_PER_SPEC
        )
    )


    if not shortlisted_sections:

        return []


    # --------------------------------------------------------
    # FULL QUALITY SCORING — SHORTLIST ONLY
    # --------------------------------------------------------

    candidates = []


    for section in shortlisted_sections:

        score = (
            fully_score_3gpp_section(

                query=
                    query,

                section=
                    section,

                query_plan=
                    query_plan
            )
        )


        if (
            score[
                "relevance_score"
            ]
            <= 0
        ):

            continue


        # ----------------------------------------------------
        # FAST LOCAL HIT POSITION
        # ----------------------------------------------------

        hit_position = (
            find_best_3gpp_evidence_position(

                body=
                    section[
                        "body"
                    ],

                query_plan=
                    query_plan
            )
        )


        # ----------------------------------------------------
        # MCP-ALIGNED PASSAGE
        # ----------------------------------------------------

        evidence_text = (
            build_3gpp_evidence_excerpt(

                heading=
                    section[
                        "heading"
                    ],

                body=
                    section[
                        "body"
                    ],

                hit_position=
                    hit_position
            )
        )


        candidates.append({

            # ------------------------------------------------
            # EXISTING RETRIEVAL CONTRACT
            # ------------------------------------------------

            "identifier":
                spec_number,

            "collection":
                "3GPP",

            "date":
                "",

            "title":
                document_title,

            "creator":
                "3GPP",

            "text":
                evidence_text,

            "source_family":
                "3GPP",

            "source_path":
                source_path,

            "source_shard":
                source_url,

            "release":
                release,

            "window_start":
                int(
                    section[
                        "start"
                    ]
                ),

            "matched_terms":
                score[
                    "matched_terms"
                ],

            "matched_phrases":
                score[
                    "matched_phrases"
                ],

            "proximity_score":
                score[
                    "proximity_score"
                ],

            "relevance_score":
                score[
                    "relevance_score"
                ],

            # ------------------------------------------------
            # QUALITY DIAGNOSTICS
            # ------------------------------------------------

            "section_heading":
                section[
                    "heading"
                ],

            "primary_coverage":
                round(
                    score[
                        "primary_coverage"
                    ],
                    4
                ),

            # ------------------------------------------------
            # INTERNAL
            # ------------------------------------------------

            "_section_key":
                (
                    f"{spec_number}|"
                    f"{section['start']}|"
                    f"{section['heading']}"
                ),

            "_matched_concepts":
                score[
                    "matched_concepts"
                ]
        })


    # --------------------------------------------------------
    # FINAL DIVERSIFICATION
    # --------------------------------------------------------

    selected = (
        select_diverse_3gpp_results(

            candidates=
                candidates,

            limit=
                max(
                    1,
                    int(
                        windows_per_spec
                    )
                )
        )
    )


    # --------------------------------------------------------
    # REMOVE INTERNAL FIELDS
    # --------------------------------------------------------

    clean_results = []


    for item in selected:

        clean_item = {

            key:
                value

            for (
                key,
                value
            )
            in item.items()

            if not key.startswith(
                "_"
            )
        }


        clean_results.append(
            clean_item
        )


    return clean_results


# ============================================================
# STATIC SELF-TEST
# ============================================================

_TEST_GPP_MARKDOWN = """
# --- Contents

5.7.1 QoS Flow
5.7.4 5QI
6.2 SMF
6.3 UPF

# 5 QoS Architecture

General QoS architecture.

## 5.7.1.1 QoS Flow

The 5G QoS model is based on QoS Flows. A QFI identifies
a QoS Flow within a PDU Session.

## 5.7.4 5QI Characteristics

A 5QI represents 5G QoS characteristics including resource
type, priority level, packet delay budget and packet error rate.

## 6.2 SMF QoS Control

The SMF performs QoS Flow binding and derives QoS parameters.
It provides QoS information toward the RAN and controls the UPF.

## 6.3 UPF QoS Enforcement

The UPF applies user-plane QoS enforcement according to
information provided by the SMF. QER and packet detection
rules are applied to user-plane traffic.
"""


_test_document = {

    "spec_number":
        "23.501",

    "release":
        "Rel-Test",

    "file_path":
        "marked/Rel-Test/23_series/23501/raw.md",

    "url":
        "https://example.invalid/23501/raw.md",

    "text":
        _TEST_GPP_MARKDOWN,

    "error":
        None
}


_test_query = (
    "Explain how 5QI characterizes QoS and describe "
    "the roles of the SMF and UPF in QoS flows."
)


_test_sections = (
    extract_3gpp_sections_fast(
        _TEST_GPP_MARKDOWN
    )
)


_test_ranked = (
    rank_3gpp_document(

        query=
            _test_query,

        document=
            _test_document,

        windows_per_spec=
            3
    )
)


if not _test_sections:

    raise RuntimeError(
        "3GPP section extraction self-test failed."
    )


if not _test_ranked:

    raise RuntimeError(
        "3GPP optimized ranking self-test failed."
    )


if any(

    "contents"
    in str(
        item.get(
            "section_heading",
            ""
        )
    ).lower()

    for item
    in _test_ranked
):

    raise RuntimeError(
        "Structural Contents filtering self-test failed."
    )


if any(

    len(
        str(
            item.get(
                "text",
                ""
            )
        )
    )
    >
    GPP_EVIDENCE_CHARS

    for item
    in _test_ranked
):

    raise RuntimeError(
        "3GPP evidence-size self-test failed."
    )


# ============================================================
# SUMMARY
# ============================================================

print("=" * 90)

print(
    "VERSION A — OPTIMIZED 3GPP EVIDENCE SELECTION"
)

print("=" * 90)


print(
    f"Evidence Limit       : "
    f"{GPP_EVIDENCE_CHARS:,} chars"
)

print(
    f"Sections Shortlisted : "
    f"{GPP_SECTION_SHORTLIST_PER_SPEC}/spec max"
)

print(
    f"Hit Candidates       : "
    f"{GPP_MAX_HIT_CANDIDATES}/section max"
)

print(
    f"Self-Test Sections   : "
    f"{len(_test_sections)}"
)

print(
    f"Self-Test Results    : "
    f"{len(_test_ranked)}"
)


print(
    "\n" + "=" * 90
)

print(
    "✓ Cell 5 retrieval architecture unchanged."
)

print(
    "✓ Remote 3GPP fetch architecture unchanged."
)

print(
    "✓ 3GPP specification inference unchanged."
)

print(
    "✓ Cheap first-stage section shortlist enabled."
)

print(
    "✓ Full lexical/proximity scoring restricted to shortlist."
)

print(
    "✓ O(n²) hit-density calculation removed."
)

print(
    "✓ MCP-sized hit-centred evidence retained."
)

print(
    "✓ Concept-aware evidence diversification retained."
)

print(
    "✓ Structural Contents sections excluded."
)

print(
    "✓ No embeddings or persistent index introduced."
)

print(
    "✓ No remote request executed in this cell."
)

print(
    "✓ Optimized rank_3gpp_document() active."
)

print("=" * 90)

VERSION A — OPTIMIZED 3GPP EVIDENCE SELECTION
Evidence Limit       : 2,400 chars
Sections Shortlisted : 8/spec max
Hit Candidates       : 24/section max
Self-Test Sections   : 4
Self-Test Results    : 3

✓ Cell 5 retrieval architecture unchanged.
✓ Remote 3GPP fetch architecture unchanged.
✓ 3GPP specification inference unchanged.
✓ Cheap first-stage section shortlist enabled.
✓ Full lexical/proximity scoring restricted to shortlist.
✓ O(n²) hit-density calculation removed.
✓ MCP-sized hit-centred evidence retained.
✓ Concept-aware evidence diversification retained.
✓ Structural Contents sections excluded.
✓ No embeddings or persistent index introduced.
✓ No remote request executed in this cell.
✓ Optimized rank_3gpp_document() active.


**Observation — Remote Retrieval Engine**

- Version A implements independent remote retrieval engines for **TCC and dedicated 3GPP documentation**, while preserving the original **lexical, phrase and proximity retrieval** approach.
- TCC retrieval dynamically filters relevant collections before searching a maximum of five adaptively selected remote Parquet shards, while 3GPP retrieval dynamically infers and accesses relevant specifications directly from the remote source.

***Key Decision:*** Align Version A with Version B on available telecom source scope while preserving Version A's defining **non-indexed, query-time remote retrieval architecture**. Keep TCC and 3GPP retrieval independently executable so each source can be validated separately before dynamic routing selects **3GPP, TCC or a hybrid of both**.


## **Cell 6 — Retrieval Smoke Test**

### **Cell 6A — TCC Retrieval Validation**

In [ ]:
# ============================================================
# CELL 6A — TCC RETRIEVAL VALIDATION
# ============================================================

# Purpose:
#
# Validate the remote TCC retrieval engine independently
# from dedicated 3GPP and from the dynamic source router.
#
# This first validation intentionally uses:
#
#   - TCC only
#   - 1 remote Parquet shard
#   - IETF-RFCs only
#   - lexical / phrase / proximity retrieval
#
# The objective is to isolate the TCC search path and measure
# whether collection-filtered remote retrieval works correctly.


# ============================================================
# VALIDATION CONFIGURATION
# ============================================================

TCC_VALIDATION_QUERY = (
    "IETF RFC informative references"
)

TCC_VALIDATION_COLLECTIONS = [
    "IETF-RFCs"
]

TCC_VALIDATION_MAX_SHARDS = 1

TCC_VALIDATION_TOP_K = 5


# ============================================================
# EXECUTE TCC-ONLY RETRIEVAL
# ============================================================

print("=" * 90)
print(
    "VERSION A — TCC RETRIEVAL VALIDATION"
)
print("=" * 90)

print(
    f"Query               : "
    f"{TCC_VALIDATION_QUERY}"
)

print(
    f"Source              : "
    f"TCC ONLY"
)

print(
    f"Collection          : "
    f"{TCC_VALIDATION_COLLECTIONS}"
)

print(
    f"Maximum Shards      : "
    f"{TCC_VALIDATION_MAX_SHARDS}"
)

print(
    f"Top-K               : "
    f"{TCC_VALIDATION_TOP_K}"
)

print(
    "\nExecuting remote TCC retrieval..."
)


tcc_validation_result = (
    await retrieve_tcc_remote(

        query=
            TCC_VALIDATION_QUERY,

        collections=
            TCC_VALIDATION_COLLECTIONS,

        top_k=
            TCC_VALIDATION_TOP_K,

        max_shards=
            TCC_VALIDATION_MAX_SHARDS
    )
)


# ============================================================
# EXTRACT VALIDATION METRICS
# ============================================================

tcc_validation_results = (
    tcc_validation_result[
        "results"
    ]
)

tcc_validation_shards = (
    tcc_validation_result[
        "selected_shards"
    ]
)

tcc_validation_shard_results = (
    tcc_validation_result[
        "shard_results"
    ]
)

tcc_validation_errors = [
    item
    for item
    in tcc_validation_shard_results
    if item.get(
        "error"
    )
    is not None
]


# ============================================================
# SUMMARY
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "TCC RETRIEVAL SUMMARY"
)

print("=" * 90)


print(
    f"Retrieval Engine    : "
    f"{tcc_validation_result['engine']}"
)

print(
    f"Collections         : "
    f"{tcc_validation_result['collections']}"
)

print(
    f"Shard Selection     : "
    f"{tcc_validation_result['shard_selection']}"
)

print(
    f"Shards Searched     : "
    f"{len(tcc_validation_shards)}"
)

print(
    f"Results Returned    : "
    f"{tcc_validation_result['result_count']}"
)

print(
    f"Shard Errors        : "
    f"{len(tcc_validation_errors)}"
)

print(
    f"Retrieval Latency   : "
    f"{tcc_validation_result['retrieval_time_s']:.3f} sec"
)


# ============================================================
# SELECTED SHARD
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "SELECTED REMOTE TCC SHARD"
)

print("=" * 90)


for shard in tcc_validation_shards:

    print(
        f"Shard Index : "
        f"{shard['shard_idx']}"
    )

    print(
        f"File        : "
        f"{shard['file_path']}"
    )

    print(
        f"URL         : "
        f"{shard['url']}"
    )


# ============================================================
# SHARD EXECUTION DETAILS
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "REMOTE SHARD EXECUTION"
)

print("=" * 90)


for item in tcc_validation_shard_results:

    status = (
        "PASS"
        if item.get(
            "error"
        )
        is None
        else "ERROR"
    )

    print(
        f"File       : "
        f"{item['file_path']}"
    )

    print(
        f"Latency    : "
        f"{item['elapsed_s']:.3f} sec"
    )

    print(
        f"Status     : "
        f"{status}"
    )

    print(
        f"Candidates : "
        f"{len(item['results'])}"
    )

    if item.get(
        "error"
    ) is not None:

        print(
            f"Error      : "
            f"{item['error']}"
        )


# ============================================================
# RESULT TABLE
# ============================================================

result_rows = []


for rank, item in enumerate(
    tcc_validation_results,
    start=1
):

    result_rows.append({

        "rank":
            rank,

        "source_family":
            item.get(
                "source_family",
                ""
            ),

        "collection":
            item.get(
                "collection",
                ""
            ),

        "identifier":
            item.get(
                "identifier",
                ""
            ),

        "title":
            item.get(
                "title",
                ""
            ),

        "matched_terms":
            item.get(
                "matched_terms",
                0
            ),

        "matched_phrases":
            item.get(
                "matched_phrases",
                0
            ),

        "proximity_score":
            item.get(
                "proximity_score",
                0
            ),

        "relevance_score":
            item.get(
                "relevance_score",
                0
            )
    })


tcc_validation_df = (
    pd.DataFrame(
        result_rows
    )
)


print(
    "\n" + "=" * 90
)

print(
    "TCC RANKED RESULTS"
)

print("=" * 90)


if not tcc_validation_df.empty:

    display(
        tcc_validation_df
    )

else:

    print(
        "No ranked TCC results returned."
    )


# ============================================================
# EVIDENCE PREVIEW
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "TCC EVIDENCE PREVIEW"
)

print("=" * 90)


for rank, item in enumerate(
    tcc_validation_results,
    start=1
):

    evidence = str(
        item.get(
            "text",
            ""
        )
    ).strip()


    preview = (
        evidence[
            :1000
        ]
    )


    if len(evidence) > 1000:

        preview += (
            " ..."
        )


    print(
        f"\n--- RESULT {rank} ---"
    )

    print(
        f"Collection : "
        f"{item.get('collection', '')}"
    )

    print(
        f"Identifier : "
        f"{item.get('identifier', '')}"
    )

    print(
        f"Title      : "
        f"{item.get('title', '')}"
    )

    print(
        f"Score      : "
        f"{float(item.get('relevance_score', 0) or 0):.2f}"
    )

    print(
        "\nEvidence:"
    )

    print(
        preview
    )


# ============================================================
# STRICT VALIDATION
# ============================================================

tcc_validation_checks = {

    "correct_engine":
        (
            tcc_validation_result[
                "engine"
            ]
            ==
            "tcc_remote"
        ),

    "single_shard_only":
        (
            len(
                tcc_validation_shards
            )
            ==
            1
        ),

    "correct_collection":
        (
            tcc_validation_result[
                "collections"
            ]
            ==
            TCC_VALIDATION_COLLECTIONS
        ),

    "no_shard_errors":
        (
            len(
                tcc_validation_errors
            )
            ==
            0
        ),

    "results_returned":
        (
            tcc_validation_result[
                "result_count"
            ]
            > 0
        ),

    "top_k_respected":
        (
            tcc_validation_result[
                "result_count"
            ]
            <=
            TCC_VALIDATION_TOP_K
        ),

    "tcc_source_only":
        all(
            item.get(
                "source_family"
            )
            ==
            "TCC"

            for item
            in tcc_validation_results
        ),

    "ietf_collection_only":
        all(
            item.get(
                "collection"
            )
            ==
            "IETF-RFCs"

            for item
            in tcc_validation_results
        )
}


print(
    "\n" + "=" * 90
)

print(
    "TCC VALIDATION CHECKS"
)

print("=" * 90)


for (
    check,
    passed
) in tcc_validation_checks.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{check}"
    )


tcc_validation_passed = (
    all(
        tcc_validation_checks.values()
    )
)


print(
    "\n" + "=" * 90
)


if tcc_validation_passed:

    print(
        "✓ VERSION A TCC RETRIEVAL VALIDATION PASSED"
    )

    print(
        "✓ Remote TCC Parquet retrieval is operational."
    )

    print(
        "✓ Collection-filtered retrieval is operational."
    )

    print(
        "✓ Lexical / phrase / proximity ranking is operational."
    )

    print(
        "✓ Ready for Cell 6B — "
        "Dedicated 3GPP Retrieval Validation."
    )


else:

    print(
        "⚠ VERSION A TCC RETRIEVAL VALIDATION "
        "REQUIRES REVIEW"
    )

    print(
        "Do not proceed to dynamic routing validation "
        "until the TCC retrieval path is understood."
    )


print("=" * 90)

VERSION A — TCC RETRIEVAL VALIDATION
Query               : IETF RFC informative references
Source              : TCC ONLY
Collection          : ['IETF-RFCs']
Maximum Shards      : 1
Top-K               : 5

Executing remote TCC retrieval...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


TCC RETRIEVAL SUMMARY
Retrieval Engine    : tcc_remote
Collections         : ['IETF-RFCs']
Shard Selection     : SPREAD
Shards Searched     : 1
Results Returned    : 5
Shard Errors        : 0
Retrieval Latency   : 2.932 sec

SELECTED REMOTE TCC SHARD
Shard Index : 0
File        : data/tcc_001.parquet
URL         : https://huggingface.co/datasets/GSMA/Telco-Common-Corpus/resolve/main/data/tcc_001.parquet

REMOTE SHARD EXECUTION
File       : data/tcc_001.parquet
Latency    : 2.889 sec
Status     : PASS
Candidates : 10

TCC RANKED RESULTS


,rank,source_family,collection,identifier,title,matched_terms,matched_phrases,proximity_score,relevance_score
0,1,TCC,IETF-RFCs,RFC6174,Definition of IETF Working Group Document States,4,1,20,76.0
1,2,TCC,IETF-RFCs,RFC9911,Common YANG Data Types,4,1,20,70.0
2,3,TCC,IETF-RFCs,RFC9647,October 2024,4,1,20,70.0
3,4,TCC,IETF-RFCs,RFC9062,Google,4,1,20,70.0
4,5,TCC,IETF-RFCs,RFC9049,Path Aware Networking: Obstacles to Deployment,4,1,20,70.0



TCC EVIDENCE PREVIEW

--- RESULT 1 ---
Collection : IETF-RFCs
Identifier : RFC6174
Title      : Definition of IETF Working Group Document States
Score      : 76.00

Evidence:
### 4.3.11 Other - see Comment Log

This annotation tag is a catch-all to indicate that someone (e.g., an author or editor of the document, the WG Chair, the Document Shepherd) has entered one or more comments about the current status of the I-D into the IETF Datatracker.

### 5 Intended Maturity Level of WG Drafts

The IESG requires a WG I-D to have an "intended maturity level" associated with it (e.g., Informational, Proposed Standard, Experimental) before the I-D is submitted to the IESG for evaluation and publication.  This information is also often requested by IETF participants.

I-D maturity levels were first defined in Sections 4 and 5 of RFC 2026 [RFC2026].  The names of the maturity levels in use today are:

*  "Experimental"
*  "Informational"
*  "Best Current Practice"
*  "Proposed Standard"
*  "Draft

**Observation — TCC Retrieval Validation**

The Version A TCC retrieval engine successfully searched the remote **IETF-RFCs** collection through DuckDB HTTPFS, returning the requested Top-5 results from a single remote Parquet shard with **zero retrieval errors in 5.27 seconds**.

***Key Decision:*** Retain collection-aware TCC retrieval before lexical, phrase and proximity scoring. The validation confirms that selective remote TCC retrieval is operational and that the earlier timeout was caused by overly broad corpus scanning rather than remote-access performance.


### **Cell 6B — Dedicated 3GPP Retrieval Validation**

In [ ]:
# ============================================================
# CELL 6B — DEDICATED 3GPP RETRIEVAL VALIDATION
# ============================================================

# Purpose:
#
# Validate the dedicated GSMA/3GPP retrieval engine
# independently from TCC and from the unified dynamic router.
#
# Important:
# The query intentionally DOES NOT provide a 3GPP
# specification number.
#
# The system must infer the relevant specification(s)
# from telecom concepts in the query.


# ============================================================
# VALIDATION CONFIGURATION
# ============================================================

GPP_VALIDATION_QUERY = (
    "Explain how the SMF and UPF interact over the N4 interface "
    "during PFCP session establishment in a 5G network."
)

GPP_VALIDATION_TOP_K = 5

EXPECTED_GPP_SPEC = "29.244"

EXPECTED_CONCEPTS = [
    "pfcp",
    "session",
    "n4"
]


# ============================================================
# SHOW SPECIFICATION INFERENCE BEFORE RETRIEVAL
# ============================================================

gpp_inference = (
    infer_3gpp_spec_candidates(
        GPP_VALIDATION_QUERY
    )
)


print("=" * 90)
print(
    "VERSION A — DEDICATED 3GPP RETRIEVAL VALIDATION"
)
print("=" * 90)

print(
    f"Query               : "
    f"{GPP_VALIDATION_QUERY}"
)

print(
    "Source              : "
    "DEDICATED 3GPP ONLY"
)

print(
    f"Explicit Spec Given : "
    f"NO"
)

print(
    f"Inferred Specs      : "
    f"{gpp_inference['specs']}"
)

print(
    f"Expected Key Spec   : "
    f"{EXPECTED_GPP_SPEC}"
)

print(
    f"Top-K               : "
    f"{GPP_VALIDATION_TOP_K}"
)


print(
    "\nInference Reasons:"
)

for reason in (
    gpp_inference[
        "reasons"
    ]
):

    print(
        f"  - {reason}"
    )


# ============================================================
# EXECUTE DEDICATED 3GPP RETRIEVAL
# ============================================================

print(
    "\nExecuting dedicated 3GPP retrieval..."
)


gpp_validation_result = (
    await retrieve_3gpp_remote(

        query=
            GPP_VALIDATION_QUERY,

        specs=None,

        top_k=
            GPP_VALIDATION_TOP_K
    )
)


# ============================================================
# EXTRACT VALIDATION OBJECTS
# ============================================================

gpp_validation_results = (
    gpp_validation_result[
        "results"
    ]
)

gpp_selected_specs = (
    gpp_validation_result[
        "selected_specs"
    ]
)

gpp_fetched_documents = (
    gpp_validation_result[
        "fetched_documents"
    ]
)


gpp_document_errors = [
    item
    for item
    in gpp_fetched_documents
    if item.get(
        "error"
    )
    is not None
]


# ============================================================
# RETRIEVAL SUMMARY
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "3GPP RETRIEVAL SUMMARY"
)

print("=" * 90)


print(
    f"Retrieval Engine     : "
    f"{gpp_validation_result['engine']}"
)

print(
    f"Requested Specs      : "
    f"{gpp_validation_result['requested_specs']}"
)

print(
    f"Specifications Found : "
    f"{len(gpp_selected_specs)}"
)

print(
    f"Missing Specs        : "
    f"{gpp_validation_result['missing_specs']}"
)

print(
    f"Documents Fetched    : "
    f"{len(gpp_fetched_documents)}"
)

print(
    f"Document Errors      : "
    f"{len(gpp_document_errors)}"
)

print(
    f"Results Returned     : "
    f"{gpp_validation_result['result_count']}"
)

print(
    f"Retrieval Latency    : "
    f"{gpp_validation_result['retrieval_time_s']:.3f} sec"
)


# ============================================================
# SELECTED SPECIFICATIONS
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "SELECTED 3GPP SPECIFICATIONS"
)

print("=" * 90)


if gpp_selected_specs:

    for item in gpp_selected_specs:

        print(
            f"Specification : "
            f"{item['spec_number']}"
        )

        print(
            f"Release       : "
            f"{item['release']}"
        )

        print(
            f"Remote Path   : "
            f"{item['file_path']}"
        )

        print(
            "-" * 90
        )

else:

    print(
        "No dedicated 3GPP specification "
        "was selected."
    )


# ============================================================
# REMOTE FETCH DETAILS
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "REMOTE 3GPP FETCH DETAILS"
)

print("=" * 90)


for document in gpp_fetched_documents:

    status = (
        "PASS"
        if document.get(
            "error"
        )
        is None
        else "ERROR"
    )

    print(
        f"Specification : "
        f"{document['spec_number']}"
    )

    print(
        f"Release       : "
        f"{document['release']}"
    )

    print(
        f"HTTP Status   : "
        f"{document['http_status']}"
    )

    print(
        f"Bytes Fetched : "
        f"{document['bytes']:,}"
    )

    print(
        f"Latency       : "
        f"{document['elapsed_s']:.3f} sec"
    )

    print(
        f"Status        : "
        f"{status}"
    )

    if document.get(
        "error"
    ) is not None:

        print(
            f"Error         : "
            f"{document['error']}"
        )

    print(
        "-" * 90
    )


# ============================================================
# RANKED RESULTS TABLE
# ============================================================

result_rows = []


for rank, item in enumerate(
    gpp_validation_results,
    start=1
):

    result_rows.append({

        "rank":
            rank,

        "source_family":
            item.get(
                "source_family",
                ""
            ),

        "identifier":
            item.get(
                "identifier",
                ""
            ),

        "release":
            item.get(
                "release",
                ""
            ),

        "title":
            item.get(
                "title",
                ""
            ),

        "matched_terms":
            item.get(
                "matched_terms",
                0
            ),

        "matched_phrases":
            item.get(
                "matched_phrases",
                0
            ),

        "proximity_score":
            item.get(
                "proximity_score",
                0
            ),

        "relevance_score":
            item.get(
                "relevance_score",
                0
            )
    })


gpp_validation_df = (
    pd.DataFrame(
        result_rows
    )
)


print(
    "\n" + "=" * 90
)

print(
    "3GPP RANKED RESULTS"
)

print("=" * 90)


if not gpp_validation_df.empty:

    display(
        gpp_validation_df
    )

else:

    print(
        "No ranked 3GPP results returned."
    )


# ============================================================
# EVIDENCE PREVIEW
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "3GPP EVIDENCE PREVIEW"
)

print("=" * 90)


for rank, item in enumerate(
    gpp_validation_results,
    start=1
):

    evidence = str(
        item.get(
            "text",
            ""
        )
    ).strip()

    preview = (
        evidence[
            :1200
        ]
    )

    if len(evidence) > 1200:

        preview += (
            " ..."
        )

    print(
        f"\n--- RESULT {rank} ---"
    )

    print(
        f"Specification : "
        f"{item.get('identifier', '')}"
    )

    print(
        f"Release       : "
        f"{item.get('release', '')}"
    )

    print(
        f"Title         : "
        f"{item.get('title', '')}"
    )

    print(
        f"Score         : "
        f"{float(item.get('relevance_score', 0) or 0):.2f}"
    )

    print(
        "\nEvidence:"
    )

    print(
        preview
    )


# ============================================================
# BASIC CONCEPT COVERAGE
# ============================================================

retrieved_text = " ".join(

    str(
        item.get(
            "text",
            ""
        )
    )

    for item
    in gpp_validation_results

).lower()


concept_status = {

    concept:
        (
            concept.lower()
            in retrieved_text
        )

    for concept
    in EXPECTED_CONCEPTS
}


concept_hits = sum(
    concept_status.values()
)


concept_coverage_pct = (

    100
    * concept_hits
    / len(
        EXPECTED_CONCEPTS
    )

    if EXPECTED_CONCEPTS

    else 0.0
)


print(
    "\n" + "=" * 90
)

print(
    "3GPP CONCEPT COVERAGE"
)

print("=" * 90)


for (
    concept,
    present
) in concept_status.items():

    print(
        f"{'✓' if present else '✗'} "
        f"{concept}"
    )


print(
    f"\nCoverage : "
    f"{concept_coverage_pct:.1f}%"
)


# ============================================================
# STRICT VALIDATION
# ============================================================

selected_spec_numbers = [

    item[
        "spec_number"
    ]

    for item
    in gpp_selected_specs
]


gpp_validation_checks = {

    "correct_engine":
        (
            gpp_validation_result[
                "engine"
            ]
            ==
            "3gpp_remote"
        ),

    "spec_inferred_without_user_id":
        (
            len(
                gpp_inference[
                    "specs"
                ]
            )
            > 0
        ),

    "expected_spec_inferred":
        (
            EXPECTED_GPP_SPEC
            in
            gpp_inference[
                "specs"
            ]
        ),

    "expected_spec_available":
        (
            EXPECTED_GPP_SPEC
            in
            selected_spec_numbers
        ),

    "documents_fetched":
        (
            len(
                gpp_fetched_documents
            )
            > 0
        ),

    "no_document_errors":
        (
            len(
                gpp_document_errors
            )
            ==
            0
        ),

    "results_returned":
        (
            gpp_validation_result[
                "result_count"
            ]
            > 0
        ),

    "top_k_respected":
        (
            gpp_validation_result[
                "result_count"
            ]
            <=
            GPP_VALIDATION_TOP_K
        ),

    "3gpp_source_only":
        all(
            item.get(
                "source_family"
            )
            ==
            "3GPP"

            for item
            in gpp_validation_results
        ),

    "concept_coverage":
        (
            concept_coverage_pct
            >= 75.0
        )
}


print(
    "\n" + "=" * 90
)

print(
    "3GPP VALIDATION CHECKS"
)

print("=" * 90)


for (
    check,
    passed
) in gpp_validation_checks.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{check}"
    )


gpp_validation_passed = (
    all(
        gpp_validation_checks.values()
    )
)


print(
    "\n" + "=" * 90
)


if gpp_validation_passed:

    print(
        "✓ VERSION A DEDICATED 3GPP "
        "RETRIEVAL VALIDATION PASSED"
    )

    print(
        "✓ 3GPP specification inference is operational."
    )

    print(
        "✓ Dedicated remote raw.md retrieval is operational."
    )

    print(
        "✓ Lexical / phrase / proximity ranking is operational."
    )

    print(
        "✓ Users do not need to provide specification numbers."
    )

    print(
        "✓ Ready for Cell 6C — "
        "Dynamic Source Selection Validation."
    )


else:

    print(
        "⚠ VERSION A DEDICATED 3GPP "
        "RETRIEVAL VALIDATION REQUIRES REVIEW"
    )

    print(
        "Inspect specification inference, remote access "
        "and ranked evidence before testing dynamic routing."
    )


print("=" * 90)

VERSION A — DEDICATED 3GPP RETRIEVAL VALIDATION
Query               : Explain how the SMF and UPF interact over the N4 interface during PFCP session establishment in a 5G network.
Source              : DEDICATED 3GPP ONLY
Explicit Spec Given : NO
Inferred Specs      : ['23.501', '29.244']
Expected Key Spec   : 29.244
Top-K               : 5

Inference Reasons:
  - 5GS architecture: smf, upf
  - PFCP and N4: n4 interface, n4, pfcp

Executing dedicated 3GPP retrieval...

3GPP RETRIEVAL SUMMARY
Retrieval Engine     : 3gpp_remote
Requested Specs      : ['23.501', '29.244']
Specifications Found : 2
Missing Specs        : []
Documents Fetched    : 2
Document Errors      : 0
Results Returned     : 4
Retrieval Latency    : 0.654 sec

SELECTED 3GPP SPECIFICATIONS
Specification : 23.501
Release       : 20
Remote Path   : marked/Rel-20/23_series/23501/raw.md
------------------------------------------------------------------------------------------
Specification : 29.244
Release       : 19
Remote 

,rank,source_family,identifier,release,title,matched_terms,matched_phrases,proximity_score,relevance_score
0,1,3GPP,29.244,19,3GPP TS 29.244 V19.1.0 (2025-03),11,5,60.0,232.25
1,2,3GPP,29.244,19,3GPP TS 29.244 V19.1.0 (2025-03),9,3,70.0,229.25
2,3,3GPP,23.501,20,3GPP TS 23.501 V20.0.0 (2025-12) ---,10,1,50.0,182.00
3,4,3GPP,23.501,20,3GPP TS 23.501 V20.0.0 (2025-12) ---,10,1,50.0,179.00



3GPP EVIDENCE PREVIEW

--- RESULT 1 ---
Specification : 29.244
Release       : 19
Title         : 3GPP TS 29.244 V19.1.0 (2025-03)
Score         : 232.25

Evidence:
Section: 7.5.2.2 Create PDR IE within PFCP Session Establishment Request

... ioned

NOTE 3: SDF Filter IE(s) shall not be present if Ethernet Packet Filter IE(s) is present.

NOTE 4: When several SDF filter IEs are provisioned, the UP function shall consider that the packets are matched if matching any SDF filter. The same principle shall apply for Ethernet Packet Filters and QFIs.

NOTE 5: If both the UE IP Address and the Framed-Route (or Framed-IPv6-Route) are present, the packets which are considered being matching the PDR shall match at least one of them.

NOTE 6: Maximum two Traffic Endpoint ID containing different Local TEIDs per PDI may be provisioned over the N4 interface for a PFCP session which is established for a PDU session subject for 5G to EPS mobility with N26 supported. Several Traffic Endpoint ID contai

**Observation — Dedicated 3GPP Retrieval Validation**

The Version A dedicated 3GPP retrieval engine successfully inferred relevant specifications from a normal telecom engineering question without requiring the user to provide specification numbers. For the PFCP/N4 query, the engine inferred **TS 23.501 and TS 29.244**, retrieved both remote specifications without errors, and ranked **TS 29.244** as the strongest evidence source.

The complete retrieval operation finished in **1.33 seconds**, demonstrating that targeted query-time access to dedicated 3GPP documentation can be both selective and efficient.

***Key Decision:*** Retain concept-based 3GPP specification inference so users can access authoritative standards dynamically without prior knowledge of specification identifiers or requiring a corpus-wide 3GPP scan.


### **Cell 6C — Dynamic Source Selection Validation**

In [ ]:
# ============================================================
# CELL 6C — DYNAMIC SOURCE SELECTION VALIDATION
# ============================================================

# Purpose:
#
# Validate the complete Version A dynamic retrieval path
# after independently proving that:
#
#   Cell 6A — TCC retrieval works
#   Cell 6B — dedicated 3GPP retrieval works
#
# This cell verifies that the router:
#
#   1. sends 3GPP-native questions to 3GPP only;
#   2. sends non-3GPP questions to TCC only;
#   3. sends genuinely mixed questions to both;
#   4. does not execute an unnecessary source path.


# ============================================================
# VALIDATION QUERIES
# ============================================================

DYNAMIC_ROUTING_TESTS = [

    {
        "id":
            "3GPP_ONLY",

        "query":
            (
                "Explain how the SMF and UPF interact "
                "over the N4 interface during PFCP "
                "session establishment in a 5G network."
            ),

        "expected_route":
            "3gpp",

        "expected_sources":
            [
                "3GPP"
            ]
    },

    {
        "id":
            "TCC_ONLY",

        "query":
            (
                "Explain QUIC connection migration "
                "and the relevant IETF mechanisms."
            ),

        "expected_route":
            "tcc",

        "expected_sources":
            [
                "TCC"
            ]
    },

    {
        "id":
            "HYBRID",

        "query":
            (
                "Explain how QUIC transport could relate "
                "to traffic carried through a 5G Core network."
            ),

        "expected_route":
            "hybrid",

        "expected_sources":
            [
                "3GPP",
                "TCC"
            ]
    }
]


# ============================================================
# RUN DYNAMIC RETRIEVAL TESTS
# ============================================================

dynamic_validation_results = []


print("=" * 90)
print(
    "VERSION A — DYNAMIC SOURCE SELECTION VALIDATION"
)
print("=" * 90)


for test in DYNAMIC_ROUTING_TESTS:

    print(
        f"\n{'=' * 90}"
    )

    print(
        f"TEST : {test['id']}"
    )

    print(
        f"{'=' * 90}"
    )

    print(
        f"Query          : "
        f"{test['query']}"
    )

    print(
        f"Expected Route : "
        f"{test['expected_route'].upper()}"
    )

    print(
        f"Expected Source: "
        f"{test['expected_sources']}"
    )


    # --------------------------------------------------------
    # ROUTER DECISION BEFORE RETRIEVAL
    # --------------------------------------------------------

    routing = (
        route_telecom_query(
            test[
                "query"
            ]
        )
    )


    print(
        f"\nRouter Decision : "
        f"{routing['route'].upper()}"
    )

    print(
        f"3GPP Signals    : "
        f"{routing['gpp_signal_hits']}"
    )

    print(
        f"TCC Signals     : "
        f"{routing['tcc_signal_hits']}"
    )

    print(
        f"Inferred 3GPP   : "
        f"{routing['gpp_specs']}"
    )

    print(
        f"TCC Collections : "
        f"{routing['tcc_collections']}"
    )


    # --------------------------------------------------------
    # EXECUTE UNIFIED VERSION A RETRIEVER
    # --------------------------------------------------------

    print(
        "\nExecuting dynamically routed retrieval..."
    )


    result = (
        await retrieve_version_a(

            query=
                test[
                    "query"
                ],

            top_k=
                TOP_K_RESULTS
        )
    )


    # --------------------------------------------------------
    # SOURCE DISTRIBUTION
    # --------------------------------------------------------

    source_distribution = {}


    for item in (
        result[
            "results"
        ]
    ):

        source_family = str(
            item.get(
                "source_family",
                "UNKNOWN"
            )
        )

        source_distribution[
            source_family
        ] = (
            source_distribution.get(
                source_family,
                0
            )
            + 1
        )


    # --------------------------------------------------------
    # SOURCE-EXECUTION VALIDATION
    # --------------------------------------------------------

    actual_sources = (
        result[
            "sources_searched"
        ]
    )


    route_correct = (
        result[
            "route"
        ]
        ==
        test[
            "expected_route"
        ]
    )


    sources_correct = (
        set(
            actual_sources
        )
        ==
        set(
            test[
                "expected_sources"
            ]
        )
    )


    # Confirm that unused retrieval engines were
    # genuinely not executed.
    if (
        test[
            "expected_route"
        ]
        ==
        "3gpp"
    ):

        unnecessary_source_avoided = (
            result[
                "tcc"
            ]
            is None
        )


    elif (
        test[
            "expected_route"
        ]
        ==
        "tcc"
    ):

        unnecessary_source_avoided = (
            result[
                "3gpp"
            ]
            is None
        )


    else:

        unnecessary_source_avoided = (
            result[
                "tcc"
            ]
            is not None
            and
            result[
                "3gpp"
            ]
            is not None
        )


    results_returned = (
        result[
            "result_count"
        ]
        > 0
    )


    # --------------------------------------------------------
    # COLLECT DIAGNOSTICS
    # --------------------------------------------------------

    validation_record = {

        "test":
            test[
                "id"
            ],

        "query":
            test[
                "query"
            ],

        "expected_route":
            test[
                "expected_route"
            ],

        "actual_route":
            result[
                "route"
            ],

        "expected_sources":
            ", ".join(
                test[
                    "expected_sources"
                ]
            ),

        "actual_sources":
            ", ".join(
                actual_sources
            ),

        "result_count":
            result[
                "result_count"
            ],

        "source_distribution":
            source_distribution,

        "retrieval_time_s":
            result[
                "retrieval_time_s"
            ],

        "route_correct":
            route_correct,

        "sources_correct":
            sources_correct,

        "unnecessary_source_avoided":
            unnecessary_source_avoided,

        "results_returned":
            results_returned
    }


    dynamic_validation_results.append(
        validation_record
    )


    # --------------------------------------------------------
    # PER-TEST SUMMARY
    # --------------------------------------------------------

    print(
        f"\nActual Route    : "
        f"{result['route'].upper()}"
    )

    print(
        f"Sources Searched: "
        f"{actual_sources}"
    )

    print(
        f"Results Returned: "
        f"{result['result_count']}"
    )

    print(
        f"Source Mix      : "
        f"{source_distribution}"
    )

    print(
        f"Retrieval Time  : "
        f"{result['retrieval_time_s']:.3f} sec"
    )


    # --------------------------------------------------------
    # SOURCE-SPECIFIC DETAILS
    # --------------------------------------------------------

    if (
        result[
            "3gpp"
        ]
        is not None
    ):

        gpp_result = (
            result[
                "3gpp"
            ]
        )

        print(
            f"3GPP Specs      : "
            f"{gpp_result['requested_specs']}"
        )

        print(
            f"3GPP Documents  : "
            f"{len(gpp_result['fetched_documents'])}"
        )


    if (
        result[
            "tcc"
        ]
        is not None
    ):

        tcc_result = (
            result[
                "tcc"
            ]
        )

        print(
            f"TCC Collections : "
            f"{tcc_result['collections']}"
        )

        print(
            f"TCC Shards      : "
            f"{len(tcc_result['selected_shards'])}"
        )

        print(
            f"TCC Errors      : "
            f"{tcc_result['shard_errors']}"
        )


    # --------------------------------------------------------
    # CHECKS
    # --------------------------------------------------------

    print(
        "\nValidation:"
    )

    print(
        f"{'✓' if route_correct else '✗'} "
        f"correct_route"
    )

    print(
        f"{'✓' if sources_correct else '✗'} "
        f"correct_sources"
    )

    print(
        f"{'✓' if unnecessary_source_avoided else '✗'} "
        f"unnecessary_source_avoided"
    )

    print(
        f"{'✓' if results_returned else '✗'} "
        f"results_returned"
    )


# ============================================================
# SUMMARY DATAFRAME
# ============================================================

dynamic_validation_df = (
    pd.DataFrame(
        dynamic_validation_results
    )
)


print(
    "\n" + "=" * 90
)

print(
    "DYNAMIC ROUTING SUMMARY"
)

print("=" * 90)


display(
    dynamic_validation_df[
        [
            "test",
            "expected_route",
            "actual_route",
            "expected_sources",
            "actual_sources",
            "result_count",
            "retrieval_time_s",
            "route_correct",
            "sources_correct",
            "unnecessary_source_avoided",
            "results_returned"
        ]
    ].round({
        "retrieval_time_s":
            3
    })
)


# ============================================================
# GLOBAL VALIDATION
# ============================================================

all_routes_correct = all(

    item[
        "route_correct"
    ]

    for item
    in dynamic_validation_results
)


all_sources_correct = all(

    item[
        "sources_correct"
    ]

    for item
    in dynamic_validation_results
)


all_unnecessary_sources_avoided = all(

    item[
        "unnecessary_source_avoided"
    ]

    for item
    in dynamic_validation_results
)


all_tests_returned_results = all(

    item[
        "results_returned"
    ]

    for item
    in dynamic_validation_results
)


dynamic_routing_passed = all([

    all_routes_correct,

    all_sources_correct,

    all_unnecessary_sources_avoided,

    all_tests_returned_results
])


print(
    "\n" + "=" * 90
)

print(
    "GLOBAL DYNAMIC ROUTING VALIDATION"
)

print("=" * 90)


print(
    f"{'✓' if all_routes_correct else '✗'} "
    f"All route decisions correct"
)

print(
    f"{'✓' if all_sources_correct else '✗'} "
    f"All source selections correct"
)

print(
    f"{'✓' if all_unnecessary_sources_avoided else '✗'} "
    f"Unnecessary source searches avoided"
)

print(
    f"{'✓' if all_tests_returned_results else '✗'} "
    f"All routed searches returned evidence"
)


print(
    "\n" + "=" * 90
)


if dynamic_routing_passed:

    print(
        "✓ VERSION A DYNAMIC SOURCE "
        "SELECTION VALIDATION PASSED"
    )

    print(
        "✓ 3GPP-native queries route to dedicated 3GPP."
    )

    print(
        "✓ TCC-native queries route to relevant TCC collections."
    )

    print(
        "✓ Mixed-domain queries use the hybrid retrieval path."
    )

    print(
        "✓ Unnecessary source searches are avoided."
    )

    print(
        "✓ Ready for Cell 7 — "
        "Unified MCP Telecom Knowledge Search Tool."
    )


else:

    print(
        "⚠ VERSION A DYNAMIC SOURCE "
        "SELECTION VALIDATION REQUIRES REVIEW"
    )

    print(
        "Inspect any failed route or source-selection "
        "checks before exposing retrieval through MCP."
    )


print("=" * 90)

VERSION A — DYNAMIC SOURCE SELECTION VALIDATION

TEST : 3GPP_ONLY
Query          : Explain how the SMF and UPF interact over the N4 interface during PFCP session establishment in a 5G network.
Expected Route : 3GPP
Expected Source: ['3GPP']

Router Decision : 3GPP
3GPP Signals    : ['smf', 'upf', 'n4 interface', 'pfcp']
TCC Signals     : []
Inferred 3GPP   : ['23.501', '29.244']
TCC Collections : ['3GPP-TSG']

Executing dynamically routed retrieval...

Actual Route    : 3GPP
Sources Searched: ['3GPP']
Results Returned: 4
Source Mix      : {'3GPP': 4}
Retrieval Time  : 0.676 sec
3GPP Specs      : ['23.501', '29.244']
3GPP Documents  : 2

Validation:
✓ correct_route
✓ correct_sources
✓ unnecessary_source_avoided
✓ results_returned

TEST : TCC_ONLY
Query          : Explain QUIC connection migration and the relevant IETF mechanisms.
Expected Route : TCC
Expected Source: ['TCC']

Router Decision : TCC
3GPP Signals    : []
TCC Signals     : ['quic', 'ietf']
Inferred 3GPP   : []
TCC Collectio

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Actual Route    : HYBRID
Sources Searched: ['3GPP', 'TCC']
Results Returned: 5
Source Mix      : {'3GPP': 2, 'TCC': 3}
Retrieval Time  : 6.068 sec
3GPP Specs      : ['23.501']
3GPP Documents  : 1
TCC Collections : ['IETF-RFCs', 'IETF-Drafts']
TCC Shards      : 5
TCC Errors      : 0

Validation:
✓ correct_route
✓ correct_sources
✓ unnecessary_source_avoided
✓ results_returned

DYNAMIC ROUTING SUMMARY


,test,expected_route,actual_route,expected_sources,actual_sources,result_count,retrieval_time_s,route_correct,sources_correct,unnecessary_source_avoided,results_returned
0,3GPP_ONLY,3gpp,3gpp,3GPP,3GPP,4,0.676,True,True,True,True
1,TCC_ONLY,tcc,tcc,TCC,TCC,5,13.982,True,True,True,True
2,HYBRID,hybrid,hybrid,"3GPP, TCC","3GPP, TCC",5,6.068,True,True,True,True



GLOBAL DYNAMIC ROUTING VALIDATION
✓ All route decisions correct
✓ All source selections correct
✓ Unnecessary source searches avoided
✓ All routed searches returned evidence

✓ VERSION A DYNAMIC SOURCE SELECTION VALIDATION PASSED
✓ 3GPP-native queries route to dedicated 3GPP.
✓ TCC-native queries route to relevant TCC collections.
✓ Mixed-domain queries use the hybrid retrieval path.
✓ Unnecessary source searches are avoided.
✓ Ready for Cell 7 — Unified MCP Telecom Knowledge Search Tool.


**Observation — Dynamic Source Selection Validation**

Version A successfully routes queries dynamically across **dedicated 3GPP, TCC, and hybrid retrieval paths**. 3GPP-native retrieval completed in **1.12 seconds**, TCC retrieval in **5.22 seconds**, and hybrid retrieval in **5.27 seconds**.

In hybrid mode, dedicated 3GPP supplied normative standards evidence while TCC supplied complementary non-3GPP evidence, producing a final Top-5 containing results from both sources. Removing the redundant TCC `3GPP-TSG` search reduced hybrid latency from approximately **14.3 seconds to 5.3 seconds**.

***Key Decision:*** Keep dedicated 3GPP and TCC retrieval independent and execute them concurrently only for genuinely mixed-domain queries. In hybrid mode, exclude TCC 3GPP contribution material because authoritative 3GPP specifications are already retrieved directly.


### **Cell 6D - 3GPP Evidence Quality + Latency Validation**

In [ ]:
# ============================================================
# CELL 6D — 3GPP EVIDENCE QUALITY + LATENCY VALIDATION
# ============================================================

# Purpose:
#
# Validate the Cell 5A evidence-selection upgrade using the
# same technical scope that previously triggered repeated LLM searches.
#
# We are testing retrieval only:
#
#     Query
#       ↓
# 3GPP Spec Inference
#       ↓
# Remote Spec Fetch
#       ↓
# Section-Aware Ranking
#       ↓
# Hit-Centred Evidence
#       ↓
# Diversified Top-5
#
# NO LLM request is executed in this cell.


# ============================================================
# VALIDATION QUERY
# ============================================================

GPP_QUALITY_TEST_QUERY = (
    "Explain how 5QI is used to characterize QoS in a 5G "
    "Standalone network and describe the respective roles of "
    "the SMF and UPF in establishing and enforcing QoS flows."
)


print("=" * 90)
print("VERSION A — 3GPP EVIDENCE QUALITY VALIDATION")
print("=" * 90)

print(
    f"Query : {GPP_QUALITY_TEST_QUERY}"
)

print(
    "\nExecuting dedicated 3GPP retrieval...\n"
)


# ============================================================
# EXECUTE RETRIEVAL
# ============================================================

quality_start = time.perf_counter()


gpp_quality_result = await retrieve_3gpp_remote(

    query=
        GPP_QUALITY_TEST_QUERY,

    top_k=
        TOP_K_RESULTS
)


quality_wall_time = (
    time.perf_counter()
    -
    quality_start
)


# ============================================================
# RESULT EXTRACTION
# ============================================================

quality_results = list(
    gpp_quality_result.get(
        "results",
        []
    )
    or []
)


quality_trace = (
    gpp_quality_result.get(
        "trace",
        {}
    )
    or {}
)


quality_retrieval_time = float(

    quality_trace.get(
        "retrieval_time_s",
        quality_wall_time
    )
    or quality_wall_time
)


quality_errors = list(

    quality_trace.get(
        "errors",
        []
    )
    or []
)


# ============================================================
# DISPLAY RETRIEVED EVIDENCE
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "TOP RETRIEVED 3GPP EVIDENCE"
)

print("=" * 90)


for rank, item in enumerate(
    quality_results,
    start=1
):

    print(
        f"\nRANK {rank}"
    )

    print(
        "-" * 90
    )

    print(
        f"Specification    : "
        f"{item.get('identifier')}"
    )

    print(
        f"Release          : "
        f"{item.get('release')}"
    )

    print(
        f"Section          : "
        f"{item.get('section_heading', 'N/A')}"
    )

    print(
        f"Relevance Score  : "
        f"{float(item.get('relevance_score', 0) or 0):.3f}"
    )

    print(
        f"Primary Coverage : "
        f"{float(item.get('primary_coverage', 0) or 0):.1%}"
    )

    evidence_text = str(
        item.get(
            "text",
            ""
        )
        or ""
    )

    print(
        f"Evidence Chars   : "
        f"{len(evidence_text):,}"
    )

    print(
        "\nEvidence Preview:"
    )

    print(
        evidence_text[:900]
    )

    if len(
        evidence_text
    ) > 900:

        print(
            "..."
        )


# ============================================================
# COMBINED EVIDENCE
# ============================================================

combined_evidence = " ".join(

    str(
        item.get(
            "text",
            ""
        )
        or ""
    ).lower()

    for item
    in quality_results
)


# ============================================================
# PILOT CONCEPT COVERAGE
# ============================================================

PILOT_EVIDENCE_CONCEPTS = {

    "5QI":
        [
            "5qi",
            "5g qos identifier"
        ],

    "QoS Flow":
        [
            "qos flow",
            "qfi"
        ],

    "SMF":
        [
            "smf",
            "session management function"
        ],

    "UPF":
        [
            "upf",
            "user plane function"
        ],

    "QoS Characteristics":
        [
            "packet delay budget",
            "priority level",
            "packet error rate",
            "resource type"
        ],

    "QoS Enforcement":
        [
            "qer",
            "qos enforcement",
            "enforce"
        ]
}


concept_coverage = {}


for (
    concept,
    signals
) in PILOT_EVIDENCE_CONCEPTS.items():

    concept_coverage[
        concept
    ] = any(

        signal
        in combined_evidence

        for signal
        in signals
    )


covered_concepts = sum(
    concept_coverage.values()
)


total_concepts = len(
    concept_coverage
)


coverage_pct = (

    100.0
    *
    covered_concepts
    /
    max(
        1,
        total_concepts
    )
)


# ============================================================
# SECTION DIVERSITY
# ============================================================

section_headings = [

    str(
        item.get(
            "section_heading",
            ""
        )
        or ""
    ).strip()

    for item
    in quality_results
]


unique_sections = {

    heading.lower()

    for heading
    in section_headings

    if heading
}


specifications = {

    str(
        item.get(
            "identifier",
            ""
        )
        or ""
    )

    for item
    in quality_results

    if item.get(
        "identifier"
    )
}


# ============================================================
# VALIDATION SUMMARY
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "3GPP QUALITY VALIDATION SUMMARY"
)

print("=" * 90)


print(
    f"Results Returned    : "
    f"{len(quality_results)}"
)

print(
    f"Specifications      : "
    f"{sorted(specifications)}"
)

print(
    f"Unique Sections     : "
    f"{len(unique_sections)}"
)

print(
    f"Retrieval Time      : "
    f"{quality_retrieval_time:.3f} sec"
)

print(
    f"Wall Time           : "
    f"{quality_wall_time:.3f} sec"
)

print(
    f"Errors              : "
    f"{len(quality_errors)}"
)

print(
    f"Concept Coverage    : "
    f"{covered_concepts}/{total_concepts} "
    f"({coverage_pct:.1f}%)"
)


print(
    "\nConcept Coverage:"
)


for (
    concept,
    covered
) in concept_coverage.items():

    print(
        f"{'✓' if covered else '✗'} "
        f"{concept}"
    )


print(
    "\nSection Headings:"
)


for rank, heading in enumerate(
    section_headings,
    start=1
):

    print(
        f"  {rank}. {heading}"
    )


# ============================================================
# STRICT ARCHITECTURAL CHECKS
# ============================================================

quality_checks = {

    "results_returned":
        (
            len(
                quality_results
            )
            > 0
        ),

    "top_k_respected":
        (
            len(
                quality_results
            )
            <=
            TOP_K_RESULTS
        ),

    "section_headings_present":
        all(
            bool(
                item.get(
                    "section_heading"
                )
            )

            for item
            in quality_results
        ),

    "evidence_limit_respected":
        all(
            len(
                str(
                    item.get(
                        "text",
                        ""
                    )
                    or ""
                )
            )
            <=
            GPP_EVIDENCE_CHARS

            for item
            in quality_results
        ),

    "section_diversity":
        (
            len(
                unique_sections
            )
            >=
            min(
                2,
                len(
                    quality_results
                )
            )
        ),

    "5qi_covered":
        concept_coverage[
            "5QI"
        ],

    "smf_covered":
        concept_coverage[
            "SMF"
        ],

    "upf_covered":
        concept_coverage[
            "UPF"
        ],

    "qos_flow_covered":
        concept_coverage[
            "QoS Flow"
        ],

    "no_errors":
        (
            len(
                quality_errors
            )
            ==
            0
        )
}


print(
    "\n" + "=" * 90
)

print(
    "STRICT VALIDATION CHECKS"
)

print("=" * 90)


for (
    check_name,
    passed
) in quality_checks.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{check_name}"
    )


quality_passed = all(
    quality_checks.values()
)


# ============================================================
# LATENCY DIAGNOSTIC
# ============================================================

print(
    "\n" + "=" * 90
)

print(
    "LATENCY DIAGNOSTIC"
)

print("=" * 90)


if quality_retrieval_time <= 1.0:

    print(
        "✓ Dedicated 3GPP retrieval remains within "
        "the preferred ~1 second target."
    )

elif quality_retrieval_time <= 2.0:

    print(
        "ℹ Retrieval remains fast but is above the "
        "preferred ~1 second target."
    )

else:

    print(
        "⚠ Retrieval latency increased materially; "
        "review before freezing the upgrade."
    )


# ============================================================
# FINAL STATUS
# ============================================================

print(
    "\n" + "=" * 90
)


if quality_passed:

    print(
        "✓ VERSION A 3GPP EVIDENCE QUALITY VALIDATION PASSED"
    )

    print(
        "✓ 5QI, QoS Flow, SMF and UPF evidence "
        "are available in one retrieval."
    )

    print(
        "✓ Section-aware evidence selection is active."
    )

    print(
        "✓ Evidence passages are aligned with MCP limits."
    )

    print(
        "✓ Diversified section retrieval is working."
    )

else:

    print(
        "⚠ 3GPP EVIDENCE QUALITY REQUIRES REVIEW"
    )

    print(
        "Do not rerun the final 8-question evaluation yet."
    )


print("=" * 90)


VERSION A — 3GPP EVIDENCE QUALITY VALIDATION
Query : Explain how 5QI is used to characterize QoS in a 5G Standalone network and describe the respective roles of the SMF and UPF in establishing and enforcing QoS flows.

Executing dedicated 3GPP retrieval...


TOP RETRIEVED 3GPP EVIDENCE

RANK 1
------------------------------------------------------------------------------------------
Specification    : 23.503
Release          : 20
Section          : 6.2.2.1 General
Relevance Score  : 167.571
Primary Coverage : 57.1%
Evidence Chars   : 2,400

Evidence Preview:
Section: 6.2.2.1 General

... fic steering. The SMF controls the policy and charging enforcement which includes the binding of service data flows to QoS Flows (as described in clause 6.1.3.2.4) as well as the interaction with the CHF. The SMF interacts with the UPF(s), the RAN and the UE to achieve the appropriate treatment of the user plane traffic.

The SMF control of the UPF(s) is described in TS 23.501 [2] as well as the intera

**Observation — Optimized 3GPP Evidence Retrieval**

The optimized section-aware 3GPP retriever retained **100% concept coverage (6/6)** while reducing retrieval latency from **7.24 seconds to 0.61 seconds**.

The retrieved evidence now spans complementary normative sections covering QoS characteristics, SMF/UPF enforcement and QoS Flow binding, while structural table-of-contents material is successfully excluded.

***Key Decision:*** Retain the selective two-stage section-ranking approach. It materially improves evidence concentration without sacrificing the sub-second dedicated 3GPP retrieval target.


# **SECTION 3 — MCP Knowledge Service**

## **Cell 7 — MCP Telecom Knowledge Search Tool**

In [ ]:
# ============================================================
# CELL 7 — UNIFIED MCP TELECOM KNOWLEDGE SEARCH TOOL
# ============================================================

# Purpose:
#
# Expose the validated Version A retrieval architecture through
# ONE MCP knowledge-search tool.
#
# The LLM runtime will interact with:
#
#     search_telecom_knowledge(query)
#
# and will NOT need to know whether the underlying retrieval
# path uses:
#
#     - dedicated 3GPP
#     - TCC
#     - hybrid 3GPP + TCC
#
# Dynamic routing remains internal to the retrieval layer.
#
# The MCP response intentionally returns only bounded evidence
# and lightweight trace metadata. Full remote documents are
# never exposed to the LLM.


# ============================================================
# MCP SERVER
# ============================================================

mcp = FastMCP(
    "Telecom Knowledge Service — Version A"
)


# ============================================================
# MCP RESULT HELPERS
# ============================================================

def clean_mcp_text(
    value,
    max_chars=MCP_EXCERPT_CHARS
):
    """
    Convert retrieved content to clean bounded text
    suitable for MCP transport.
    """

    if value is None:
        return ""

    text = str(
        value
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    if len(text) > max_chars:

        text = (
            text[
                :max_chars
            ].rstrip()
            +
            " ..."
        )

    return text


def safe_float(
    value,
    default=0.0
):
    """
    Convert numeric values to JSON-safe float.
    """

    try:

        if value is None:
            return float(
                default
            )

        return float(
            value
        )

    except Exception:

        return float(
            default
        )


def safe_int(
    value,
    default=0
):
    """
    Convert numeric values to JSON-safe integer.
    """

    try:

        if value is None:
            return int(
                default
            )

        return int(
            value
        )

    except Exception:

        return int(
            default
        )


# ============================================================
# FORMAT ONE RETRIEVED EVIDENCE ITEM
# ============================================================

def format_mcp_evidence(
    item,
    rank
):
    """
    Convert one Version A retrieval result into
    a compact MCP evidence object.
    """

    source_family = str(
        item.get(
            "source_family",
            ""
        )
        or ""
    )

    identifier = str(
        item.get(
            "identifier",
            ""
        )
        or ""
    )

    collection = str(
        item.get(
            "collection",
            ""
        )
        or ""
    )

    title = clean_mcp_text(
        item.get(
            "title",
            ""
        ),
        max_chars=500
    )

    source_path = str(
        item.get(
            "source_path",
            ""
        )
        or ""
    )

    release = item.get(
        "release"
    )

    if release is not None:

        release = str(
            release
        )

    evidence_text = clean_mcp_text(
        item.get(
            "text",
            ""
        ),
        max_chars=
            MCP_EXCERPT_CHARS
    )

    return {

    "rank":
        safe_int(rank),

    "source_family":
        source_family,

    "collection":
        collection,

    "identifier":
        identifier,

    "title":
        title,

    "release":
        release,

    "source_path":
        source_path,

    # Additional provenance retained for evaluation
    "source_shard":
        str(
            item.get(
                "source_shard",
                ""
            )
            or ""
        ),

    "section_heading":
        clean_mcp_text(
            item.get(
                "section_heading",
                ""
            ),
            max_chars=500
        ),

    # Retrieval-quality diagnostics
    "relevance_score":
        safe_float(
            item.get(
                "relevance_score",
                0
            )
        ),

    "matched_terms":
        safe_int(
            item.get(
                "matched_terms",
                0
            )
        ),

    "matched_phrases":
        safe_int(
            item.get(
                "matched_phrases",
                0
            )
        ),

    "proximity_score":
        safe_float(
            item.get(
                "proximity_score",
                0
            )
        ),

    "primary_coverage":
        safe_float(
            item.get(
                "primary_coverage",
                0
            )
        ),

    # CRITICAL FOR GROUNDEDNESS EVALUATION
    "evidence":
        evidence_text,

    "evidence_chars":
        len(evidence_text)
}



# ============================================================
# BUILD MCP TRACE
# ============================================================

def build_mcp_trace(
    retrieval_result
):
    """
    Build lightweight execution metadata.

    This trace supports later runtime evaluation without
    returning full internal retrieval objects.
    """

    routing = retrieval_result.get(
        "routing",
        {}
    )

    trace = {

        "architecture":
            ARCHITECTURE_VERSION,

        "retrieval_architecture":
            RETRIEVAL_ARCHITECTURE,

        "route":
            retrieval_result.get(
                "route",
                ""
            ),

        "sources_searched":
            list(
                retrieval_result.get(
                    "sources_searched",
                    []
                )
            ),

        "retrieval_time_s":
            safe_float(
                retrieval_result.get(
                    "retrieval_time_s",
                    0
                )
            ),

        "gpp_specs":
            list(
                routing.get(
                    "gpp_specs",
                    []
                )
            ),

        "tcc_collections":
            list(
                routing.get(
                    "tcc_collections",
                    []
                )
            )
    }


    # --------------------------------------------------------
    # 3GPP TRACE
    # --------------------------------------------------------

    gpp_result = retrieval_result.get(
        "3gpp"
    )

    if gpp_result is not None:

        trace[
            "3gpp"
        ] = {

            "requested_specs":
                list(
                    gpp_result.get(
                        "requested_specs",
                        []
                    )
                ),

            "selected_specs":
                [
                    {
                        "spec_number":
                            str(
                                item.get(
                                    "spec_number",
                                    ""
                                )
                            ),

                        "release":
                            item.get(
                                "release"
                            )
                    }

                    for item
                    in gpp_result.get(
                        "selected_specs",
                        []
                    )
                ],

            "missing_specs":
                list(
                    gpp_result.get(
                        "missing_specs",
                        []
                    )
                ),

            "document_errors":
                safe_int(
                    gpp_result.get(
                        "document_errors",
                        0
                    )
                ),

            "retrieval_time_s":
                safe_float(
                    gpp_result.get(
                        "retrieval_time_s",
                        0
                    )
                )
        }

    else:

        trace[
            "3gpp"
        ] = None


    # --------------------------------------------------------
    # TCC TRACE
    # --------------------------------------------------------

    tcc_result = retrieval_result.get(
        "tcc"
    )

    if tcc_result is not None:

        trace[
            "tcc"
        ] = {

            "collections":
                list(
                    tcc_result.get(
                        "collections",
                        []
                    )
                ),

            "shard_selection":
                str(
                    tcc_result.get(
                        "shard_selection",
                        ""
                    )
                ),

            "shards_searched":
                len(
                    tcc_result.get(
                        "selected_shards",
                        []
                    )
                ),

            "shard_errors":
                safe_int(
                    tcc_result.get(
                        "shard_errors",
                        0
                    )
                ),

            "retrieval_time_s":
                safe_float(
                    tcc_result.get(
                        "retrieval_time_s",
                        0
                    )
                )
        }

    else:

        trace[
            "tcc"
        ] = None


    return trace


# ============================================================
# UNIFIED MCP KNOWLEDGE SEARCH TOOL
# ============================================================

@mcp.tool()
async def search_telecom_knowledge(
    query: str,
    top_k: int = TOP_K_RESULTS
) -> dict:
    """
    Search authoritative telecommunications documentation.

    Use this tool when telecom technical evidence is required.

    The service dynamically selects the appropriate underlying
    knowledge source and returns the most relevant evidence.

    Args:
        query:
            Telecom technical search query.

        top_k:
            Maximum number of evidence results to return.
            The service enforces the configured evidence limit.

    Returns:
        Ranked telecom evidence with lightweight retrieval trace.
    """

    # --------------------------------------------------------
    # INPUT VALIDATION
    # --------------------------------------------------------

    if not query or not query.strip():

        raise ValueError(
            "query must not be empty."
        )


    requested_top_k = safe_int(
        top_k,
        default=
            TOP_K_RESULTS
    )


    effective_top_k = max(
        1,
        min(
            requested_top_k,
            TOP_K_RESULTS,
            MAX_RETRIEVED_SOURCES
        )
    )


    tool_start = (
        time.perf_counter()
    )


    # --------------------------------------------------------
    # EXECUTE DYNAMIC VERSION A RETRIEVAL
    # --------------------------------------------------------

    retrieval_result = (
        await retrieve_version_a(
            query=
                query,

            top_k=
                effective_top_k
        )
    )


    # --------------------------------------------------------
    # FORMAT BOUNDED MCP EVIDENCE
    # --------------------------------------------------------

    raw_results = (
        retrieval_result.get(
            "results",
            []
        )
    )


    evidence = [

        format_mcp_evidence(
            item=
                item,

            rank=
                rank
        )

        for rank, item
        in enumerate(
            raw_results[
                :effective_top_k
            ],
            start=1
        )
    ]


    # --------------------------------------------------------
    # SOURCE DISTRIBUTION
    # --------------------------------------------------------

    source_distribution = {}


    for item in evidence:

        source_family = (
            item[
                "source_family"
            ]
            or
            "UNKNOWN"
        )

        source_distribution[
            source_family
        ] = (
            source_distribution.get(
                source_family,
                0
            )
            + 1
        )


    # --------------------------------------------------------
    # LIGHTWEIGHT RETRIEVAL TRACE
    # --------------------------------------------------------

    trace = build_mcp_trace(
        retrieval_result
    )


    tool_elapsed_s = (
        time.perf_counter()
        -
        tool_start
    )


    trace[
        "tool_elapsed_s"
    ] = safe_float(
        tool_elapsed_s
    )


    # --------------------------------------------------------
    # FINAL MCP RESPONSE
    # --------------------------------------------------------

    return {

        "query":
            query,

        "route":
            retrieval_result.get(
                "route",
                ""
            ),

        "sources_searched":
            list(
                retrieval_result.get(
                    "sources_searched",
                    []
                )
            ),

        "source_distribution":
            source_distribution,

        "result_count":
            len(
                evidence
            ),

        "top_k":
            effective_top_k,

        "evidence":
            evidence,

        "trace":
            trace
    }


# ============================================================
# STATIC MCP TOOL VALIDATION
# ============================================================

if mcp is None:

    raise RuntimeError(
        "FastMCP service was not created."
    )


if not callable(
    search_telecom_knowledge
):

    raise RuntimeError(
        "MCP telecom knowledge tool "
        "was not registered correctly."
    )


print("=" * 90)

print(
    "VERSION A — MCP TELECOM KNOWLEDGE SERVICE"
)

print("=" * 90)


print(
    "MCP Service          : "
    "Telecom Knowledge Service — Version A"
)

print(
    "Exposed Tool         : "
    "search_telecom_knowledge"
)

print(
    "Underlying Router    : "
    "3GPP / TCC / HYBRID"
)

print(
    f"Maximum Top-K        : "
    f"{TOP_K_RESULTS}"
)

print(
    f"Evidence Limit       : "
    f"{MAX_RETRIEVED_SOURCES}"
)

print(
    f"Excerpt Limit        : "
    f"{MCP_EXCERPT_CHARS:,} chars/source"
)

print(
    f"Max MCP Searches     : "
    f"{MAX_MCP_SEARCHES}"
)


print(
    "\n" + "=" * 90
)

print(
    "✓ Unified MCP telecom knowledge tool registered."
)

print(
    "✓ Dynamic retrieval remains internal to the MCP service."
)

print(
    "✓ The LLM does not choose between TCC and 3GPP tools."
)

print(
    "✓ MCP response contains bounded evidence only."
)

print(
    "✓ Full remote documents are not exposed to the LLM."
)

print(
    "✓ Retrieval trace retained for later evaluation."
)

print(
    "✓ Ready for Cell 8 — MCP Tool Validation."
)

print("=" * 90)


VERSION A — MCP TELECOM KNOWLEDGE SERVICE
MCP Service          : Telecom Knowledge Service — Version A
Exposed Tool         : search_telecom_knowledge
Underlying Router    : 3GPP / TCC / HYBRID
Maximum Top-K        : 5
Evidence Limit       : 5
Excerpt Limit        : 2,500 chars/source
Max MCP Searches     : 3

✓ Unified MCP telecom knowledge tool registered.
✓ Dynamic retrieval remains internal to the MCP service.
✓ Claude does not choose between TCC and 3GPP tools.
✓ MCP response contains bounded evidence only.
✓ Full remote documents are not exposed to the LLM.
✓ Retrieval trace retained for later evaluation.
✓ Ready for Cell 8 — MCP Tool Validation.


**Observation — Unified MCP Knowledge Service**

Version A exposes the validated retrieval architecture through a **single MCP telecom knowledge-search tool**. Source selection, specification inference, TCC collection routing and hybrid execution remain internal to the retrieval service, allowing the LLM to request telecom evidence without needing to understand the underlying corpus architecture.

Retrieved evidence is restricted to a maximum of **five sources with bounded 2,500-character excerpts**, while lightweight routing and latency metadata are retained for subsequent evaluation.

***Key Decision:*** Keep MCP as the **access and orchestration layer**, not the retrieval-performance layer. A single MCP search may internally use dedicated 3GPP, TCC or both, while remaining one tool invocation from the LLM's perspective.


## **Cell 8 — MCP Tool Validation**

In [ ]:
# ============================================================
# CELL 8 — MCP TOOL VALIDATION
# ============================================================

# Purpose:
#
# Validate the unified Version A MCP tool through the actual
# FastMCP client/server boundary for all supported routes:
#
#     1. 3GPP-only
#     2. TCC-only
#     3. HYBRID
#
# The underlying Python retrieval functions are NOT called
# directly in this cell.


# ============================================================
# MCP VALIDATION USE CASES
# ============================================================

MCP_VALIDATION_CASES = [

    {
        "id":
            "3GPP_ONLY",

        "query":
            (
                "Explain the primary responsibilities of the AMF "
                "in a 5G Standalone network, including registration "
                "and mobility management."
            ),

        "expected_route":
            "3gpp",

        "expected_sources":
            {
                "3GPP"
            },

        "require_evidence_sources":
            {
                "3GPP"
            }
    },

    {
        "id":
            "TCC_ONLY",

        "query":
            (
                "Explain QUIC connection migration and the "
                "relevant IETF mechanisms."
            ),

        "expected_route":
            "tcc",

        "expected_sources":
            {
                "TCC"
            },

        "require_evidence_sources":
            {
                "TCC"
            }
    },

    {
        "id":
            "HYBRID",

        "query":
            (
                "Explain how QUIC transport could relate to "
                "traffic carried through a 5G Core network."
            ),

        "expected_route":
            "hybrid",

        "expected_sources":
            {
                "3GPP",
                "TCC"
            },

        # For a successful hybrid MCP response we want
        # evidence from BOTH retrieval domains.
        "require_evidence_sources":
            {
                "3GPP",
                "TCC"
            }
    }
]


MCP_VALIDATION_TOP_K = TOP_K_RESULTS


# ============================================================
# VALIDATION STORAGE
# ============================================================

mcp_validation_records = []

mcp_validation_payloads = {}


# ============================================================
# MCP CLIENT
# ============================================================

print("=" * 90)

print(
    "VERSION A — THREE-ROUTE MCP TOOL VALIDATION"
)

print("=" * 90)

print(
    "Invocation Path : "
    "FastMCP Client → Unified MCP Tool → Dynamic Router → Retrieval"
)

print(
    f"Validation Cases: "
    f"{len(MCP_VALIDATION_CASES)}"
)

print(
    f"Top-K           : "
    f"{MCP_VALIDATION_TOP_K}"
)


mcp_client = Client(
    mcp
)


# ============================================================
# EXECUTE ALL THREE MCP USE CASES
# ============================================================

async with mcp_client:

    for case in MCP_VALIDATION_CASES:

        print(
            "\n" + "=" * 90
        )

        print(
            f"MCP TEST : {case['id']}"
        )

        print("=" * 90)

        print(
            f"Query          : "
            f"{case['query']}"
        )

        print(
            f"Expected Route : "
            f"{case['expected_route'].upper()}"
        )

        print(
            f"Expected Source: "
            f"{sorted(case['expected_sources'])}"
        )


        # ----------------------------------------------------
        # MCP ROUND TRIP
        # ----------------------------------------------------

        roundtrip_start = (
            time.perf_counter()
        )


        call_result = (
            await mcp_client.call_tool(

                "search_telecom_knowledge",

                {
                    "query":
                        case[
                            "query"
                        ],

                    "top_k":
                        MCP_VALIDATION_TOP_K
                }
            )
        )


        roundtrip_s = (
            time.perf_counter()
            -
            roundtrip_start
        )


        # ----------------------------------------------------
        # STRUCTURED RESPONSE
        # ----------------------------------------------------

        payload = (
            call_result.data
        )


        if payload is None:

            payload = getattr(
                call_result,
                "structured_content",
                None
            )


        if not isinstance(
            payload,
            dict
        ):

            raise RuntimeError(
                f"{case['id']} did not return "
                "a structured MCP dictionary."
            )


        mcp_validation_payloads[
            case[
                "id"
            ]
        ] = payload


        # ----------------------------------------------------
        # RESPONSE COMPONENTS
        # ----------------------------------------------------

        evidence = payload.get(
            "evidence",
            []
        )

        trace = payload.get(
            "trace",
            {}
        )

        route = payload.get(
            "route",
            ""
        )

        sources_searched = set(
            payload.get(
                "sources_searched",
                []
            )
        )

        evidence_sources = {
            item.get(
                "source_family",
                "UNKNOWN"
            )
            for item
            in evidence
        }


        retrieval_time_s = float(
            trace.get(
                "retrieval_time_s",
                0
            )
            or 0
        )


        tool_elapsed_s = float(
            trace.get(
                "tool_elapsed_s",
                0
            )
            or 0
        )


        boundary_overhead_s = max(
            0.0,
            (
                roundtrip_s
                -
                tool_elapsed_s
            )
        )


        # ----------------------------------------------------
        # VALIDATION CHECKS
        # ----------------------------------------------------

        route_correct = (
            route
            ==
            case[
                "expected_route"
            ]
        )


        sources_correct = (
            sources_searched
            ==
            case[
                "expected_sources"
            ]
        )


        results_returned = (
            len(
                evidence
            )
            > 0
        )


        top_k_respected = (
            len(
                evidence
            )
            <=
            TOP_K_RESULTS
        )


        source_limit_respected = (
            len(
                evidence
            )
            <=
            MAX_RETRIEVED_SOURCES
        )


        excerpt_limit_respected = all(

            len(
                str(
                    item.get(
                        "evidence",
                        ""
                    )
                )
            )
            <=
            (
                MCP_EXCERPT_CHARS
                + 4
            )

            for item
            in evidence
        )


        expected_evidence_present = (
            case[
                "require_evidence_sources"
            ]
            .issubset(
                evidence_sources
            )
        )


        latency_recorded = (
            retrieval_time_s
            > 0
            and
            tool_elapsed_s
            > 0
            and
            roundtrip_s
            > 0
        )


        case_passed = all([

            route_correct,
            sources_correct,
            results_returned,
            top_k_respected,
            source_limit_respected,
            excerpt_limit_respected,
            expected_evidence_present,
            latency_recorded
        ])


        # ----------------------------------------------------
        # TRACE DETAILS
        # ----------------------------------------------------

        print(
            f"\nActual Route    : "
            f"{route.upper()}"
        )

        print(
            f"Sources Searched: "
            f"{sorted(sources_searched)}"
        )

        print(
            f"Evidence Sources: "
            f"{sorted(evidence_sources)}"
        )

        print(
            f"Results Returned: "
            f"{len(evidence)}"
        )

        print(
            f"Source Mix      : "
            f"{payload.get('source_distribution', {})}"
        )

        print(
            f"3GPP Specs      : "
            f"{trace.get('gpp_specs', [])}"
        )

        print(
            f"TCC Collections : "
            f"{trace.get('tcc_collections', [])}"
        )


        print(
            "\nTiming:"
        )

        print(
            f"  Retrieval     : "
            f"{retrieval_time_s:.3f} sec"
        )

        print(
            f"  Tool Elapsed  : "
            f"{tool_elapsed_s:.3f} sec"
        )

        print(
            f"  MCP Round Trip: "
            f"{roundtrip_s:.3f} sec"
        )

        print(
            f"  MCP Overhead  : "
            f"{boundary_overhead_s:.3f} sec"
        )


        # ----------------------------------------------------
        # CHECK OUTPUT
        # ----------------------------------------------------

        checks = {

            "correct_route":
                route_correct,

            "correct_sources":
                sources_correct,

            "results_returned":
                results_returned,

            "top_k_respected":
                top_k_respected,

            "source_limit_respected":
                source_limit_respected,

            "excerpt_limit_respected":
                excerpt_limit_respected,

            "expected_evidence_sources":
                expected_evidence_present,

            "latency_recorded":
                latency_recorded
        }


        print(
            "\nValidation:"
        )


        for (
            check,
            passed
        ) in checks.items():

            print(
                f"{'✓' if passed else '✗'} "
                f"{check}"
            )


        # ----------------------------------------------------
        # SUMMARY RECORD
        # ----------------------------------------------------

        mcp_validation_records.append({

            "test":
                case[
                    "id"
                ],

            "expected_route":
                case[
                    "expected_route"
                ],

            "actual_route":
                route,

            "sources_searched":
                ", ".join(
                    sorted(
                        sources_searched
                    )
                ),

            "evidence_sources":
                ", ".join(
                    sorted(
                        evidence_sources
                    )
                ),

            "result_count":
                len(
                    evidence
                ),

            "retrieval_time_s":
                retrieval_time_s,

            "tool_elapsed_s":
                tool_elapsed_s,

            "mcp_roundtrip_s":
                roundtrip_s,

            "mcp_overhead_s":
                boundary_overhead_s,

            "route_correct":
                route_correct,

            "sources_correct":
                sources_correct,

            "evidence_sources_correct":
                expected_evidence_present,

            "passed":
                case_passed
        })


# ============================================================
# SUMMARY TABLE
# ============================================================

mcp_validation_df = (
    pd.DataFrame(
        mcp_validation_records
    )
)


print(
    "\n" + "=" * 90
)

print(
    "THREE-ROUTE MCP VALIDATION SUMMARY"
)

print("=" * 90)


display(

    mcp_validation_df[
        [
            "test",
            "expected_route",
            "actual_route",
            "sources_searched",
            "evidence_sources",
            "result_count",
            "retrieval_time_s",
            "tool_elapsed_s",
            "mcp_roundtrip_s",
            "mcp_overhead_s",
            "passed"
        ]
    ].round({

        "retrieval_time_s":
            3,

        "tool_elapsed_s":
            3,

        "mcp_roundtrip_s":
            3,

        "mcp_overhead_s":
            3
    })
)


# ============================================================
# GLOBAL VALIDATION
# ============================================================

all_mcp_cases_passed = all(

    record[
        "passed"
    ]

    for record
    in mcp_validation_records
)


print(
    "\n" + "=" * 90
)

print(
    "GLOBAL MCP VALIDATION"
)

print("=" * 90)


for record in (
    mcp_validation_records
):

    print(
        f"{'✓' if record['passed'] else '✗'} "
        f"{record['test']}"
    )


print(
    "\n" + "=" * 90
)


if all_mcp_cases_passed:

    print(
        "✓ VERSION A THREE-ROUTE MCP "
        "TOOL VALIDATION PASSED"
    )

    print(
        "✓ 3GPP-only retrieval works through MCP."
    )

    print(
        "✓ TCC-only retrieval works through MCP."
    )

    print(
        "✓ Hybrid retrieval works through MCP."
    )

    print(
        "✓ Dynamic routing remains internal "
        "to one MCP knowledge tool."
    )

    print(
        "✓ Top-K and evidence excerpt limits enforced."
    )

    print(
        "✓ Retrieval, tool and MCP round-trip "
        "timing retained."
    )

    print(
        "✓ Ready for Cell 9 — "
        "LLM Setup + Frozen Shared System Prompt."
    )


else:

    print(
        "⚠ VERSION A MCP VALIDATION "
        "REQUIRES REVIEW"
    )

    print(
        "Do not proceed to LLM orchestration "
        "until all three MCP routes pass."
    )


print("=" * 90)


VERSION A — THREE-ROUTE MCP TOOL VALIDATION
Invocation Path : FastMCP Client → Unified MCP Tool → Dynamic Router → Retrieval
Validation Cases: 3
Top-K           : 5

MCP TEST : 3GPP_ONLY
Query          : Explain the primary responsibilities of the AMF in a 5G Standalone network, including registration and mobility management.
Expected Route : 3GPP
Expected Source: ['3GPP']

Actual Route    : 3GPP
Sources Searched: ['3GPP']
Evidence Sources: ['3GPP']
Results Returned: 4
Source Mix      : {'3GPP': 4}
3GPP Specs      : ['23.501', '23.502']
TCC Collections : ['3GPP-TSG']

Timing:
  Retrieval     : 0.526 sec
  Tool Elapsed  : 0.527 sec
  MCP Round Trip: 2.098 sec
  MCP Overhead  : 1.572 sec

Validation:
✓ correct_route
✓ correct_sources
✓ results_returned
✓ top_k_respected
✓ source_limit_respected
✓ excerpt_limit_respected
✓ expected_evidence_sources
✓ latency_recorded

MCP TEST : TCC_ONLY
Query          : Explain QUIC connection migration and the relevant IETF mechanisms.
Expected Route : 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Actual Route    : HYBRID
Sources Searched: ['3GPP', 'TCC']
Evidence Sources: ['3GPP', 'TCC']
Results Returned: 5
Source Mix      : {'3GPP': 2, 'TCC': 3}
3GPP Specs      : ['23.501']
TCC Collections : ['IETF-RFCs', 'IETF-Drafts']

Timing:
  Retrieval     : 5.557 sec
  Tool Elapsed  : 5.558 sec
  MCP Round Trip: 5.561 sec
  MCP Overhead  : 0.003 sec

Validation:
✓ correct_route
✓ correct_sources
✓ results_returned
✓ top_k_respected
✓ source_limit_respected
✓ excerpt_limit_respected
✓ expected_evidence_sources
✓ latency_recorded

THREE-ROUTE MCP VALIDATION SUMMARY


,test,expected_route,actual_route,sources_searched,evidence_sources,result_count,retrieval_time_s,tool_elapsed_s,mcp_roundtrip_s,mcp_overhead_s,passed
0,3GPP_ONLY,3gpp,3gpp,3GPP,3GPP,4,0.526,0.527,2.098,1.572,True
1,TCC_ONLY,tcc,tcc,TCC,TCC,5,4.561,4.562,4.565,0.003,True
2,HYBRID,hybrid,hybrid,"3GPP, TCC","3GPP, TCC",5,5.557,5.558,5.561,0.003,True



GLOBAL MCP VALIDATION
✓ 3GPP_ONLY
✓ TCC_ONLY
✓ HYBRID

✓ VERSION A THREE-ROUTE MCP TOOL VALIDATION PASSED
✓ 3GPP-only retrieval works through MCP.
✓ TCC-only retrieval works through MCP.
✓ Hybrid retrieval works through MCP.
✓ Dynamic routing remains internal to one MCP knowledge tool.
✓ Top-K and evidence excerpt limits enforced.
✓ Retrieval, tool and MCP round-trip timing retained.
✓ Ready for Cell 9 — Claude Setup + Frozen Shared System Prompt.


**Observation — MCP Tool Validation**

The unified Version A MCP knowledge service successfully validated all three retrieval paths through the **actual FastMCP client/server interface**.

Dedicated 3GPP retrieval completed in approximately **0.97 seconds**, TCC retrieval in **5.34 seconds**, and hybrid retrieval in **5.64 seconds**. MCP boundary overhead was negligible at only a few milliseconds across all three tests.

The hybrid path successfully returned evidence from both dedicated 3GPP and TCC sources while executing the two retrieval branches concurrently.

***Key Decision:*** Retain a single MCP knowledge tool with internal dynamic routing. Dedicated 3GPP provides a very low-latency authoritative standards path, while TCC supplies complementary non-3GPP evidence only when required. MCP remains an orchestration layer with negligible retrieval overhead.


# **SECTION 4 — DeepSeek V4 + MCP Orchestration**


## **Cell 9 — DeepSeek V4 Setup + Frozen Shared System Prompt**


In [ ]:
# ============================================================
# CELL 9 — DEEPSEEK V4 FLASH + OPENROUTER SETUP
#          + FROZEN SHARED SYSTEM PROMPT
# ============================================================

# Purpose:
#
# Configure DeepSeek V4 Flash through OpenRouter for the
# Version A end-to-end runtime.
#
# Retrieval and MCP architecture remain unchanged.
#
# Experimental controls are intentionally aligned with the
# Claude Sonnet 5, Claude Haiku 4.5 and Gemma 4 experiments:
#
#     Model               : deepseek/deepseek-v4-flash-0731
#     Provider            : OpenRouter
#     Max output tokens   : 1800
#     Temperature         : API default / not explicitly set
#     Max MCP searches    : 3
#     Final evidence      : maximum 5 sources
#     Evidence excerpt    : 2500 characters/source
#
# IMPORTANT:
#
# The system prompt below is FROZEN and shared with the
# previous Version A / Version B experiments.
#
# Do not modify its wording during DeepSeek evaluation.


# ============================================================
# OPENAI-COMPATIBLE CLIENT IMPORT
# ============================================================

from openai import OpenAI


# ============================================================
# OPENROUTER API KEY
# ============================================================

# Prefer an existing environment variable.
OPENROUTER_API_KEY = os.environ.get(
    "OPENROUTER_API_KEY"
)


# Google Colab fallback.
if not OPENROUTER_API_KEY:

    try:

        from google.colab import userdata

        OPENROUTER_API_KEY = userdata.get(
            "OPENROUTER_API_KEY"
        )

    except Exception:

        OPENROUTER_API_KEY = None


if not OPENROUTER_API_KEY:

    raise ValueError(
        "OPENROUTER_API_KEY was not found. "
        "Add it to Google Colab Secrets or the "
        "OPENROUTER_API_KEY environment variable."
    )


os.environ[
    "OPENROUTER_API_KEY"
] = OPENROUTER_API_KEY


# ============================================================
# OPENROUTER CONFIGURATION
# ============================================================

OPENROUTER_BASE_URL = (
    "https://openrouter.ai/api/v1"
)


LLM_PROVIDER = (
    "OpenRouter"
)


LLM_MODEL = (
    "deepseek/deepseek-v4-flash-0731"
)


LLM_DISPLAY_NAME = (
    "DeepSeek V4 Flash 0731"
)


LLM_MAX_TOKENS = 1800


# Temperature intentionally remains unset.
#
# This preserves the same experimental convention used
# across the other LLM evaluations:
#
#     provider/API default
#
# Cell 10 must therefore omit the temperature parameter
# when LLM_TEMPERATURE is None.
LLM_TEMPERATURE = None


# ============================================================
# EXPERIMENTAL CONTROL VALIDATION
# ============================================================

if MAX_MCP_SEARCHES != 3:

    raise RuntimeError(
        "Version A experimental control violation: "
        "MAX_MCP_SEARCHES must remain 3."
    )


if MAX_RETRIEVED_SOURCES != 5:

    raise RuntimeError(
        "Version A experimental control violation: "
        "MAX_RETRIEVED_SOURCES must remain 5."
    )


if TOP_K_RESULTS != 5:

    raise RuntimeError(
        "Version A experimental control violation: "
        "TOP_K_RESULTS must remain 5."
    )


if MCP_EXCERPT_CHARS != 2500:

    raise RuntimeError(
        "Version A experimental control violation: "
        "MCP_EXCERPT_CHARS must remain 2500."
    )


if LLM_MAX_TOKENS != 1800:

    raise RuntimeError(
        "Version A experimental control violation: "
        "LLM_MAX_TOKENS must remain 1800."
    )


# ============================================================
# OPENROUTER CLIENT
# ============================================================

openrouter = OpenAI(

    base_url=
        OPENROUTER_BASE_URL,

    api_key=
        OPENROUTER_API_KEY
)


# ============================================================
# FROZEN SHARED SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT = """
You are an expert telecommunications network engineer with access to
a telecom knowledge search tool through MCP.

Use retrieved telecom documentation as the authoritative basis for
technical answers.

Instructions:

1. Use the MCP knowledge tool when technical documentation is required.
   Use ONE search by default, a SECOND only if important evidence is
   missing, and a THIRD only as an exceptional fallback. Never exceed
   3 searches.

2. Reason across the retrieved evidence. You may connect facts, compare
   functions, explain relationships, infer technical implications, and
   synthesize information from multiple retrieved documents.

3. Do not invent facts. Any reasoning or inference must remain consistent
   with and supported by the retrieved evidence. If the evidence does not
   support a required technical point, do not fill the gap with assumptions
   or unsupported knowledge.

4. Prefer authoritative standards and technical documentation when
   evaluating conflicting or overlapping evidence. Use no more than
   5 retrieved sources for the final answer.

5. Answer every technical element requested by the user. Keep the response
   precise, practical, and suitable for a telecommunications engineer.

6. Target approximately 300–500 words unless the question clearly requires
   more detail. Prefer concise sections, bullets, or numbered steps where
   they improve clarity.

7. Do not discuss sources, citations, retrieval mechanics, BM25, DuckDB,
   shards, candidate terms, search profiles, or other internal implementation
   details unless the user explicitly asks.

8. If the retrieved documentation is insufficient to answer the question,
   state:
   "The requested details are not available in the documentation."
""".strip()


# ============================================================
# SYSTEM PROMPT SAFETY CHECKS
# ============================================================

_REQUIRED_PROMPT_FRAGMENTS = [

    "Never exceed\n   3 searches.",

    "Use no more than\n   5 retrieved sources",

    "Target approximately 300–500 words",

    (
        "The requested details are not "
        "available in the documentation."
    )
]


for fragment in _REQUIRED_PROMPT_FRAGMENTS:

    if fragment not in SYSTEM_PROMPT:

        raise RuntimeError(
            "Frozen shared system prompt validation "
            f"failed for fragment: {fragment}"
        )


# ============================================================
# PROVIDER / MODEL SAFETY CHECKS
# ============================================================

if LLM_PROVIDER != "OpenRouter":

    raise RuntimeError(
        "DeepSeek experiment requires OpenRouter."
    )


if (
    LLM_MODEL
    !=
    "deepseek/deepseek-v4-flash-0731"
):

    raise RuntimeError(
        "Unexpected model configured for "
        "the DeepSeek V4 Flash experiment."
    )


# ============================================================
# CONFIGURATION SUMMARY
# ============================================================

print("=" * 90)

print(
    "VERSION A — DEEPSEEK V4 FLASH / OPENROUTER "
    "RUNTIME CONFIGURATION"
)

print("=" * 90)


print(
    f"LLM Provider          : "
    f"{LLM_PROVIDER}"
)

print(
    f"LLM Model             : "
    f"{LLM_MODEL}"
)

print(
    f"Model Display Name    : "
    f"{LLM_DISPLAY_NAME}"
)

print(
    f"OpenRouter Base URL   : "
    f"{OPENROUTER_BASE_URL}"
)

print(
    f"Max Output Tokens     : "
    f"{LLM_MAX_TOKENS}"
)

print(
    "Temperature           : "
    "API DEFAULT"
)

print(
    f"Max MCP Searches      : "
    f"{MAX_MCP_SEARCHES}"
)

print(
    f"Top-K Retrieval       : "
    f"{TOP_K_RESULTS}"
)

print(
    f"Max Final Sources     : "
    f"{MAX_RETRIEVED_SOURCES}"
)

print(
    f"Evidence Excerpt      : "
    f"{MCP_EXCERPT_CHARS:,} chars/source"
)

print(
    f"System Prompt Chars   : "
    f"{len(SYSTEM_PROMPT):,}"
)


print(
    "\n" + "=" * 90
)

print(
    "✓ OpenRouter API key loaded."
)

print(
    "✓ OpenAI-compatible OpenRouter client initialized."
)

print(
    f"✓ {LLM_DISPLAY_NAME} configured."
)

print(
    "✓ Shared Version A/B system prompt loaded unchanged."
)

print(
    "✓ Maximum MCP search budget fixed at 3."
)

print(
    "✓ Top-K and evidence controls remain frozen."
)

print(
    "✓ No OpenRouter API request executed in this cell."
)

print(
    "✓ Ready for Cell 10 — "
    "DeepSeek V4 Flash + MCP End-to-End Orchestration."
)

print("=" * 90)

VERSION A — DEEPSEEK V4 FLASH / OPENROUTER RUNTIME CONFIGURATION
LLM Provider          : OpenRouter
LLM Model             : deepseek/deepseek-v4-flash-0731
Model Display Name    : DeepSeek V4 Flash 0731
OpenRouter Base URL   : https://openrouter.ai/api/v1
Max Output Tokens     : 1800
Temperature           : API DEFAULT
Max MCP Searches      : 3
Top-K Retrieval       : 5
Max Final Sources     : 5
Evidence Excerpt      : 2,500 chars/source
System Prompt Chars   : 1,752

✓ OpenRouter API key loaded.
✓ OpenAI-compatible OpenRouter client initialized.
✓ DeepSeek V4 Flash 0731 configured.
✓ Shared Version A/B system prompt loaded unchanged.
✓ Maximum MCP search budget fixed at 3.
✓ Top-K and evidence controls remain frozen.
✓ No OpenRouter API request executed in this cell.
✓ Ready for Cell 10 — DeepSeek V4 Flash + MCP End-to-End Orchestration.


**Observation — DeepSeek V4 Runtime Configuration**

Version A uses **DeepSeek V4 Flash 0731 (`deepseek/deepseek-v4-flash-0731`) through OpenRouter** with the frozen Benchmark v2 prompt, three-search budget, Top-K 5, 2,500-character evidence limit and 1,800-token output cap.

***Key Decision:*** Change only the model/provider integration layer; keep Version A retrieval and MCP behaviour frozen.


## **Cell 10 — End-to-End MCP Orchestration**


In [ ]:
# ============================================================
# CELL 10 — END-TO-END OPENROUTER LLM + MCP ORCHESTRATION
# ============================================================

# Purpose:
#
# Connect the configured OpenRouter LLM to the validated
# Version A MCP knowledge service.
#
# Architecture:
#
#     User Question
#          ↓
#      OpenRouter LLM
#          ↓
#      tool_calls
#          ↓
#   Input Validation
#          ↓
#       FastMCP
#          ↓
# search_telecom_knowledge()
#          ↓
# Dynamic Version A Retrieval
#          ↓
#    role="tool"
#          ↓
#      OpenRouter LLM
#          ↓
#     Final Answer
#
# Experimental controls remain unchanged:
#
#     - ONE MCP knowledge tool
#     - Maximum 3 EXECUTED MCP searches
#     - Top-K maximum 5
#     - Frozen shared system prompt
#     - No explicit temperature
#     - Full token / latency / routing trace
#
# IMPORTANT:
#
# OpenRouter uses the OpenAI-compatible tool-calling format:
#
#     request:
#         tools=[{"type": "function", ...}]
#
#     response:
#         message.tool_calls
#
#     tool result:
#         role="tool"
#         tool_call_id=<matching call id>


# ============================================================
# GENERIC NORMALIZATION HELPERS
# ============================================================

def to_plain_dict(value):
    """
    Convert common dictionary-like objects into a plain dict.

    Handles:
        - dict
        - Pydantic v2 model_dump()
        - Pydantic v1 dict()
        - JSON strings
    """

    if value is None:
        return {}

    if isinstance(value, dict):
        return dict(value)

    model_dump = getattr(
        value,
        "model_dump",
        None
    )

    if callable(model_dump):

        try:

            dumped = model_dump()

            if isinstance(dumped, dict):
                return dict(dumped)

        except Exception:
            pass

    dict_method = getattr(
        value,
        "dict",
        None
    )

    if callable(dict_method):

        try:

            dumped = dict_method()

            if isinstance(dumped, dict):
                return dict(dumped)

        except Exception:
            pass

    if isinstance(value, str):

        try:

            parsed = json.loads(value)

            if isinstance(parsed, dict):
                return parsed

        except Exception:
            pass

    return {}


def safe_int_value(
    value,
    default=0
):
    """
    Safely convert a scalar to int.
    """

    if isinstance(value, bool):
        return int(default)

    if value is None:
        return int(default)

    try:
        return int(value)
    except Exception:
        pass

    try:
        return int(
            float(
                str(value).strip()
            )
        )
    except Exception:
        return int(default)


def safe_float_value(
    value,
    default=0.0
):
    """
    Safely convert a scalar to float.
    """

    try:

        if value is None:
            return float(default)

        return float(value)

    except Exception:

        return float(default)


def safe_list_value(value):
    """
    Safely normalize a list-like object.
    """

    if value is None:
        return []

    if isinstance(
        value,
        list
    ):
        return list(value)

    if isinstance(
        value,
        tuple
    ):
        return list(value)

    if isinstance(
        value,
        set
    ):
        return list(value)

    return [value]


# ============================================================
# FASTMCP INPUT SCHEMA NORMALIZATION
# ============================================================

def normalize_mcp_input_schema(schema):
    """
    Convert FastMCP input_schema to a plain JSON Schema
    suitable for OpenAI-compatible function calling.
    """

    schema_dict = to_plain_dict(
        schema
    )

    if not schema_dict:

        raise RuntimeError(
            "FastMCP tool input_schema could not "
            "be converted to a dictionary."
        )

    schema_type = schema_dict.get(
        "type"
    )

    if schema_type is None:

        schema_dict[
            "type"
        ] = "object"

    elif schema_type != "object":

        raise RuntimeError(
            "Expected MCP tool input_schema type "
            f"'object', received: {schema_type}"
        )

    properties = schema_dict.get(
        "properties"
    )

    if properties is None:

        schema_dict[
            "properties"
        ] = {}

    elif not isinstance(
        properties,
        dict
    ):

        raise RuntimeError(
            "MCP input_schema.properties "
            "must be a dictionary."
        )

    required = schema_dict.get(
        "required"
    )

    if (
        required is not None
        and
        not isinstance(
            required,
            list
        )
    ):

        schema_dict[
            "required"
        ] = list(
            required
        )

    return schema_dict


# ============================================================
# FASTMCP → OPENAI-COMPATIBLE TOOL CONVERSION
# ============================================================

def mcp_tool_to_openrouter(tool):
    """
    Convert a FastMCP tool into the OpenAI-compatible
    function-tool schema expected by OpenRouter.
    """

    tool_name = getattr(
        tool,
        "name",
        None
    )

    tool_description = (
        getattr(
            tool,
            "description",
            ""
        )
        or ""
    )

    input_schema = getattr(
        tool,
        "input_schema",
        None
    )

    if not tool_name:

        raise RuntimeError(
            "MCP tool is missing a name."
        )

    if input_schema is None:

        raise RuntimeError(
            f"MCP tool '{tool_name}' "
            "is missing input_schema."
        )

    normalized_schema = (
        normalize_mcp_input_schema(
            input_schema
        )
    )

    return {

        "type":
            "function",

        "function": {

            "name":
                str(
                    tool_name
                ),

            "description":
                str(
                    tool_description
                ),

            "parameters":
                normalized_schema
        }
    }


# ============================================================
# DISCOVER OPENROUTER-ACCESSIBLE MCP TOOL
# ============================================================

async with Client(
    mcp
) as _tool_discovery_client:

    _available_mcp_tools = (
        await _tool_discovery_client.list_tools()
    )


_version_a_search_tools = [

    tool

    for tool
    in _available_mcp_tools

    if (
        getattr(
            tool,
            "name",
            ""
        )
        ==
        "search_telecom_knowledge"
    )
]


if len(
    _version_a_search_tools
) != 1:

    raise RuntimeError(
        "Expected exactly one "
        "search_telecom_knowledge MCP tool. "
        f"Found {len(_version_a_search_tools)}."
    )


OPENROUTER_MCP_TOOLS = [

    mcp_tool_to_openrouter(
        _version_a_search_tools[
            0
        ]
    )
]


# ============================================================
# TOOL INPUT NORMALIZATION
# ============================================================

def normalize_telecom_tool_input(
    raw_input
):
    """
    Validate and normalize LLM-generated arguments before
    forwarding them to FastMCP.

    Allowed arguments:

        query
        top_k
    """

    tool_input = to_plain_dict(
        raw_input
    )

    # --------------------------------------------------------
    # QUERY
    # --------------------------------------------------------

    raw_query = tool_input.get(
        "query"
    )

    if raw_query is None:

        return (
            None,
            "Tool input is missing required field 'query'."
        )

    if not isinstance(
        raw_query,
        str
    ):

        try:

            raw_query = str(
                raw_query
            )

        except Exception:

            return (
                None,
                "Tool input 'query' could not be converted "
                "to text."
            )

    query = re.sub(
        r"\s+",
        " ",
        raw_query
    ).strip()

    if not query:

        return (
            None,
            "Tool input 'query' must contain non-empty text."
        )

    # --------------------------------------------------------
    # TOP-K
    # --------------------------------------------------------

    raw_top_k = tool_input.get(
        "top_k",
        TOP_K_RESULTS
    )

    top_k = safe_int_value(
        raw_top_k,
        default=
            TOP_K_RESULTS
    )

    top_k = max(
        1,
        min(
            top_k,
            TOP_K_RESULTS,
            MAX_RETRIEVED_SOURCES
        )
    )

    # --------------------------------------------------------
    # RETURN ONLY SUPPORTED ARGUMENTS
    # --------------------------------------------------------

    normalized_input = {

        "query":
            query,

        "top_k":
            top_k
    }

    return (
        normalized_input,
        None
    )


# ============================================================
# OPENROUTER RESPONSE HELPERS
# ============================================================

def get_openrouter_message(
    response
):
    """
    Safely extract the first assistant message.
    """

    choices = getattr(
        response,
        "choices",
        []
    ) or []

    if not choices:

        raise RuntimeError(
            "OpenRouter returned no completion choices."
        )

    message = getattr(
        choices[0],
        "message",
        None
    )

    if message is None:

        raise RuntimeError(
            "OpenRouter response contained no assistant message."
        )

    return message


def extract_openrouter_text(
    response
):
    """
    Extract assistant text safely.
    """

    message = get_openrouter_message(
        response
    )

    content = getattr(
        message,
        "content",
        ""
    )

    if content is None:
        return ""

    if isinstance(
        content,
        str
    ):
        return content.strip()

    # Defensive support for structured content.
    if isinstance(
        content,
        list
    ):

        text_parts = []

        for item in content:

            item_dict = to_plain_dict(
                item
            )

            text_value = (
                item_dict.get(
                    "text"
                )
                or
                getattr(
                    item,
                    "text",
                    ""
                )
            )

            if text_value:

                text_parts.append(
                    str(
                        text_value
                    )
                )

        return "\n".join(
            text_parts
        ).strip()

    return str(
        content
    ).strip()


def extract_openrouter_tool_calls(
    response
):
    """
    Extract OpenAI-compatible tool_calls.
    """

    message = get_openrouter_message(
        response
    )

    tool_calls = getattr(
        message,
        "tool_calls",
        None
    )

    if not tool_calls:
        return []

    return list(
        tool_calls
    )


def extract_usage(
    response
):
    """
    Extract OpenAI/OpenRouter token usage.

    Mapped to the same input/output labels used by
    the previous model experiments.
    """

    usage = getattr(
        response,
        "usage",
        None
    )

    if usage is None:

        return {

            "input_tokens":
                0,

            "output_tokens":
                0
        }

    return {

        "input_tokens":
            safe_int_value(
                getattr(
                    usage,
                    "prompt_tokens",
                    0
                ),
                0
            ),

        "output_tokens":
            safe_int_value(
                getattr(
                    usage,
                    "completion_tokens",
                    0
                ),
                0
            )
    }


def get_raw_finish_reason(
    response
):
    """
    Return OpenAI-compatible finish_reason.
    """

    choices = getattr(
        response,
        "choices",
        []
    ) or []

    if not choices:
        return None

    return getattr(
        choices[0],
        "finish_reason",
        None
    )


def normalize_finish_reason(
    finish_reason
):
    """
    Normalize OpenAI/OpenRouter finish reasons into the
    labels used by the existing Version A/B evaluation.

        stop       → end_turn
        length     → max_tokens
        tool_calls → tool_use

    This preserves comparison compatibility.
    """

    mapping = {

        "stop":
            "end_turn",

        "length":
            "max_tokens",

        "tool_calls":
            "tool_use"
    }

    if finish_reason is None:
        return None

    return mapping.get(
        str(
            finish_reason
        ),
        str(
            finish_reason
        )
    )


def serialize_openrouter_assistant_message(
    response
):
    """
    Preserve an OpenRouter assistant response in conversation
    history, including all tool calls.
    """

    message = get_openrouter_message(
        response
    )

    serialized = {

        "role":
            "assistant"
    }

    content = getattr(
        message,
        "content",
        None
    )

    if content is not None:

        serialized[
            "content"
        ] = content

    tool_calls = getattr(
        message,
        "tool_calls",
        None
    )

    if tool_calls:

        serialized[
            "tool_calls"
        ] = []

        for tool_call in tool_calls:

            function = getattr(
                tool_call,
                "function",
                None
            )

            serialized[
                "tool_calls"
            ].append({

                "id":
                    getattr(
                        tool_call,
                        "id",
                        None
                    ),

                "type":
                    "function",

                "function": {

                    "name":
                        getattr(
                            function,
                            "name",
                            ""
                        ),

                    "arguments":
                        getattr(
                            function,
                            "arguments",
                            "{}"
                        )
                }
            })

    return serialized


# ============================================================
# MCP RESPONSE EXTRACTION
# ============================================================

def extract_mcp_payload(
    tool_result
):
    """
    Extract structured dictionary from FastMCP CallToolResult.
    """

    payload = getattr(
        tool_result,
        "data",
        None
    )

    if payload is None:

        payload = getattr(
            tool_result,
            "structured_content",
            None
        )

    payload = to_plain_dict(
        payload
    )

    if not payload:

        raise RuntimeError(
            "MCP telecom knowledge tool did not return "
            "a usable structured response."
        )

    return payload


# ============================================================
# EVIDENCE TRACE EXTRACTION
# ============================================================

def extract_evidence_trace(
    payload
):
    """
    Preserve the complete bounded MCP evidence returned to the LLM.

    The retained evidence enables downstream evaluation of:

        - retrieval relevance
        - source authority
        - technical correctness
        - answer groundedness
        - claim-to-evidence support
        - unsupported-claim / hallucination risk

    Only the bounded MCP excerpts are stored. Full source
    documents are never copied into the evaluation artifact.
    """

    evidence_items = payload.get(
        "evidence",
        []
    )

    if not isinstance(
        evidence_items,
        list
    ):
        evidence_items = []


    evidence_trace = []


    for raw_item in evidence_items:

        item = to_plain_dict(
            raw_item
        )

        if not item:
            continue


        evidence_text = str(
            item.get(
                "evidence",
                ""
            )
            or ""
        )


        evidence_trace.append({

            # ------------------------------------------------
            # RANKING
            # ------------------------------------------------

            "rank":
                item.get(
                    "rank"
                ),

            "relevance_score":
                item.get(
                    "relevance_score"
                ),

            "matched_terms":
                item.get(
                    "matched_terms"
                ),

            "matched_phrases":
                item.get(
                    "matched_phrases"
                ),

            "proximity_score":
                item.get(
                    "proximity_score"
                ),

            "primary_coverage":
                item.get(
                    "primary_coverage"
                ),

            # ------------------------------------------------
            # SOURCE PROVENANCE
            # ------------------------------------------------

            "source_family":
                item.get(
                    "source_family"
                ),

            "collection":
                item.get(
                    "collection"
                ),

            "identifier":
                item.get(
                    "identifier"
                ),

            "release":
                item.get(
                    "release"
                ),

            "title":
                item.get(
                    "title"
                ),

            "section_heading":
                item.get(
                    "section_heading"
                ),

            "source_path":
                item.get(
                    "source_path"
                ),

            "source_shard":
                item.get(
                    "source_shard"
                ),

            # ------------------------------------------------
            # GROUNDEDNESS EVIDENCE
            # ------------------------------------------------

            "evidence":
                evidence_text,

            "evidence_chars":
                len(
                    evidence_text
                )
        })


    return evidence_trace


# ============================================================
# SINGLE OPENROUTER API CALL
# ============================================================

def call_openrouter(
    messages,
    tools=None
):
    """
    Execute one OpenRouter Chat Completions request.

    Temperature intentionally remains unset.

    The frozen SYSTEM_PROMPT is carried as the first system
    message in the conversation.
    """

    if not isinstance(
        messages,
        list
    ):

        raise TypeError(
            "OpenRouter messages must be supplied as a list."
        )

    if not messages:

        raise ValueError(
            "OpenRouter messages must not be empty."
        )

    request_args = {

        "model":
            LLM_MODEL,

        "max_tokens":
            LLM_MAX_TOKENS,

        "messages":
            messages
    }

    if tools:

        request_args[
            "tools"
        ] = tools

        request_args[
            "tool_choice"
        ] = "auto"

    # LLM_TEMPERATURE intentionally remains None.
    #
    # Therefore no temperature argument is sent.

    return (
        openrouter
        .chat
        .completions
        .create(
            **request_args
        )
    )


# ============================================================
# END-TO-END OPENROUTER LLM + MCP ORCHESTRATION
# ============================================================

async def run_version_a_e2e(
    mcp_client,
    question,
    verbose=True
):
    """
    Execute one complete Version A question through:

        OpenRouter LLM
              ↓
             MCP
              ↓
        Dynamic Retrieval
              ↓
        OpenRouter LLM

    Maximum EXECUTED MCP searches:
        MAX_MCP_SEARCHES = 3

    Invalid tool arguments do not execute retrieval and therefore
    do not count as an MCP search.

    Returns a structured record suitable for:
        - pilot validation
        - 8-question evaluation
        - Version A/B analysis
        - cross-LLM analysis
    """

    # ========================================================
    # QUESTION VALIDATION
    # ========================================================

    if question is None:

        raise ValueError(
            "Question must not be None."
        )

    if not isinstance(
        question,
        str
    ):

        question = str(
            question
        )

    question = re.sub(
        r"\s+",
        " ",
        question
    ).strip()

    if not question:

        raise ValueError(
            "Question must not be empty."
        )

    # ========================================================
    # EXECUTION STATE
    # ========================================================

    e2e_start = (
        time.perf_counter()
    )

    messages = [

        {
            "role":
                "system",

            "content":
                SYSTEM_PROMPT
        },

        {
            "role":
                "user",

            "content":
                question
        }
    ]

    # Successfully dispatched MCP searches.
    tool_call_count = 0

    # Every model tool request, including malformed ones.
    tool_request_count = 0

    llm_call_count = 0

    tool_queries = []
    tool_routes = []
    tool_sources = []
    tool_source_distributions = []

    retrieval_times = []
    tool_elapsed_times = []
    mcp_roundtrip_times = []
    mcp_overheads = []

    evidence_trace = []
    error_log = []
    turn_usage = []

    total_input_tokens = 0
    total_output_tokens = 0

    final_answer = ""
    final_answer_tokens = 0

    final_stop_reason = None
    final_raw_finish_reason = None

    completion_mode = None

    # ========================================================
    # OPTIONAL HEADER
    # ========================================================

    if verbose:

        print(
            "=" * 90
        )

        print(
            "VERSION A — OPENROUTER LLM + MCP "
            "END-TO-END EXECUTION"
        )

        print(
            "=" * 90
        )

        print(
            f"Question        : "
            f"{question}"
        )

        print(
            f"LLM Provider    : "
            f"{LLM_PROVIDER}"
        )

        print(
            f"LLM Model       : "
            f"{LLM_MODEL}"
        )

        print(
            f"Max MCP Searches: "
            f"{MAX_MCP_SEARCHES}"
        )

    # ========================================================
    # LLM / MCP LOOP
    # ========================================================

    while True:

        # ----------------------------------------------------
        # LLM TURN
        # ----------------------------------------------------

        llm_start = (
            time.perf_counter()
        )

        try:

            response = call_openrouter(

                messages=
                    messages,

                tools=
                    OPENROUTER_MCP_TOOLS
            )

        except Exception as exc:

            error_log.append({

                "stage":
                    "openrouter_api",

                "error_type":
                    type(
                        exc
                    ).__name__,

                "error":
                    str(
                        exc
                    )
            })

            raise

        llm_elapsed_s = (
            time.perf_counter()
            -
            llm_start
        )

        llm_call_count += 1

        usage = extract_usage(
            response
        )

        total_input_tokens += (
            usage[
                "input_tokens"
            ]
        )

        total_output_tokens += (
            usage[
                "output_tokens"
            ]
        )

        raw_finish_reason = (
            get_raw_finish_reason(
                response
            )
        )

        tool_calls = (
            extract_openrouter_tool_calls(
                response
            )
        )

        turn_usage.append({

            "turn":
                llm_call_count,

            "input_tokens":
                usage[
                    "input_tokens"
                ],

            "output_tokens":
                usage[
                    "output_tokens"
                ],

            "elapsed_s":
                round(
                    llm_elapsed_s,
                    6
                ),

            "finish_reason":
                raw_finish_reason,

            "tool_calls":
                len(
                    tool_calls
                )
        })

        # ----------------------------------------------------
        # PRESERVE COMPLETE ASSISTANT MESSAGE
        # ----------------------------------------------------

        messages.append(
            serialize_openrouter_assistant_message(
                response
            )
        )

        # ====================================================
        # NO TOOL REQUEST → FINAL ANSWER
        # ====================================================

        if not tool_calls:

            final_answer = (
                extract_openrouter_text(
                    response
                )
            )

            final_answer_tokens = (
                usage[
                    "output_tokens"
                ]
            )

            final_raw_finish_reason = (
                raw_finish_reason
            )

            final_stop_reason = (
                normalize_finish_reason(
                    raw_finish_reason
                )
            )

            completion_mode = (
                "normal"
            )

            if verbose:

                print(
                    "\nLLM completed without "
                    "another MCP request."
                )

                print(
                    f"LLM Turn Time: "
                    f"{llm_elapsed_s:.3f} sec"
                )

            break

        # ====================================================
        # PROCESS TOOL REQUESTS
        # ====================================================

        tool_messages = []

        for tool_call in tool_calls:

            tool_request_count += 1

            tool_call_id = getattr(
                tool_call,
                "id",
                None
            )

            function = getattr(
                tool_call,
                "function",
                None
            )

            tool_name = getattr(
                function,
                "name",
                ""
            )

            raw_arguments = getattr(
                function,
                "arguments",
                "{}"
            )

            # ------------------------------------------------
            # TOOL CALL ID VALIDATION
            # ------------------------------------------------

            if not tool_call_id:

                raise RuntimeError(
                    "LLM returned a tool call "
                    "without an id."
                )

            # ------------------------------------------------
            # UNKNOWN TOOL
            # ------------------------------------------------

            if (
                tool_name
                !=
                "search_telecom_knowledge"
            ):

                error_message = (
                    f"Unsupported tool requested: "
                    f"{tool_name or 'UNKNOWN'}"
                )

                error_log.append({

                    "stage":
                        "tool_selection",

                    "tool":
                        tool_name,

                    "error":
                        error_message
                })

                tool_messages.append({

                    "role":
                        "tool",

                    "tool_call_id":
                        tool_call_id,

                    "content":
                        error_message
                })

                continue

            # ------------------------------------------------
            # NORMALIZE / VALIDATE TOOL INPUT
            # ------------------------------------------------

            (
                normalized_input,
                input_error
            ) = normalize_telecom_tool_input(
                raw_arguments
            )

            if input_error is not None:

                error_log.append({

                    "stage":
                        "tool_input_validation",

                    "tool":
                        tool_name,

                    "error":
                        input_error
                })

                tool_messages.append({

                    "role":
                        "tool",

                    "tool_call_id":
                        tool_call_id,

                    "content":
                        (
                            "Invalid tool input: "
                            f"{input_error}"
                        )
                })

                if verbose:

                    print(
                        "\nMCP TOOL INPUT ERROR"
                    )

                    print(
                        input_error
                    )

                continue

            query = (
                normalized_input[
                    "query"
                ]
            )

            # ------------------------------------------------
            # MCP SEARCH CAP
            # ------------------------------------------------

            if (
                tool_call_count
                >=
                MAX_MCP_SEARCHES
            ):

                tool_messages.append({

                    "role":
                        "tool",

                    "tool_call_id":
                        tool_call_id,

                    "content":
                        (
                            "MCP search was not executed because "
                            "the maximum search budget has already "
                            "been reached."
                        )
                })

                continue

            # =================================================
            # EXECUTE VALID MCP SEARCH
            # =================================================

            tool_call_count += 1

            tool_queries.append(
                query
            )

            if verbose:

                print(
                    f"\nMCP SEARCH "
                    f"{tool_call_count}/"
                    f"{MAX_MCP_SEARCHES}"
                )

                print(
                    f"Query : "
                    f"{query}"
                )

                print(
                    f"Top-K : "
                    f"{normalized_input['top_k']}"
                )

            mcp_start = (
                time.perf_counter()
            )

            try:

                tool_result = (
                    await mcp_client.call_tool(

                        tool_name,

                        normalized_input
                    )
                )

                mcp_roundtrip_s = (
                    time.perf_counter()
                    -
                    mcp_start
                )

                payload = (
                    extract_mcp_payload(
                        tool_result
                    )
                )

                trace = to_plain_dict(
                    payload.get(
                        "trace",
                        {}
                    )
                )

                retrieval_time_s = (
                    safe_float_value(
                        trace.get(
                            "retrieval_time_s",
                            0
                        )
                    )
                )

                tool_elapsed_s = (
                    safe_float_value(
                        trace.get(
                            "tool_elapsed_s",
                            0
                        )
                    )
                )

                mcp_overhead_s = max(

                    0.0,

                    (
                        mcp_roundtrip_s
                        -
                        tool_elapsed_s
                    )
                )

                route = str(
                    payload.get(
                        "route",
                        ""
                    )
                    or ""
                )

                sources = [

                    str(
                        source
                    )

                    for source
                    in safe_list_value(
                        payload.get(
                            "sources_searched",
                            []
                        )
                    )
                ]

                source_distribution = (
                    to_plain_dict(
                        payload.get(
                            "source_distribution",
                            {}
                        )
                    )
                )

                current_evidence = (
                    extract_evidence_trace(
                        payload
                    )
                )

                # --------------------------------------------
                # TRACE
                # --------------------------------------------

                tool_routes.append(
                    route
                )

                tool_sources.append(
                    sources
                )

                tool_source_distributions.append(
                    source_distribution
                )

                retrieval_times.append(
                    retrieval_time_s
                )

                tool_elapsed_times.append(
                    tool_elapsed_s
                )

                mcp_roundtrip_times.append(
                    mcp_roundtrip_s
                )

                mcp_overheads.append(
                    mcp_overhead_s
                )


                # ------------------------------------------------------------
                # COMPLETE SEARCH TRACE FOR EVALUATION
                # ------------------------------------------------------------

                retrieval_trace = to_plain_dict(
                    payload.get(
                        "trace",
                        {}
                    )
                )


                evidence_trace.append({

                    # --------------------------------------------------------
                    # SEARCH IDENTITY
                    # --------------------------------------------------------

                    "search_number":
                        tool_call_count,

                    "query":
                        query,

                    "top_k":
                        payload.get(
                            "top_k"
                        ),

                    # --------------------------------------------------------
                    # ROUTING
                    # --------------------------------------------------------

                    "route":
                        route,

                    "sources_searched":
                        sources,

                    "source_distribution":
                        source_distribution,

                    "retrieval_trace":
                        retrieval_trace,

                    # --------------------------------------------------------
                    # RESULTS
                    # --------------------------------------------------------

                    "result_count":
                        payload.get(
                            "result_count",
                            len(
                                current_evidence
                            )
                        ),

                    "results":
                        current_evidence,

                    # --------------------------------------------------------
                    # SEARCH-LEVEL TIMING
                    # --------------------------------------------------------

                    "retrieval_time_s":
                        retrieval_time_s,

                    "tool_elapsed_s":
                        tool_elapsed_s,

                    "mcp_roundtrip_s":
                        mcp_roundtrip_s,

                    "mcp_overhead_s":
                        mcp_overhead_s
                })


                # --------------------------------------------
                # MCP PAYLOAD → OPENAI TOOL MESSAGE
                # --------------------------------------------

                tool_result_text = json.dumps(

                    payload,

                    ensure_ascii=False,

                    default=str
                )

                tool_messages.append({

                    "role":
                        "tool",

                    "tool_call_id":
                        tool_call_id,

                    "content":
                        tool_result_text
                })

                if verbose:

                    print(
                        f"Route     : "
                        f"{route.upper()}"
                    )

                    print(
                        f"Sources   : "
                        f"{sources}"
                    )

                    print(
                        f"Results   : "
                        f"{payload.get('result_count', 0)}"
                    )

                    print(
                        f"Retrieval : "
                        f"{retrieval_time_s:.3f} sec"
                    )

                    print(
                        f"MCP Round : "
                        f"{mcp_roundtrip_s:.3f} sec"
                    )

            # =================================================
            # MCP EXECUTION ERROR
            # =================================================

            except Exception as exc:

                mcp_roundtrip_s = (
                    time.perf_counter()
                    -
                    mcp_start
                )

                mcp_roundtrip_times.append(
                    mcp_roundtrip_s
                )

                error_entry = {

                    "stage":
                        "mcp_tool_call",

                    "search_number":
                        tool_call_count,

                    "tool":
                        tool_name,

                    "query":
                        query,

                    "error_type":
                        type(
                            exc
                        ).__name__,

                    "error":
                        str(
                            exc
                        )
                }

                error_log.append(
                    error_entry
                )

                tool_messages.append({

                    "role":
                        "tool",

                    "tool_call_id":
                        tool_call_id,

                    "content":
                        (
                            "Tool execution failed: "
                            f"{type(exc).__name__}: "
                            f"{exc}"
                        )
                })

                if verbose:

                    print(
                        f"MCP ERROR: "
                        f"{type(exc).__name__}: "
                        f"{exc}"
                    )

        # ====================================================
        # OPENAI / OPENROUTER TOOL MESSAGE ORDERING
        # ====================================================
        #
        # One role="tool" message is returned for every tool_call,
        # each carrying the corresponding tool_call_id.
        # ====================================================

        messages.extend(
            tool_messages
        )

        search_cap_reached = (
            tool_call_count
            >=
            MAX_MCP_SEARCHES
        )

        # ====================================================
        # SEARCH CAP → FINAL LLM TURN WITHOUT TOOLS
        # ====================================================

        if search_cap_reached:

            messages.append({

                "role":
                    "user",

                "content":
                    (
                        "The external MCP search budget has now "
                        "been reached. Using only the telecom "
                        "evidence already retrieved, provide the "
                        "final answer. Do not request another tool."
                    )
            })

            if verbose:

                print(
                    "\nMaximum MCP search budget reached."
                )

                print(
                    "Requesting final answer using "
                    "retrieved evidence only..."
                )

            final_start = (
                time.perf_counter()
            )

            try:

                final_response = call_openrouter(

                    messages=
                        messages,

                    tools=
                        None
                )

            except Exception as exc:

                error_log.append({

                    "stage":
                        "openrouter_final_answer",

                    "error_type":
                        type(
                            exc
                        ).__name__,

                    "error":
                        str(
                            exc
                        )
                })

                raise

            final_llm_elapsed_s = (
                time.perf_counter()
                -
                final_start
            )

            llm_call_count += 1

            final_usage = (
                extract_usage(
                    final_response
                )
            )

            total_input_tokens += (
                final_usage[
                    "input_tokens"
                ]
            )

            total_output_tokens += (
                final_usage[
                    "output_tokens"
                ]
            )

            final_raw_finish_reason = (
                get_raw_finish_reason(
                    final_response
                )
            )

            final_stop_reason = (
                normalize_finish_reason(
                    final_raw_finish_reason
                )
            )

            final_answer = (
                extract_openrouter_text(
                    final_response
                )
            )

            final_answer_tokens = (
                final_usage[
                    "output_tokens"
                ]
            )

            turn_usage.append({

                "turn":
                    llm_call_count,

                "input_tokens":
                    final_usage[
                        "input_tokens"
                    ],

                "output_tokens":
                    final_usage[
                        "output_tokens"
                    ],

                "elapsed_s":
                    round(
                        final_llm_elapsed_s,
                        6
                    ),

                "finish_reason":
                    final_raw_finish_reason,

                "tool_calls":
                    0
            })

            completion_mode = (
                "forced_after_search_cap"
            )

            if verbose:

                print(
                    f"Final LLM Turn: "
                    f"{final_llm_elapsed_s:.3f} sec"
                )

            break

    # ========================================================
    # FINAL EXECUTION METRICS
    # ========================================================

    e2e_time_s = (
        time.perf_counter()
        -
        e2e_start
    )

    total_retrieval_time_s = sum(
        retrieval_times
    )

    total_mcp_roundtrip_s = sum(
        mcp_roundtrip_times
    )

    total_mcp_overhead_s = sum(
        mcp_overheads
    )

    final_word_count = len(
        final_answer.split()
    )

    # ========================================================
    # FINAL RESULT OBJECT
    # ========================================================

    result = {

        "architecture":
            ARCHITECTURE_VERSION,

        "retrieval_architecture":
            RETRIEVAL_ARCHITECTURE,

        "provider":
            LLM_PROVIDER,

        "model":
            LLM_MODEL,

        "model_display_name":
            LLM_DISPLAY_NAME,

        "question":
            question,

        "completed":
            bool(
                final_answer.strip()
            ),

        "completion_mode":
            completion_mode,

        # Normalized to existing Version A/B labels.
        "stop_reason":
            final_stop_reason,

        # Provider-native value retained as well.
        "raw_finish_reason":
            final_raw_finish_reason,

        "final_answer":
            final_answer,

        "word_count":
            final_word_count,

        # ----------------------------------------------------
        # LLM
        # ----------------------------------------------------

        "llm_calls":
            llm_call_count,

        # Compatibility alias for existing evaluation code.
        # This can be removed once Cell 12 becomes provider-neutral.
        "claude_calls":
            llm_call_count,

        "input_tokens":
            total_input_tokens,

        "output_tokens":
            total_output_tokens,

        "total_tokens":
            (
                total_input_tokens
                +
                total_output_tokens
            ),

        "final_answer_tokens":
            final_answer_tokens,

        "turn_usage":
            turn_usage,

        # ----------------------------------------------------
        # MCP
        # ----------------------------------------------------

        "mcp_tool_requests":
            tool_request_count,

        "mcp_searches":
            tool_call_count,

        "tool_queries":
            tool_queries,

        "tool_routes":
            tool_routes,

        "tool_sources":
            tool_sources,

        "tool_source_distributions":
            tool_source_distributions,

        # ----------------------------------------------------
        # EVIDENCE
        # ----------------------------------------------------

        "evidence_trace":
            evidence_trace,

        # ----------------------------------------------------
        # TIMING
        # ----------------------------------------------------

        "retrieval_times_s":
            retrieval_times,

        "tool_elapsed_times_s":
            tool_elapsed_times,

        "mcp_roundtrip_times_s":
            mcp_roundtrip_times,

        "mcp_overheads_s":
            mcp_overheads,

        "total_retrieval_time_s":
            total_retrieval_time_s,

        "total_mcp_roundtrip_s":
            total_mcp_roundtrip_s,

        "total_mcp_overhead_s":
            total_mcp_overhead_s,

        "e2e_time_s":
            e2e_time_s,

        # ----------------------------------------------------
        # ERRORS
        # ----------------------------------------------------

        "errors":
            error_log
    }

    # ========================================================
    # OPTIONAL EXECUTION SUMMARY
    # ========================================================

    if verbose:

        print(
            "\n" + "=" * 90
        )

        print(
            "VERSION A — OPENROUTER LLM E2E EXECUTION SUMMARY"
        )

        print(
            "=" * 90
        )

        print(
            f"Completed          : "
            f"{result['completed']}"
        )

        print(
            f"Completion Mode    : "
            f"{completion_mode}"
        )

        print(
            f"Stop Reason        : "
            f"{final_stop_reason}"
        )

        print(
            f"Raw Finish Reason  : "
            f"{final_raw_finish_reason}"
        )

        print(
            f"LLM Calls          : "
            f"{llm_call_count}"
        )

        print(
            f"MCP Tool Requests  : "
            f"{tool_request_count}"
        )

        print(
            f"MCP Searches       : "
            f"{tool_call_count}"
        )

        print(
            f"Tool Routes        : "
            f"{tool_routes}"
        )

        print(
            f"Retrieval Time     : "
            f"{total_retrieval_time_s:.3f} sec"
        )

        print(
            f"MCP Round Trip     : "
            f"{total_mcp_roundtrip_s:.3f} sec"
        )

        print(
            f"E2E Time           : "
            f"{e2e_time_s:.3f} sec"
        )

        print(
            f"Input Tokens       : "
            f"{total_input_tokens:,}"
        )

        print(
            f"Output Tokens      : "
            f"{total_output_tokens:,}"
        )

        print(
            f"Total Tokens       : "
            f"{result['total_tokens']:,}"
        )

        print(
            f"Final Answer Tokens: "
            f"{final_answer_tokens:,}"
        )

        print(
            f"Final Word Count   : "
            f"{final_word_count:,}"
        )

        print(
            f"Errors             : "
            f"{len(error_log)}"
        )

        print(
            "\n" + "=" * 90
        )

        print(
            "FINAL ANSWER"
        )

        print(
            "=" * 90
        )

        print(
            final_answer
        )

        print(
            "=" * 90
        )

    return result


# ============================================================
# STATIC ORCHESTRATION VALIDATION
# ============================================================

if len(
    OPENROUTER_MCP_TOOLS
) != 1:

    raise RuntimeError(
        "Configured LLM must receive exactly one "
        "MCP knowledge-search tool."
    )


if (
    OPENROUTER_MCP_TOOLS[
        0
    ][
        "type"
    ]
    !=
    "function"
):

    raise RuntimeError(
        "OpenRouter MCP tool is not "
        "declared as a function."
    )


if (
    OPENROUTER_MCP_TOOLS[
        0
    ][
        "function"
    ][
        "name"
    ]
    !=
    "search_telecom_knowledge"
):

    raise RuntimeError(
        "Unexpected OpenRouter MCP tool."
    )


if (
    OPENROUTER_MCP_TOOLS[
        0
    ][
        "function"
    ][
        "parameters"
    ].get(
        "type"
    )
    !=
    "object"
):

    raise RuntimeError(
        "OpenRouter function parameters are not "
        "a valid object schema."
    )


if MAX_MCP_SEARCHES != 3:

    raise RuntimeError(
        "Maximum MCP search budget "
        "must remain 3."
    )


if TOP_K_RESULTS != 5:

    raise RuntimeError(
        "TOP_K_RESULTS must remain 5."
    )


if MAX_RETRIEVED_SOURCES != 5:

    raise RuntimeError(
        "MAX_RETRIEVED_SOURCES must remain 5."
    )


if LLM_MAX_TOKENS != 1800:

    raise RuntimeError(
        "LLM_MAX_TOKENS must remain 1800."
    )


# ============================================================
# TOOL INPUT NORMALIZATION SELF-TEST
# ============================================================

_test_valid_input, _test_valid_error = (
    normalize_telecom_tool_input({

        "query":
            "  Explain   AMF registration  ",

        "top_k":
            "5",

        "unexpected_parameter":
            "ignored"
    })
)


if _test_valid_error is not None:

    raise RuntimeError(
        "Tool input normalization self-test failed."
    )


if (
    _test_valid_input
    !=
    {
        "query":
            "Explain AMF registration",

        "top_k":
            5
    }
):

    raise RuntimeError(
        "Tool input normalization produced "
        "unexpected output."
    )


_test_missing_query, _test_missing_error = (
    normalize_telecom_tool_input({
        "top_k":
            5
    })
)


if (
    _test_missing_query is not None
    or
    _test_missing_error is None
):

    raise RuntimeError(
        "Missing-query validation self-test failed."
    )


_test_clamped_input, _ = (
    normalize_telecom_tool_input({

        "query":
            "Explain PFCP",

        "top_k":
            999
    })
)


if (
    _test_clamped_input[
        "top_k"
    ]
    !=
    5
):

    raise RuntimeError(
        "Top-K clamping self-test failed."
    )


# ============================================================
# FINISH-REASON NORMALIZATION SELF-TEST
# ============================================================

if (
    normalize_finish_reason(
        "stop"
    )
    !=
    "end_turn"
):

    raise RuntimeError(
        "Finish-reason normalization failed for 'stop'."
    )


if (
    normalize_finish_reason(
        "length"
    )
    !=
    "max_tokens"
):

    raise RuntimeError(
        "Finish-reason normalization failed for 'length'."
    )


# ============================================================
# CONFIGURATION SUMMARY
# ============================================================

print(
    "=" * 90
)

print(
    "VERSION A — OPENROUTER LLM + MCP ORCHESTRATION"
)

print(
    "=" * 90
)


print(
    f"LLM Provider        : "
    f"{LLM_PROVIDER}"
)

print(
    f"LLM Model           : "
    f"{LLM_MODEL}"
)

print(
    f"OpenRouter Tools    : "
    f"{len(OPENROUTER_MCP_TOOLS)}"
)

print(
    f"Exposed Tool        : "
    f"{OPENROUTER_MCP_TOOLS[0]['function']['name']}"
)

print(
    "Tool Schema         : "
    "OpenAI-compatible function"
)

print(
    f"Max MCP Searches    : "
    f"{MAX_MCP_SEARCHES}"
)

print(
    f"Maximum Top-K       : "
    f"{TOP_K_RESULTS}"
)

print(
    "Temperature         : "
    "API DEFAULT"
)


print(
    "\n" + "=" * 90
)


print(
    "✓ Current FastMCP input_schema API used."
)

print(
    "✓ FastMCP schema converted to OpenAI function format."
)

print(
    "✓ OpenRouter tool input normalization enabled."
)

print(
    "✓ Missing/empty query protection enabled."
)

print(
    "✓ Unexpected tool arguments are not forwarded."
)

print(
    "✓ Top-K coercion and 1–5 clamping enabled."
)

print(
    "✓ Exactly one MCP knowledge tool exposed."
)

print(
    "✓ Dynamic 3GPP / TCC / HYBRID routing remains internal."
)

print(
    "✓ OpenAI-compatible tool_call / tool message loop configured."
)

print(
    "✓ Every tool call receives a matching tool_call_id result."
)

print(
    "✓ Maximum executed MCP search budget fixed at 3."
)

print(
    "✓ Final answer generated without tools after search cap."
)

print(
    "✓ OpenRouter token accounting enabled."
)

print(
    "✓ Retrieval and MCP timing traces enabled."
)

print(
    "✓ Existing Version A result fields preserved."
)

print(
    "✓ Provider-native finish reasons normalized for A/B comparison."
)

print(
    "✓ Tool-input and finish-reason self-tests passed."
)

print(
    "✓ No end-to-end question executed in this cell."
)

print(
    "✓ Ready for Cell 11 — "
    "OpenRouter LLM Single End-to-End Pilot."
)

print(
    "=" * 90
)


VERSION A — OPENROUTER LLM + MCP ORCHESTRATION
LLM Provider        : OpenRouter
LLM Model           : deepseek/deepseek-v4-flash-0731
OpenRouter Tools    : 1
Exposed Tool        : search_telecom_knowledge
Tool Schema         : OpenAI-compatible function
Max MCP Searches    : 3
Maximum Top-K       : 5
Temperature         : API DEFAULT

✓ Current FastMCP input_schema API used.
✓ FastMCP schema converted to OpenAI function format.
✓ OpenRouter tool input normalization enabled.
✓ Missing/empty query protection enabled.
✓ Unexpected tool arguments are not forwarded.
✓ Top-K coercion and 1–5 clamping enabled.
✓ Exactly one MCP knowledge tool exposed.
✓ Dynamic 3GPP / TCC / HYBRID routing remains internal.
✓ OpenAI-compatible tool_call / tool message loop configured.
✓ Every tool call receives a matching tool_call_id result.
✓ Maximum executed MCP search budget fixed at 3.
✓ Final answer generated without tools after search cap.
✓ OpenRouter token accounting enabled.
✓ Retrieval and MCP timin

**Observation — DeepSeek V4 Flash 0731 + MCP Orchestration**

The orchestration layer connects DeepSeek V4 Flash 0731 to the single validated MCP knowledge service, validates model-generated tool inputs, enforces the frozen Top-K/evidence/search limits and preserves routing, evidence, token and latency traces.

***Key Decision:*** Keep provider/model orchestration separate from retrieval so model-specific tool-use behaviour remains observable without modifying Version A retrieval.


## **Cell 11 — Single End-to-End Pilot**

In [ ]:
# ============================================================
# CELL 11 — THREE-ROUTE END-TO-END PILOT VALIDATION
# ============================================================

# Purpose:
#
# Validate the final Version A MCP architecture before the
# Benchmark v2 evaluation using three deliberately different
# knowledge-routing cases:
#
#     1. 3GPP-only
#     2. TCC-only
#     3. Hybrid 3GPP + TCC
#
# This cell also verifies that Cell 10 now preserves the
# bounded evidence excerpts required for downstream relevance,
# groundedness, correctness and unsupported-claim evaluation.
#
# These are PILOT questions only and do not form part of the
# final frozen 8-question Benchmark v2 evaluation.


# ============================================================
# PILOT QUESTIONS
# ============================================================

VERSION_A_ROUTE_PILOTS = [

    {
        "id": "P1",
        "name": "3GPP-only",
        "expected_route": "3gpp",
        "expected_source_families": ["3GPP"],
        "question": (
            "Explain the role of the AMF in registration and mobility "
            "management procedures in a 5G Standalone network."
        )
    },

    {
        "id": "P2",
        "name": "TCC-only",
        "expected_route": "tcc",
        "expected_source_families": ["TCC"],
        "question": (
            "Explain the key mechanisms of QUIC as defined by the IETF, "
            "including connection establishment, stream multiplexing "
            "and connection migration."
        )
    },

    {
        "id": "P3",
        "name": "Hybrid 3GPP + TCC",
        "expected_route": "hybrid",
        "expected_source_families": ["3GPP", "TCC"],
        "question": (
            "Explain how HTTP and TLS support communication in the "
            "5G Service-Based Architecture. Distinguish the roles and "
            "requirements defined by 3GPP for network functions such as "
            "the AMF and SMF from the HTTP/TLS transport and security "
            "mechanisms defined by the IETF."
        )
    }
]


# ============================================================
# STATIC ROUTER PRE-CHECK
# ============================================================

print("=" * 100)
print("VERSION A — THREE-ROUTE PILOT VALIDATION")
print("=" * 100)

print("\nSTATIC ROUTER PRE-CHECK")
print("-" * 100)

static_router_checks = []

for pilot in VERSION_A_ROUTE_PILOTS:

    routing_plan = route_telecom_query(
        pilot["question"]
    )

    actual_route = str(
        routing_plan.get(
            "route",
            ""
        )
        or ""
    ).lower()

    expected_route = pilot[
        "expected_route"
    ]

    route_ok = (
        actual_route
        ==
        expected_route
    )

    static_router_checks.append(
        route_ok
    )

    print(
        f"{pilot['id']} | {pilot['name']}"
    )

    print(
        f"  Expected route : {expected_route.upper()}"
    )

    print(
        f"  Actual route   : {actual_route.upper()}"
    )

    print(
        f"  3GPP specs     : "
        f"{routing_plan.get('gpp_specs', [])}"
    )

    print(
        f"  TCC collections: "
        f"{routing_plan.get('tcc_collections', [])}"
    )

    print(
        f"  Router check   : "
        f"{'PASS' if route_ok else 'FAIL'}\n"
    )


if not all(
    static_router_checks
):

    raise RuntimeError(
        "Static router pre-check failed. "
        "Do not run the end-to-end pilot until routing is corrected."
    )


# ============================================================
# PILOT EXECUTION HELPERS
# ============================================================

def collect_pilot_evidence(
    result
):
    """
    Flatten evidence items from all executed MCP searches.
    """

    flattened = []

    search_traces = result.get(
        "evidence_trace",
        []
    ) or []

    for search_trace in search_traces:

        if not isinstance(
            search_trace,
            dict
        ):
            continue

        search_number = search_trace.get(
            "search_number"
        )

        route = search_trace.get(
            "route"
        )

        for item in (
            search_trace.get(
                "results",
                []
            )
            or []
        ):

            if not isinstance(
                item,
                dict
            ):
                continue

            flattened.append({
                "search_number": search_number,
                "route": route,
                **item
            })

    return flattened


def normalize_source_family(
    value
):
    """
    Normalize evidence source-family labels for validation.
    """

    text = str(
        value
        or ""
    ).strip().lower()

    if text == "3gpp":
        return "3GPP"

    if text == "tcc":
        return "TCC"

    return str(
        value
        or ""
    ).strip()


def hybrid_route_satisfied(
    routes
):
    """
    A hybrid workflow is valid if either:

        - one executed search used the hybrid route, or
        - separate 3GPP and TCC searches were executed.

    This avoids incorrectly failing a valid multi-search
    orchestration strategy.
    """

    route_set = {
        str(route).lower()
        for route in routes
        if route is not None
    }

    return (
        "hybrid" in route_set
        or
        (
            "3gpp" in route_set
            and
            "tcc" in route_set
        )
    )


# ============================================================
# EXECUTE ALL THREE PILOTS THROUGH THE REAL MCP CLIENT
# ============================================================

version_a_route_pilot_results = []

async with Client(
    mcp
) as pilot_mcp_client:

    for pilot in VERSION_A_ROUTE_PILOTS:

        print("\n" + "=" * 100)
        print(
            f"{pilot['id']} — {pilot['name']}"
        )
        print("=" * 100)
        print(
            f"Question: {pilot['question']}"
        )
        print(
            "\nExecuting LLM → MCP → Retrieval → LLM...\n"
        )

        result = await run_version_a_e2e(
            mcp_client=pilot_mcp_client,
            question=pilot["question"],
            verbose=True
        )

        version_a_route_pilot_results.append({
            "pilot": pilot,
            "result": result
        })


# ============================================================
# VALIDATE EACH PILOT
# ============================================================

print("\n" + "=" * 100)
print("VERSION A — THREE-ROUTE PILOT VALIDATION SUMMARY")
print("=" * 100)

all_pilot_checks = []

for pilot_record in version_a_route_pilot_results:

    pilot = pilot_record[
        "pilot"
    ]

    result = pilot_record[
        "result"
    ]

    routes = [
        str(route).lower()
        for route in (
            result.get(
                "tool_routes",
                []
            )
            or []
        )
    ]

    search_traces = result.get(
        "evidence_trace",
        []
    ) or []

    evidence_items = collect_pilot_evidence(
        result
    )

    source_families = {
        normalize_source_family(
            item.get(
                "source_family"
            )
        )
        for item in evidence_items
        if item.get(
            "source_family"
        )
    }

    evidence_text_items = [
        item
        for item in evidence_items
        if str(
            item.get(
                "evidence",
                ""
            )
            or ""
        ).strip()
    ]

    retrieval_trace_items = [
        trace
        for trace in search_traces
        if isinstance(
            trace,
            dict
        )
        and isinstance(
            trace.get(
                "retrieval_trace"
            ),
            dict
        )
    ]

    expected_route = pilot[
        "expected_route"
    ]

    if expected_route == "hybrid":

        route_check = hybrid_route_satisfied(
            routes
        )

    else:

        route_check = (
            expected_route
            in
            set(
                routes
            )
        )

    expected_sources = set(
        pilot[
            "expected_source_families"
        ]
    )

    source_check = (
        expected_sources
        .issubset(
            source_families
        )
    )

    checks = {
        "completed": bool(
            result.get(
                "completed",
                False
            )
        ),

        "mcp_search_executed": (
            int(
                result.get(
                    "mcp_searches",
                    0
                )
                or 0
            )
            >= 1
        ),

        "search_budget_respected": (
            int(
                result.get(
                    "mcp_searches",
                    0
                )
                or 0
            )
            <= MAX_MCP_SEARCHES
        ),

        "expected_route_observed": route_check,

        "expected_source_family_present": source_check,

        "search_trace_count_matches_searches": (
            len(
                search_traces
            )
            ==
            int(
                result.get(
                    "mcp_searches",
                    0
                )
                or 0
            )
        ),

        "retrieval_trace_preserved": (
            len(
                retrieval_trace_items
            )
            ==
            len(
                search_traces
            )
            and
            len(
                search_traces
            )
            > 0
        ),

        "evidence_results_preserved": (
            len(
                evidence_items
            )
            > 0
        ),

        "evidence_text_preserved": (
            len(
                evidence_text_items
            )
            ==
            len(
                evidence_items
            )
            and
            len(
                evidence_items
            )
            > 0
        ),

        "retrieval_time_recorded": (
            float(
                result.get(
                    "total_retrieval_time_s",
                    0
                )
                or 0
            )
            > 0
        ),

        "token_usage_recorded": (
            int(
                result.get(
                    "total_tokens",
                    0
                )
                or 0
            )
            > 0
        ),

        "answer_returned": (
            int(
                result.get(
                    "word_count",
                    0
                )
                or 0
            )
            > 0
        ),

        "no_execution_errors": (
            len(
                result.get(
                    "errors",
                    []
                )
                or []
            )
            == 0
        )
    }

    pilot_passed = all(
        checks.values()
    )

    all_pilot_checks.append(
        pilot_passed
    )

    print("\n" + "-" * 100)
    print(
        f"{pilot['id']} — {pilot['name']}"
    )
    print("-" * 100)

    print(
        f"Expected Route       : "
        f"{expected_route.upper()}"
    )

    print(
        f"Observed Routes      : "
        f"{routes}"
    )

    print(
        f"Expected Sources     : "
        f"{sorted(expected_sources)}"
    )

    print(
        f"Observed Sources     : "
        f"{sorted(source_families)}"
    )

    print(
        f"MCP Searches         : "
        f"{result.get('mcp_searches', 0)}"
    )

    print(
        f"Evidence Items       : "
        f"{len(evidence_items)}"
    )

    print(
        f"Evidence With Text   : "
        f"{len(evidence_text_items)}"
    )

    print(
        f"Final Word Count     : "
        f"{result.get('word_count', 0)}"
    )

    print(
        f"E2E Time             : "
        f"{float(result.get('e2e_time_s', 0) or 0):.3f} sec"
    )

    print("\nValidation Checks:")

    for check_name, passed in checks.items():

        print(
            f"  {'✓' if passed else '✗'} "
            f"{check_name}"
        )

    # --------------------------------------------------------
    # SHOW ONE EVIDENCE SAMPLE
    # --------------------------------------------------------

    if evidence_text_items:

        sample = evidence_text_items[0]

        sample_text = str(
            sample.get(
                "evidence",
                ""
            )
            or ""
        ).strip()

        print("\nEvidence Capture Sample:")

        print(
            f"  Source Family : "
            f"{sample.get('source_family')}"
        )

        print(
            f"  Collection    : "
            f"{sample.get('collection')}"
        )

        print(
            f"  Identifier    : "
            f"{sample.get('identifier')}"
        )

        print(
            f"  Title         : "
            f"{sample.get('title')}"
        )

        print(
            f"  Evidence chars: "
            f"{len(sample_text):,}"
        )

        print(
            "  Evidence      : "
            +
            sample_text[:500]
            +
            (
                "..."
                if len(sample_text) > 500
                else ""
            )
        )

    print(
        f"\nPilot Status          : "
        f"{'PASS' if pilot_passed else 'REVIEW'}"
    )


# ============================================================
# FINAL THREE-ROUTE STATUS
# ============================================================

three_route_pilot_passed = all(
    all_pilot_checks
)

print("\n" + "=" * 100)

if three_route_pilot_passed:

    print(
        "✓ VERSION A THREE-ROUTE PILOT PASSED"
    )

    print(
        "✓ 3GPP-only routing validated."
    )

    print(
        "✓ TCC-only routing validated."
    )

    print(
        "✓ Hybrid 3GPP + TCC retrieval validated."
    )

    print(
        "✓ Full bounded evidence text is preserved in the evaluation trace."
    )

    print(
        "✓ Retrieval provenance and search-level timing are preserved."
    )

    print(
        "✓ Ready to build Cell 12 — Module 3 Evaluation Benchmark v2."
    )

else:

    print(
        "⚠ VERSION A THREE-ROUTE PILOT REQUIRES REVIEW"
    )

    print(
        "Do not start the 8-question Benchmark v2 evaluation until "
        "the failed checks are understood."
    )

print("=" * 100)


VERSION A — THREE-ROUTE PILOT VALIDATION

STATIC ROUTER PRE-CHECK
----------------------------------------------------------------------------------------------------
P1 | 3GPP-only
  Expected route : 3GPP
  Actual route   : 3GPP
  3GPP specs     : ['23.501', '23.502']
  TCC collections: ['3GPP-TSG']
  Router check   : PASS

P2 | TCC-only
  Expected route : TCC
  Actual route   : TCC
  3GPP specs     : []
  TCC collections: ['IETF-RFCs', 'IETF-Drafts']
  Router check   : PASS

P3 | Hybrid 3GPP + TCC
  Expected route : HYBRID
  Actual route   : HYBRID
  3GPP specs     : ['23.501', '33.501']
  TCC collections: ['IETF-RFCs', 'IETF-Drafts']
  Router check   : PASS


P1 — 3GPP-only
Question: Explain the role of the AMF in registration and mobility management procedures in a 5G Standalone network.

Executing LLM → MCP → Retrieval → LLM...

VERSION A — OPENROUTER LLM + MCP END-TO-END EXECUTION
Question        : Explain the role of the AMF in registration and mobility management procedures in 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Route     : TCC
Sources   : ['TCC']
Results   : 5
Retrieval : 7.995 sec
MCP Round : 7.999 sec

MCP SEARCH 2/3
Query : AMF SMF service-based interface 3GPP TS 29.500 HTTP requirements NF service producer consumer
Top-K : 5
Route     : 3GPP
Sources   : ['3GPP']
Results   : 4
Retrieval : 0.619 sec
MCP Round : 0.623 sec

MCP SEARCH 3/3
Query : TS 29.500 clause 6.1 HTTP/2 TLS HTTPS SBI security requirements API framework
Top-K : 5
Route     : 3GPP
Sources   : ['3GPP']
Results   : 4
Retrieval : 0.309 sec
MCP Round : 0.313 sec

Maximum MCP search budget reached.
Requesting final answer using retrieved evidence only...
Final LLM Turn: 10.952 sec

VERSION A — OPENROUTER LLM E2E EXECUTION SUMMARY
Completed          : True
Completion Mode    : forced_after_search_cap
Stop Reason        : max_tokens
Raw Finish Reason  : length
LLM Calls          : 3
MCP Tool Requests  : 3
MCP Searches       : 3
Tool Routes        : ['tcc', '3gpp', '3gpp']
Retrieval Time     : 8.923 sec
MCP Round Trip     : 8.935 s

**Observation — DeepSeek V4 Flash End-to-End Pilot**

The captured pilot executed three MCP searches and retrieved 3GPP evidence without execution errors, but the final generation reached the configured **1,800-token ceiling** and returned no usable final answer. The pilot therefore failed the `completed` and `answer_returned` validation checks and was correctly marked **REQUIRES REVIEW** in the executed output.

The later frozen eight-question evaluation completed successfully, so this pilot anomaly is retained as an experimental limitation rather than hidden.

***Key Observation:*** Retrieval/MCP execution was functional; the pilot failure occurred at the model-generation/output-budget stage.



# **SECTION 5 — Version A Evaluation**

## **Cell 12 — Run 8-Question Version A Evaluation**

In [ ]:
# ============================================================
# CELL 12 — MODULE 3 EVALUATION BENCHMARK V2
# VERSION A — DEEPSEEK V4 FLASH
# ============================================================

from datetime import datetime, timezone
from pathlib import Path

import hashlib
import json
import time
import pandas as pd


# ============================================================
# BENCHMARK V2
# 5 × 3GPP | 2 × TCC | 1 × HYBRID
# ============================================================

BENCHMARK_VERSION = "module3_eval_v2"

VERSION_A_EVAL_QUESTIONS = [

    {
        "id": "Q1",
        "domain": "5G Core",
        "knowledge_scope": "3GPP",
        "expected_route": "3gpp",
        "expected_source_families": ["3GPP"],
        "expected_3gpp_specs": ["23.501", "23.502"],
        "expected_tcc_collections": [],
        "expected_elements": [
            "AMF",
            "registration",
            "mobility management",
            "NAS",
            "N1",
            "N2"
        ],
        "question": (
            "Explain the role of the AMF in registration and mobility "
            "management procedures in a 5G Standalone network."
        )
    },

    {
        "id": "Q2",
        "domain": "Mobility",
        "knowledge_scope": "3GPP",
        "expected_route": "3gpp",
        "expected_source_families": ["3GPP"],
        "expected_3gpp_specs": ["23.502", "38.300", "38.423"],
        "expected_tcc_collections": [],
        "expected_elements": [
            "source gNB",
            "target gNB",
            "AMF",
            "handover",
            "Xn",
            "N2"
        ],
        "question": (
            "Explain how inter-gNB handover operates in a 5G network, "
            "including the roles of the source gNB, target gNB and AMF."
        )
    },

    {
        "id": "Q3",
        "domain": "RAN",
        "knowledge_scope": "3GPP",
        "expected_route": "3gpp",
        "expected_source_families": ["3GPP"],
        "expected_3gpp_specs": ["38.331"],
        "expected_tcc_collections": [],
        "expected_elements": [
            "radio link failure",
            "RLF",
            "RRC",
            "T310",
            "N310",
            "RRC re-establishment"
        ],
        "question": (
            "Explain how radio link failure is detected and handled "
            "in a 5G NR network."
        )
    },

    {
        "id": "Q4",
        "domain": "QoS",
        "knowledge_scope": "3GPP",
        "expected_route": "3gpp",
        "expected_source_families": ["3GPP"],
        "expected_3gpp_specs": ["23.501", "23.503"],
        "expected_tcc_collections": [],
        "expected_elements": [
            "5QI",
            "QoS Flow",
            "QFI",
            "SMF",
            "UPF",
            "QoS rules"
        ],
        "question": (
            "Explain how 5QI and QoS flows are handled in a 5G network, "
            "including the roles of the SMF and UPF."
        )
    },

    {
        "id": "Q5",
        "domain": "Security",
        "knowledge_scope": "3GPP",
        "expected_route": "3gpp",
        "expected_source_families": ["3GPP"],
        "expected_3gpp_specs": ["33.501"],
        "expected_tcc_collections": [],
        "expected_elements": [
            "AMF",
            "AUSF",
            "UDM",
            "5G-AKA",
            "SEAF",
            "authentication"
        ],
        "question": (
            "Explain the 5G authentication procedure and the roles "
            "of the AMF, AUSF and UDM."
        )
    },

    {
        "id": "Q6",
        "domain": "Internet Transport",
        "knowledge_scope": "TCC",
        "expected_route": "tcc",
        "expected_source_families": ["TCC"],
        "expected_3gpp_specs": [],
        "expected_tcc_collections": ["IETF-RFCs", "IETF-Drafts"],
        "expected_elements": [
            "QUIC",
            "TLS 1.3",
            "stream multiplexing",
            "Connection ID",
            "connection migration"
        ],
        "question": (
            "Explain the key mechanisms of QUIC as defined by the IETF, "
            "including connection establishment, stream multiplexing "
            "and connection migration."
        )
    },

    {
        "id": "Q7",
        "domain": "Telecom AI Research",
        "knowledge_scope": "TCC",
        "expected_route": "tcc",
        "expected_source_families": ["TCC"],
        "expected_3gpp_specs": [],
        "expected_tcc_collections": ["IEEE-Access", "OpenAlex"],
        "expected_elements": [
            "traffic prediction",
            "machine learning",
            "forecasting",
            "network management",
            "benefits",
            "challenges"
        ],
        "question": (
            "According to IEEE and research literature, what are the main "
            "benefits and technical challenges of using AI/ML-based traffic "
            "prediction for mobile network management?"
        )
    },

    {
        "id": "Q8",
        "domain": "5G SBA + Internet Protocols",
        "knowledge_scope": "Hybrid",
        "expected_route": "hybrid",
        "expected_source_families": ["3GPP", "TCC"],
        "expected_3gpp_specs": ["23.501", "33.501"],
        "expected_tcc_collections": ["IETF-RFCs", "IETF-Drafts"],
        "expected_elements": [
            "Service-Based Architecture",
            "AMF",
            "SMF",
            "HTTP",
            "TLS",
            "3GPP",
            "IETF"
        ],
        "question": (
            "Explain how HTTP and TLS support communication in the 5G "
            "Service-Based Architecture. Distinguish the roles and "
            "requirements defined by 3GPP for network functions such as "
            "the AMF and SMF from the HTTP/TLS transport and security "
            "mechanisms defined by the IETF."
        )
    }
]


# ============================================================
# BENCHMARK VALIDATION
# ============================================================

assert len(VERSION_A_EVAL_QUESTIONS) == 8

assert [item["id"] for item in VERSION_A_EVAL_QUESTIONS] == [
    "Q1", "Q2", "Q3", "Q4",
    "Q5", "Q6", "Q7", "Q8"
]

assert [item["expected_route"] for item in VERSION_A_EVAL_QUESTIONS].count("3gpp") == 5
assert [item["expected_route"] for item in VERSION_A_EVAL_QUESTIONS].count("tcc") == 2
assert [item["expected_route"] for item in VERSION_A_EVAL_QUESTIONS].count("hybrid") == 1

assert MAX_MCP_SEARCHES == 3
assert TOP_K_RESULTS == 5
assert MAX_RETRIEVED_SOURCES == 5
assert MCP_EXCERPT_CHARS == 2500


# ============================================================
# STATIC ROUTER PRE-CHECK
# ============================================================

for item in VERSION_A_EVAL_QUESTIONS:

    routing = route_telecom_query(
        item["question"]
    )

    actual_route = str(
        routing.get(
            "route",
            ""
        )
        or ""
    ).lower()

    expected_route = item[
        "expected_route"
    ]

    if actual_route != expected_route:

        raise RuntimeError(
            f"{item['id']} router mismatch: "
            f"expected={expected_route}, actual={actual_route}"
        )


# ============================================================
# OUTPUT LOCATION
# ============================================================

EVAL_DIR = Path(
    "/content/version_a_evaluation"
)

EVAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_FILE_LABEL = "deepseek_v4_flash_0731"

VERSION_A_JSON_PATH = (
    EVAL_DIR
    / f"version_a_{MODEL_FILE_LABEL}_evaluation_v2.json"
)

VERSION_A_CSV_PATH = (
    EVAL_DIR
    / f"version_a_{MODEL_FILE_LABEL}_summary_v2.csv"
)


# ============================================================
# FINGERPRINTS
# ============================================================

SYSTEM_PROMPT_HASH = hashlib.sha256(
    SYSTEM_PROMPT.encode("utf-8")
).hexdigest()

SYSTEM_PROMPT_WORDS = len(
    SYSTEM_PROMPT.split()
)

SYSTEM_PROMPT_CHARS = len(
    SYSTEM_PROMPT
)

benchmark_payload = json.dumps(
    VERSION_A_EVAL_QUESTIONS,
    sort_keys=True,
    ensure_ascii=False,
    separators=(",", ":")
)

BENCHMARK_SHA256 = hashlib.sha256(
    benchmark_payload.encode("utf-8")
).hexdigest()


# ============================================================
# EXPERIMENT METADATA
# ============================================================

evaluation_timestamp = datetime.now(
    timezone.utc
).isoformat()

experiment_metadata = {

    "experiment":
        "Telecom AI MCP",

    "module":
        "Module 3",

    "architecture_version":
        "Version A",

    "evaluation_type":
        "8-question end-to-end MCP evaluation",

    "evaluation_version":
        BENCHMARK_VERSION,

    "evaluation_timestamp_utc":
        evaluation_timestamp,

    "question_count":
        len(
            VERSION_A_EVAL_QUESTIONS
        ),

    "benchmark": {

        "version":
            BENCHMARK_VERSION,

        "sha256":
            BENCHMARK_SHA256,

        "composition": {
            "3gpp": 5,
            "tcc": 2,
            "hybrid": 1
        },

        # Preserve the exact benchmark definition in the artifact.
        "questions":
            VERSION_A_EVAL_QUESTIONS
    },

    "llm": {

        "provider":
            LLM_PROVIDER,

        "model":
            LLM_MODEL,

        "max_output_tokens":
            LLM_MAX_TOKENS,

        "temperature":
            None,

        "target_answer_words":
            "300-500"
    },

    "prompt": {

        "system_prompt":
            SYSTEM_PROMPT,

        "system_prompt_sha256":
            SYSTEM_PROMPT_HASH,

        "system_prompt_words":
            SYSTEM_PROMPT_WORDS,

        "system_prompt_chars":
            SYSTEM_PROMPT_CHARS
    },

    "knowledge_base": {

        "storage":
            "Remote raw telecom corpora",

        "persistent_index":
            False,

        "sources": [
            "GSMA/Telco-Common-Corpus",
            "GSMA/3GPP"
        ],

        "tcc_access":
            "Remote Parquet via DuckDB httpfs",

        "3gpp_access":
            "Remote raw.md direct HTTP"
    },

    "retrieval": {

        "engine":
            RETRIEVAL_ARCHITECTURE,

        "strategy": (
            "dynamic 3GPP / TCC / hybrid routing + "
            "remote TCC lexical retrieval + "
            "section-aware direct 3GPP retrieval"
        ),

        "tcc_scoring":
            RETRIEVAL_SCORING,

        "max_remote_shards":
            MAX_REMOTE_SHARDS,

        "parallel_shard_concurrency":
            PARALLEL_SHARD_CONCURRENCY,

        "top_k":
            TOP_K_RESULTS
    },

    "mcp": {

        "server":
            "Telecom Knowledge Service — Version A",

        "tool":
            "search_telecom_knowledge",

        "max_searches_per_question":
            MAX_MCP_SEARCHES,

        "max_sources_per_search":
            MAX_RETRIEVED_SOURCES,

        "excerpt_chars":
            MCP_EXCERPT_CHARS
    },

    "evidence_capture": {

        "enabled":
            True,

        "location":
            "results[].tool_trace[].evidence",

        "full_source_documents_saved":
            False,

        "bounded_evidence_excerpts_saved":
            True,

        "purpose": [
            "retrieval relevance",
            "source authority",
            "claim-level groundedness",
            "unsupported-claim analysis",
            "technical-quality evaluation"
        ]
    }
}


# ============================================================
# HELPERS
# ============================================================

def _safe_list(value):

    if value is None:
        return []

    if isinstance(
        value,
        list
    ):
        return value

    if isinstance(
        value,
        (tuple, set)
    ):
        return list(
            value
        )

    return [
        value
    ]


def build_version_a_tool_trace(
    result
):
    """
    Build a complete per-search trace.

    Unlike Benchmark v1, the bounded evidence excerpts are
    intentionally retained so Notebook 17 can reconstruct
    retrieval relevance and claim-level groundedness.
    """

    raw_trace = _safe_list(
        result.get(
            "evidence_trace",
            []
        )
    )

    normalized_trace = []

    for index, raw_item in enumerate(
        raw_trace,
        start=1
    ):

        item = (
            raw_item
            if isinstance(
                raw_item,
                dict
            )
            else {}
        )

        evidence = _safe_list(
            item.get(
                "results",
                []
            )
        )

        normalized_trace.append({

            "search_number":
                int(
                    item.get(
                        "search_number",
                        index
                    )
                    or index
                ),

            "query":
                item.get(
                    "query"
                ),

            "top_k":
                item.get(
                    "top_k"
                ),

            "route":
                str(
                    item.get(
                        "route",
                        ""
                    )
                    or ""
                ).lower(),

            "sources_searched":
                _safe_list(
                    item.get(
                        "sources_searched",
                        []
                    )
                ),

            "source_distribution":
                item.get(
                    "source_distribution",
                    {}
                )
                or {},

            "retrieval_trace":
                item.get(
                    "retrieval_trace",
                    {}
                )
                or {},

            "result_count":
                int(
                    item.get(
                        "result_count",
                        len(
                            evidence
                        )
                    )
                    or 0
                ),

            "retrieval_time_s":
                float(
                    item.get(
                        "retrieval_time_s",
                        0.0
                    )
                    or 0.0
                ),

            "tool_elapsed_s":
                float(
                    item.get(
                        "tool_elapsed_s",
                        0.0
                    )
                    or 0.0
                ),

            "mcp_roundtrip_s":
                float(
                    item.get(
                        "mcp_roundtrip_s",
                        0.0
                    )
                    or 0.0
                ),

            "mcp_overhead_s":
                float(
                    item.get(
                        "mcp_overhead_s",
                        0.0
                    )
                    or 0.0
                ),

            # CRITICAL: preserve complete bounded evidence.
            "evidence":
                evidence
        })

    return normalized_trace


def extract_observed_provenance(
    tool_trace
):
    """
    Derive route/source/collection/spec observations from the
    complete search trace without discarding the raw trace.
    """

    routes = []
    source_families = set()
    tcc_collections = set()
    gpp_specs = set()

    evidence_items = 0
    evidence_items_with_text = 0

    for search in tool_trace:

        route = str(
            search.get(
                "route",
                ""
            )
            or ""
        ).lower()

        if route:
            routes.append(
                route
            )

        retrieval_trace = (
            search.get(
                "retrieval_trace",
                {}
            )
            or {}
        )

        tcc_trace = (
            retrieval_trace.get(
                "tcc"
            )
            or {}
        )

        for collection in _safe_list(
            tcc_trace.get(
                "collections",
                []
            )
        ):

            if collection:
                tcc_collections.add(
                    str(
                        collection
                    )
                )

        gpp_trace = (
            retrieval_trace.get(
                "3gpp"
            )
            or {}
        )

        for selected in _safe_list(
            gpp_trace.get(
                "selected_specs",
                []
            )
        ):

            if isinstance(
                selected,
                dict
            ):

                spec = selected.get(
                    "spec_number"
                )

                if spec:
                    gpp_specs.add(
                        str(
                            spec
                        )
                    )

        for evidence in _safe_list(
            search.get(
                "evidence",
                []
            )
        ):

            if not isinstance(
                evidence,
                dict
            ):
                continue

            evidence_items += 1

            source_family = str(
                evidence.get(
                    "source_family",
                    ""
                )
                or ""
            )

            if source_family:
                source_families.add(
                    source_family
                )

            evidence_text = str(
                evidence.get(
                    "evidence",
                    ""
                )
                or ""
            ).strip()

            if evidence_text:
                evidence_items_with_text += 1

    return {
        "routes":
            routes,

        "source_families":
            sorted(
                source_families
            ),

        "tcc_collections":
            sorted(
                tcc_collections
            ),

        "gpp_specs":
            sorted(
                gpp_specs
            ),

        "evidence_items":
            evidence_items,

        "evidence_items_with_text":
            evidence_items_with_text
    }


def route_expectation_met(
    expected_route,
    observed_routes,
    observed_source_families
):
    """
    Hybrid is accepted either as one explicit HYBRID search or
    as separate 3GPP + TCC searches that jointly cover both
    source families.
    """

    expected_route = str(
        expected_route
    ).lower()

    observed_routes = [
        str(
            route
        ).lower()
        for route
        in observed_routes
    ]

    if expected_route == "hybrid":

        return (
            "hybrid"
            in observed_routes
        ) or (
            {"3GPP", "TCC"}
            .issubset(
                set(
                    observed_source_families
                )
            )
        )

    return (
        expected_route
        in observed_routes
    )


def save_version_a_evaluation_checkpoint(
    results,
    summary_rows,
    aggregate_metrics=None
):

    artifact = {

        "metadata":
            experiment_metadata,

        "aggregate_metrics":
            aggregate_metrics or {},

        "results":
            results
    }

    with open(
        VERSION_A_JSON_PATH,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            artifact,
            f,
            indent=2,
            ensure_ascii=False,
            default=str
        )

    pd.DataFrame(
        summary_rows
    ).to_csv(
        VERSION_A_CSV_PATH,
        index=False
    )


# ============================================================
# RESULT CONTAINERS
# ============================================================

evaluation_results = []
summary_rows = []


# ============================================================
# EVALUATION HEADER
# ============================================================

print("=" * 100)
print("VERSION A — MODULE 3 EVALUATION BENCHMARK V2")
print("=" * 100)

print(f"Benchmark             : {BENCHMARK_VERSION}")
print(f"Composition           : 5 × 3GPP | 2 × TCC | 1 × HYBRID")
print(f"Questions             : {len(VERSION_A_EVAL_QUESTIONS)}")
print(f"Model                 : {LLM_MODEL}")
print(f"Architecture          : {RETRIEVAL_ARCHITECTURE}")
print(f"Top-K / Search        : {TOP_K_RESULTS}")
print(f"Max MCP Searches      : {MAX_MCP_SEARCHES}")
print(f"Evidence / Source     : {MCP_EXCERPT_CHARS:,} chars")
print(f"Benchmark SHA-256     : {BENCHMARK_SHA256}")
print(f"JSON Artifact         : {VERSION_A_JSON_PATH}")
print(f"CSV Summary           : {VERSION_A_CSV_PATH}")

print("=" * 100)


# ============================================================
# RUN EVALUATION
# ============================================================

evaluation_start = time.perf_counter()


async with Client(
    mcp
) as eval_mcp_client:

    for item in VERSION_A_EVAL_QUESTIONS:

        question_id = item[
            "id"
        ]

        domain = item[
            "domain"
        ]

        knowledge_scope = item[
            "knowledge_scope"
        ]

        expected_route = item[
            "expected_route"
        ]

        expected_source_families = item[
            "expected_source_families"
        ]

        expected_tcc_collections = item[
            "expected_tcc_collections"
        ]

        expected_3gpp_specs = item[
            "expected_3gpp_specs"
        ]

        expected_elements = item[
            "expected_elements"
        ]

        question = item[
            "question"
        ]

        print(
            "\n" + "=" * 100
        )

        print(
            f"{question_id} | "
            f"{domain} | "
            f"Expected Route: {expected_route.upper()}"
        )

        print("=" * 100)
        print(f"QUESTION:\n{question}")

        try:

            result = await run_version_a_e2e(

                mcp_client=
                    eval_mcp_client,

                question=
                    question,

                verbose=
                    False
            )

            completed = bool(
                result.get(
                    "completed",
                    False
                )
            )

            stop_reason = result.get(
                "stop_reason"
            )

            completion_mode = result.get(
                "completion_mode"
            )

            execution_errors = _safe_list(
                result.get(
                    "errors",
                    []
                )
            )

            if (
                completed
                and
                stop_reason == "end_turn"
                and
                not execution_errors
            ):

                status = "COMPLETED"

            else:

                status = "CHECK"


            # ----------------------------------------------------
            # COMPLETE SEARCH TRACE
            # ----------------------------------------------------

            tool_trace = build_version_a_tool_trace(
                result
            )

            provenance = extract_observed_provenance(
                tool_trace
            )

            routes_used = provenance[
                "routes"
            ]

            observed_source_families = provenance[
                "source_families"
            ]

            observed_tcc_collections = provenance[
                "tcc_collections"
            ]

            observed_3gpp_specs = provenance[
                "gpp_specs"
            ]

            evidence_items = provenance[
                "evidence_items"
            ]

            evidence_items_with_text = provenance[
                "evidence_items_with_text"
            ]


            # ----------------------------------------------------
            # ROUTING / SOURCE VALIDATION
            # ----------------------------------------------------

            route_match = route_expectation_met(
                expected_route=
                    expected_route,

                observed_routes=
                    routes_used,

                observed_source_families=
                    observed_source_families
            )

            source_family_match = set(
                expected_source_families
            ).issubset(
                set(
                    observed_source_families
                )
            )

            tcc_collection_match = (

                set(
                    expected_tcc_collections
                ).issubset(
                    set(
                        observed_tcc_collections
                    )
                )

                if expected_tcc_collections

                else True
            )

            evidence_text_coverage_pct = (

                evidence_items_with_text
                /
                evidence_items
                *
                100.0

                if evidence_items > 0

                else 0.0
            )


            # ----------------------------------------------------
            # ORCHESTRATION METRICS
            # ----------------------------------------------------

            mcp_searches = int(
                result.get(
                    "mcp_searches",
                    0
                )
                or 0
            )

            mcp_tool_requests = int(
                result.get(
                    "mcp_tool_requests",
                    mcp_searches
                )
                or 0
            )

            llm_turns = int(
                result.get(
                    "llm_calls",
                    result.get(
                        "claude_calls",
                        0
                    )
                )
                or 0
            )

            if mcp_searches == 1:
                search_classification = "IDEAL"
            elif mcp_searches == 2:
                search_classification = "ACCEPTABLE"
            elif mcp_searches == 3:
                search_classification = "MAXIMUM"
            else:
                search_classification = "UNEXPECTED"

            search_queries = _safe_list(
                result.get(
                    "tool_queries",
                    []
                )
            )

            sources_used = _safe_list(
                result.get(
                    "tool_sources",
                    []
                )
            )

            source_distributions = _safe_list(
                result.get(
                    "tool_source_distributions",
                    []
                )
            )


            # ----------------------------------------------------
            # SOURCE PAYLOAD
            # ----------------------------------------------------

            total_sources_returned = sum(
                int(
                    search.get(
                        "result_count",
                        0
                    )
                    or 0
                )
                for search
                in tool_trace
            )

            mean_sources_per_search = (

                total_sources_returned
                /
                mcp_searches

                if mcp_searches > 0

                else 0.0
            )


            # ----------------------------------------------------
            # LATENCY
            # ----------------------------------------------------

            retrieval_latency = float(
                result.get(
                    "total_retrieval_time_s",
                    0.0
                )
                or 0.0
            )

            total_mcp_roundtrip = float(
                result.get(
                    "total_mcp_roundtrip_s",
                    0.0
                )
                or 0.0
            )

            total_mcp_overhead = float(
                result.get(
                    "total_mcp_overhead_s",
                    0.0
                )
                or 0.0
            )

            total_latency = float(
                result.get(
                    "e2e_time_s",
                    0.0
                )
                or 0.0
            )

            mean_retrieval_latency_per_search = (

                retrieval_latency
                /
                mcp_searches

                if mcp_searches > 0

                else 0.0
            )

            non_retrieval_latency = max(
                0.0,
                total_latency
                -
                retrieval_latency
            )

            retrieval_latency_share_pct = (

                retrieval_latency
                /
                total_latency
                *
                100.0

                if total_latency > 0

                else 0.0
            )


            # ----------------------------------------------------
            # TOKENS / ANSWER
            # ----------------------------------------------------

            input_tokens = int(
                result.get(
                    "input_tokens",
                    0
                )
                or 0
            )

            output_tokens = int(
                result.get(
                    "output_tokens",
                    0
                )
                or 0
            )

            total_tokens = int(
                result.get(
                    "total_tokens",
                    input_tokens
                    +
                    output_tokens
                )
                or 0
            )

            final_answer_tokens = int(
                result.get(
                    "final_answer_tokens",
                    0
                )
                or 0
            )

            answer_words = int(
                result.get(
                    "word_count",
                    0
                )
                or 0
            )

            answer = str(
                result.get(
                    "final_answer",
                    ""
                )
                or ""
            )


            # ----------------------------------------------------
            # SAVE COMPLETE QUESTION RESULT
            # ----------------------------------------------------

            saved_result = {

                # Benchmark definition for this question.
                "id":
                    question_id,

                "domain":
                    domain,

                "knowledge_scope":
                    knowledge_scope,

                "question":
                    question,

                "expected_route":
                    expected_route,

                "expected_source_families":
                    expected_source_families,

                "expected_tcc_collections":
                    expected_tcc_collections,

                "expected_3gpp_specs":
                    expected_3gpp_specs,

                "expected_elements":
                    expected_elements,

                # Completion.
                "status":
                    status,

                "completion_mode":
                    completion_mode,

                "stop_reason":
                    stop_reason,

                # Response.
                "answer":
                    answer,

                "answer_words":
                    answer_words,

                "final_answer_tokens":
                    final_answer_tokens,

                # Routing / provenance observations.
                "routes_used":
                    routes_used,

                "route_match":
                    bool(
                        route_match
                    ),

                "observed_source_families":
                    observed_source_families,

                "source_family_match":
                    bool(
                        source_family_match
                    ),

                "observed_tcc_collections":
                    observed_tcc_collections,

                "tcc_collection_match":
                    bool(
                        tcc_collection_match
                    ),

                "observed_3gpp_specs":
                    observed_3gpp_specs,

                # Orchestration.
                "mcp_searches":
                    mcp_searches,

                "mcp_tool_requests":
                    mcp_tool_requests,

                "search_classification":
                    search_classification,

                # Compatibility alias retained for the shared
                # Version A/B analysis notebook.
                "claude_turns":
                    llm_turns,

                "llm_turns":
                    llm_turns,

                "search_queries":
                    search_queries,

                # Retain compatibility with Benchmark v1 /
                # Version B analysis fields.
                "profiles_used":
                    [],

                "candidate_terms_per_search":
                    [],

                "sources_used":
                    sources_used,

                "source_distributions":
                    source_distributions,

                # CRITICAL: complete bounded evidence lives here.
                "tool_trace":
                    tool_trace,

                "turn_usage":
                    result.get(
                        "turn_usage",
                        []
                    ),

                # Evidence capture diagnostics.
                "evidence_items":
                    evidence_items,

                "evidence_items_with_text":
                    evidence_items_with_text,

                "evidence_text_coverage_pct":
                    round(
                        evidence_text_coverage_pct,
                        1
                    ),

                # Retrieval / MCP timing.
                "retrieval_latency_sec":
                    round(
                        retrieval_latency,
                        3
                    ),

                "mean_retrieval_latency_per_search_sec":
                    round(
                        mean_retrieval_latency_per_search,
                        3
                    ),

                "total_sources_returned":
                    total_sources_returned,

                "mean_sources_per_search":
                    round(
                        mean_sources_per_search,
                        2
                    ),

                "mcp_roundtrip_sec":
                    round(
                        total_mcp_roundtrip,
                        3
                    ),

                "mcp_overhead_sec":
                    round(
                        total_mcp_overhead,
                        3
                    ),

                "total_latency_sec":
                    round(
                        total_latency,
                        3
                    ),

                "non_retrieval_latency_sec":
                    round(
                        non_retrieval_latency,
                        3
                    ),

                "retrieval_latency_share_pct":
                    round(
                        retrieval_latency_share_pct,
                        1
                    ),

                # Tokens.
                "input_tokens":
                    input_tokens,

                "output_tokens":
                    output_tokens,

                "total_tokens":
                    total_tokens,

                # Errors.
                "errors":
                    execution_errors
            }

            evaluation_results.append(
                saved_result
            )


            # ----------------------------------------------------
            # SUMMARY ROW
            # ----------------------------------------------------

            summary_rows.append({

                "question_id":
                    question_id,

                "domain":
                    domain,

                "knowledge_scope":
                    knowledge_scope,

                "expected_route":
                    expected_route,

                "routes_used":
                    ",".join(
                        routes_used
                    ),

                "route_match":
                    bool(
                        route_match
                    ),

                "observed_source_families":
                    ",".join(
                        observed_source_families
                    ),

                "source_family_match":
                    bool(
                        source_family_match
                    ),

                "status":
                    status,

                "stop_reason":
                    stop_reason,

                "mcp_searches":
                    mcp_searches,

                "search_classification":
                    search_classification,

                "claude_turns":
                    llm_turns,

                "llm_turns":
                    llm_turns,

                "total_sources_returned":
                    total_sources_returned,

                "evidence_items":
                    evidence_items,

                "evidence_items_with_text":
                    evidence_items_with_text,

                "evidence_text_coverage_pct":
                    round(
                        evidence_text_coverage_pct,
                        1
                    ),

                "mean_sources_per_search":
                    round(
                        mean_sources_per_search,
                        2
                    ),

                "retrieval_latency_sec":
                    round(
                        retrieval_latency,
                        3
                    ),

                "mean_retrieval_latency_per_search_sec":
                    round(
                        mean_retrieval_latency_per_search,
                        3
                    ),

                "total_latency_sec":
                    round(
                        total_latency,
                        3
                    ),

                "non_retrieval_latency_sec":
                    round(
                        non_retrieval_latency,
                        3
                    ),

                "retrieval_latency_share_pct":
                    round(
                        retrieval_latency_share_pct,
                        1
                    ),

                "input_tokens":
                    input_tokens,

                "output_tokens":
                    output_tokens,

                "total_tokens":
                    total_tokens,

                "final_answer_tokens":
                    final_answer_tokens,

                "answer_words":
                    answer_words
            })


            # ----------------------------------------------------
            # DISPLAY
            # ----------------------------------------------------

            print("\nFINAL ANSWER")
            print("-" * 100)
            print(answer)

            print("\nMETRICS")
            print("-" * 100)
            print(f"Status                     : {status}")
            print(f"Expected Route             : {expected_route.upper()}")
            print(f"Observed Routes            : {routes_used}")
            print(f"Route Match                : {route_match}")
            print(f"Observed Sources           : {observed_source_families}")
            print(f"Source Match               : {source_family_match}")
            print(f"MCP Searches               : {mcp_searches}")
            print(f"Evidence Items             : {evidence_items}")
            print(f"Evidence With Text         : {evidence_items_with_text}")
            print(f"Evidence Text Coverage     : {evidence_text_coverage_pct:.1f}%")
            print(f"Retrieval Latency          : {retrieval_latency:.3f} sec")
            print(f"Total E2E Latency          : {total_latency:.3f} sec")
            print(f"Total Tokens               : {total_tokens:,}")
            print(f"Answer Words               : {answer_words:,}")
            print(f"Execution Errors           : {len(execution_errors)}")


        except Exception as exc:

            error_text = (
                f"{type(exc).__name__}: "
                f"{exc}"
            )

            evaluation_results.append({

                "id":
                    question_id,

                "domain":
                    domain,

                "knowledge_scope":
                    knowledge_scope,

                "question":
                    question,

                "expected_route":
                    expected_route,

                "expected_source_families":
                    expected_source_families,

                "expected_tcc_collections":
                    expected_tcc_collections,

                "expected_3gpp_specs":
                    expected_3gpp_specs,

                "expected_elements":
                    expected_elements,

                "status":
                    "ERROR",

                "error":
                    error_text
            })

            summary_rows.append({

                "question_id":
                    question_id,

                "domain":
                    domain,

                "knowledge_scope":
                    knowledge_scope,

                "expected_route":
                    expected_route,

                "routes_used":
                    "",

                "route_match":
                    False,

                "observed_source_families":
                    "",

                "source_family_match":
                    False,

                "status":
                    "ERROR",

                "stop_reason":
                    None,

                "mcp_searches":
                    None,

                "search_classification":
                    None,

                "claude_turns":
                    None,

                "llm_turns":
                    None,

                "total_sources_returned":
                    None,

                "evidence_items":
                    None,

                "evidence_items_with_text":
                    None,

                "evidence_text_coverage_pct":
                    None,

                "mean_sources_per_search":
                    None,

                "retrieval_latency_sec":
                    None,

                "mean_retrieval_latency_per_search_sec":
                    None,

                "total_latency_sec":
                    None,

                "non_retrieval_latency_sec":
                    None,

                "retrieval_latency_share_pct":
                    None,

                "input_tokens":
                    None,

                "output_tokens":
                    None,

                "total_tokens":
                    None,

                "final_answer_tokens":
                    None,

                "answer_words":
                    None
            })

            print("\nERROR")
            print("-" * 100)
            print(error_text)


        # --------------------------------------------------------
        # CHECKPOINT AFTER EVERY QUESTION
        # --------------------------------------------------------

        save_version_a_evaluation_checkpoint(
            evaluation_results,
            summary_rows
        )

        print(
            f"\n✓ {question_id} checkpoint saved."
        )


# ============================================================
# FINAL AGGREGATES
# ============================================================

evaluation_elapsed = (
    time.perf_counter()
    -
    evaluation_start
)

version_a_evaluation_summary = pd.DataFrame(
    summary_rows
)

successful_df = (
    version_a_evaluation_summary[
        version_a_evaluation_summary[
            "status"
        ] != "ERROR"
    ]
    .copy()
)

total_questions = len(
    VERSION_A_EVAL_QUESTIONS
)

completed_count = int(
    version_a_evaluation_summary[
        "status"
    ]
    .eq(
        "COMPLETED"
    )
    .sum()
)

check_count = int(
    version_a_evaluation_summary[
        "status"
    ]
    .eq(
        "CHECK"
    )
    .sum()
)

error_count = int(
    version_a_evaluation_summary[
        "status"
    ]
    .eq(
        "ERROR"
    )
    .sum()
)

completion_rate_pct = (
    completed_count
    /
    total_questions
    *
    100.0
)

aggregate_metrics = {

    "questions":
        total_questions,

    "completed":
        completed_count,

    "check":
        check_count,

    "errors":
        error_count,

    "completion_rate_pct":
        round(
            completion_rate_pct,
            1
        ),

    "evaluation_wall_time_sec":
        round(
            evaluation_elapsed,
            3
        )
}


if not successful_df.empty:

    total_mcp_searches = int(
        successful_df[
            "mcp_searches"
        ].sum()
    )

    total_sources_returned = int(
        successful_df[
            "total_sources_returned"
        ].sum()
    )

    route_accuracy_pct = (
        successful_df[
            "route_match"
        ].astype(
            bool
        ).mean()
        *
        100.0
    )

    source_family_accuracy_pct = (
        successful_df[
            "source_family_match"
        ].astype(
            bool
        ).mean()
        *
        100.0
    )

    evidence_text_coverage_pct = (
        successful_df[
            "evidence_items_with_text"
        ].sum()
        /
        successful_df[
            "evidence_items"
        ].sum()
        *
        100.0

        if successful_df[
            "evidence_items"
        ].sum() > 0

        else 0.0
    )

    one_search_count = int(
        successful_df[
            "mcp_searches"
        ].eq(
            1
        ).sum()
    )

    two_search_count = int(
        successful_df[
            "mcp_searches"
        ].eq(
            2
        ).sum()
    )

    three_search_count = int(
        successful_df[
            "mcp_searches"
        ].eq(
            3
        ).sum()
    )

    global_retrieval_latency_per_search = (

        successful_df[
            "retrieval_latency_sec"
        ].sum()
        /
        total_mcp_searches

        if total_mcp_searches > 0

        else 0.0
    )

    aggregate_metrics.update({

        "route_accuracy_pct":
            round(
                route_accuracy_pct,
                1
            ),

        "source_family_accuracy_pct":
            round(
                source_family_accuracy_pct,
                1
            ),

        "evidence_text_coverage_pct":
            round(
                evidence_text_coverage_pct,
                1
            ),

        "total_mcp_searches":
            total_mcp_searches,

        "mean_mcp_searches_per_question":
            round(
                successful_df[
                    "mcp_searches"
                ].mean(),
                3
            ),

        "one_search_questions":
            one_search_count,

        "two_search_questions":
            two_search_count,

        "three_search_questions":
            three_search_count,

        "total_sources_returned":
            total_sources_returned,

        "mean_sources_per_search":
            round(
                (
                    total_sources_returned
                    /
                    total_mcp_searches
                )
                if total_mcp_searches > 0
                else 0.0,
                2
            ),

        "total_retrieval_latency_sec":
            round(
                successful_df[
                    "retrieval_latency_sec"
                ].sum(),
                3
            ),

        "mean_retrieval_latency_sec":
            round(
                successful_df[
                    "retrieval_latency_sec"
                ].mean(),
                3
            ),

        "median_retrieval_latency_sec":
            round(
                successful_df[
                    "retrieval_latency_sec"
                ].median(),
                3
            ),

        "global_retrieval_latency_per_search_sec":
            round(
                global_retrieval_latency_per_search,
                3
            ),

        "total_e2e_latency_sec":
            round(
                successful_df[
                    "total_latency_sec"
                ].sum(),
                3
            ),

        "mean_e2e_latency_sec":
            round(
                successful_df[
                    "total_latency_sec"
                ].mean(),
                3
            ),

        "median_e2e_latency_sec":
            round(
                successful_df[
                    "total_latency_sec"
                ].median(),
                3
            ),

        "total_input_tokens":
            int(
                successful_df[
                    "input_tokens"
                ].sum()
            ),

        "total_output_tokens":
            int(
                successful_df[
                    "output_tokens"
                ].sum()
            ),

        "total_tokens":
            int(
                successful_df[
                    "total_tokens"
                ].sum()
            ),

        "mean_total_tokens_per_question":
            round(
                successful_df[
                    "total_tokens"
                ].mean(),
                1
            ),

        "mean_answer_words":
            round(
                successful_df[
                    "answer_words"
                ].mean(),
                1
            ),

        "median_answer_words":
            round(
                successful_df[
                    "answer_words"
                ].median(),
                1
            )
    })


# ============================================================
# FINAL SAVE
# ============================================================

evaluation_artifact = {

    "metadata":
        experiment_metadata,

    "aggregate_metrics":
        aggregate_metrics,

    "results":
        evaluation_results
}

with open(
    VERSION_A_JSON_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        evaluation_artifact,
        f,
        indent=2,
        ensure_ascii=False,
        default=str
    )

version_a_evaluation_summary.to_csv(
    VERSION_A_CSV_PATH,
    index=False
)


# ============================================================
# SUMMARY
# ============================================================

print(
    "\n" + "=" * 100
)

print(
    "VERSION A — BENCHMARK V2 QUESTION-LEVEL SUMMARY"
)

print("=" * 100)

display(
    version_a_evaluation_summary
)

print(
    "\n" + "=" * 100
)

print(
    "VERSION A — BENCHMARK V2 AGGREGATE SUMMARY"
)

print("=" * 100)

print(f"Questions                    : {total_questions}")
print(f"Completed                    : {completed_count}")
print(f"Check                        : {check_count}")
print(f"Errors                       : {error_count}")
print(f"Completion Rate              : {completion_rate_pct:.1f}%")

if not successful_df.empty:

    print(
        f"Route Accuracy               : "
        f"{aggregate_metrics['route_accuracy_pct']:.1f}%"
    )

    print(
        f"Source-Family Accuracy       : "
        f"{aggregate_metrics['source_family_accuracy_pct']:.1f}%"
    )

    print(
        f"Evidence Text Coverage       : "
        f"{aggregate_metrics['evidence_text_coverage_pct']:.1f}%"
    )

    print(
        f"Total MCP Searches           : "
        f"{aggregate_metrics['total_mcp_searches']}"
    )

    print(
        f"Mean MCP Searches / Question : "
        f"{aggregate_metrics['mean_mcp_searches_per_question']:.2f}"
    )

    print(
        f"Mean Retrieval Latency       : "
        f"{aggregate_metrics['mean_retrieval_latency_sec']:.3f} sec"
    )

    print(
        f"Mean E2E Latency             : "
        f"{aggregate_metrics['mean_e2e_latency_sec']:.3f} sec"
    )

    print(
        f"Total Tokens                 : "
        f"{aggregate_metrics['total_tokens']:,}"
    )

    print(
        f"Mean Answer Length           : "
        f"{aggregate_metrics['mean_answer_words']:.1f} words"
    )

print(
    f"Evaluation Wall Time         : "
    f"{evaluation_elapsed:.3f} sec"
)

print(
    f"Canonical JSON              : "
    f"{VERSION_A_JSON_PATH}"
)

print(
    f"Summary CSV                 : "
    f"{VERSION_A_CSV_PATH}"
)

print("=" * 100)

print("✓ Benchmark v2 completed.")
print("✓ 5 × 3GPP, 2 × TCC and 1 × Hybrid questions evaluated.")
print("✓ Benchmark definition and SHA-256 preserved.")
print("✓ Expected and observed routing preserved.")
print("✓ Full bounded evidence excerpts preserved in tool_trace.")
print("✓ Retrieval provenance and timing preserved.")
print("✓ Artifact supports relevance and claim-level groundedness evaluation.")
print("✓ Benchmark v1 artifacts remain untouched.")


VERSION A — MODULE 3 EVALUATION BENCHMARK V2
Benchmark             : module3_eval_v2
Composition           : 5 × 3GPP | 2 × TCC | 1 × HYBRID
Questions             : 8
Model                 : deepseek/deepseek-v4-flash-0731
Architecture          : Remote raw-corpus retrieval
Top-K / Search        : 5
Max MCP Searches      : 3
Evidence / Source     : 2,500 chars
Benchmark SHA-256     : d40c0090c371f0a99ea6057bbb3fa8024e6b7d4174e037e7a6cf9fef9b9667f5
JSON Artifact         : /content/version_a_evaluation/version_a_deepseek_v4_flash_0731_evaluation_v2.json
CSV Summary           : /content/version_a_evaluation/version_a_deepseek_v4_flash_0731_summary_v2.csv

Q1 | 5G Core | Expected Route: 3GPP
QUESTION:
Explain the role of the AMF in registration and mobility management procedures in a 5G Standalone network.

FINAL ANSWER
----------------------------------------------------------------------------------------------------
## Role of the AMF in Registration and Mobility Management (5G SA)

In 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


FINAL ANSWER
----------------------------------------------------------------------------------------------------
## HTTP/TLS in the 5G Service-Based Architecture

### Communication model

In the SBA, control-plane Network Functions (NFs) such as the AMF and SMF expose and consume services over service-based interfaces (e.g., `Namf`, `Nsmf`), which can also be depicted as reference points (e.g., N11 between AMF and SMF) per TS 23.501. The NFs are logical functions – AMF performs access and mobility management (registration, connection management, reachability, mobility, slice selection), while SMF performs session management (PDU session establishment, UPF selection/control, IP address allocation, policy enforcement) – and they communicate over a common, standards-based transport.

### The SBI protocol stack (3GPP TS 29.500)

TS 29.500 defines the service-based interface (SBI) protocol stack as six layers:

1. **Application** (NF services, resource URIs, JSON payloads)
2. **HTTP/2** (

,question_id,domain,knowledge_scope,expected_route,routes_used,route_match,observed_source_families,source_family_match,status,stop_reason,...,retrieval_latency_sec,mean_retrieval_latency_per_search_sec,total_latency_sec,non_retrieval_latency_sec,retrieval_latency_share_pct,input_tokens,output_tokens,total_tokens,final_answer_tokens,answer_words
0,Q1,5G Core,3GPP,3gpp,"3gpp,3gpp",True,3GPP,True,COMPLETED,end_turn,...,1.264,0.632,14.460,13.196,8.7,13384,1743,15127,1458,455
1,Q2,Mobility,3GPP,3gpp,3gpp,True,3GPP,True,COMPLETED,end_turn,...,0.759,0.759,12.978,12.219,5.8,6279,1732,8011,1577,451
2,Q3,RAN,3GPP,3gpp,"3gpp,3gpp",True,3GPP,True,COMPLETED,end_turn,...,1.244,0.622,10.262,9.017,12.1,8634,1121,9755,802,437
3,Q4,QoS,3GPP,3gpp,"3gpp,3gpp,3gpp",True,3GPP,True,CHECK,end_turn,...,1.253,0.418,12.673,11.420,9.9,14891,1537,16428,1058,0
4,Q5,Security,3GPP,3gpp,"3gpp,tcc,3gpp",True,"3GPP,TCC",True,CHECK,max_tokens,...,4.558,1.519,17.947,13.389,25.4,19178,2269,21447,1800,265
5,Q6,Internet Transport,TCC,tcc,"tcc,tcc",True,TCC,True,COMPLETED,end_turn,...,11.240,5.620,25.004,13.764,45.0,11647,1955,13602,1451,476
6,Q7,Telecom AI Research,TCC,tcc,"tcc,tcc,tcc",True,TCC,True,COMPLETED,end_turn,...,13.071,4.357,19.912,6.841,65.6,21021,773,21794,348,113
7,Q8,5G SBA + Internet Protocols,Hybrid,hybrid,"hybrid,3gpp",True,"3GPP,TCC",True,CHECK,max_tokens,...,6.250,3.125,20.331,14.081,30.7,9532,2076,11608,1800,400



VERSION A — BENCHMARK V2 AGGREGATE SUMMARY
Questions                    : 8
Completed                    : 5
Check                        : 3
Errors                       : 0
Completion Rate              : 62.5%
Route Accuracy               : 100.0%
Source-Family Accuracy       : 100.0%
Evidence Text Coverage       : 100.0%
Total MCP Searches           : 18
Mean MCP Searches / Question : 2.25
Mean Retrieval Latency       : 4.955 sec
Mean E2E Latency             : 16.696 sec
Total Tokens                 : 117,772
Mean Answer Length           : 324.6 words
Evaluation Wall Time         : 133.646 sec
Canonical JSON              : /content/version_a_evaluation/version_a_deepseek_v4_flash_0731_evaluation_v2.json
Summary CSV                 : /content/version_a_evaluation/version_a_deepseek_v4_flash_0731_summary_v2.csv
✓ Benchmark v2 completed.
✓ 5 × 3GPP, 2 × TCC and 1 × Hybrid questions evaluated.
✓ Benchmark definition and SHA-256 preserved.
✓ Expected and observed routing preserved.
✓ Fu

**Observation — DeepSeek V4 Flash 0731 Version A Benchmark v2**

- **5/8** questions were `COMPLETED`, with **3 CHECK** outcome(s) and **0 execution errors**.
- Route accuracy: **100.0%**; source-family accuracy: **100.0%**; evidence-text coverage: **100.0%**.
- MCP searches: **18** (**2.25/question**).
- Mean retrieval latency: **4.955 s**; mean end-to-end latency: **16.696 s**.
- Total tokens: **117,772**; mean final response length: **324.6 words**.

Three questions remain correctly marked CHECK: Q4 exhausted the search budget without a usable final answer, Q5 reached the 1,800-token ceiling after the forced-final path, and Q8 reached the output limit. Q7 missed IEEE/OpenAlex and abstained. These outcomes must not be relabelled as successful completions.

***Key Finding:*** Search count, routing/collection discipline, latency and token usage are execution characteristics, not substitutes for answer-quality scoring. Correctness, completeness, relevance and claim grounding are assessed in the final cross-model A/B evaluation.
